# Model Health &amp; Security Suite

<div style="background:linear-gradient(135deg,#1B3A4B,#117865);padding:24px 28px;border-radius:12px;margin:16px 0;color:#fff">
  <div style="font-size:26px;font-weight:700;letter-spacing:0.5px">Model Health &amp; Security Suite</div>
  <div style="font-size:14px;color:#7DBDAB;margin-top:6px">A 13-tool diagnostic, remediation &amp; security auditing toolkit for Microsoft Fabric semantic models</div>
  <div style="margin-top:12px;padding:10px 14px;background:rgba(255,255,255,0.12);border-radius:8px;font-size:13px;line-height:1.5;color:#E0F0EC">
    <b style="color:#F2C811">One notebook. 13 tools. Zero guesswork.</b> Scan any semantic model for Copilot readiness, DAX errors,
    data quality, lineage, performance, and best practices &mdash; then auto-fix descriptions, synonyms, format strings, and more with one click.
    The <b>Security X-Ray</b> goes further: it expands AAD groups to real users and flags when workspace privileges <b>silently bypass</b> your RLS filters.
  </div>
</div>

---

## 🚀 How to Run This Notebook

<div style="background:#F0F6F5;border:1px solid #D2ECE8;border-left:4px solid #117865;border-radius:8px;padding:16px 20px;margin:16px 0">

**6-step quickstart:**

1. **Run Cell 1** — installs `semantic-link`, `semantic-link-labs`, and `openai` (kernel restarts automatically; re-run the cell if needed)
2. **Run Cell 2** — loads the scan engine (13 tools + AI client). You should see `AI: Azure OpenAI (Fabric Copilot) ready`
3. **Run Cell 3** — the interactive console appears. Configure:
   - Select your **Workspace** and **Semantic Model** from the dropdowns
   - Check the tools you want to run (all 9 unique tools selected by default)
   - Enable **"Use Service Principal"** and enter SPN credentials — **required for Security X-Ray AAD group expansion**
   - Optionally enable **"Use AI Descriptions"** and provide business context in the textarea for better AI-generated labels
   - Click the green **Run Scan** button and wait ~60–90 seconds. The form stays visible — scan log is captured to the next cell.
4. **Run Cell 4 (Scan Log)** — prints the full verbose scan output (tool-by-tool progress, AI calls, Graph expansion, etc.)
5. **Run Cell 5 (View Report)** — renders the full HTML dashboard with scores, findings, fix plan, and Security X-Ray tabs
6. **Run Cell 6 (Apply & Verify)** — review the fix plan, edit `SELECTED_FIXES` (use `"all"` or pick specific numbers), re-run to apply fixes to the model

</div>

## 📌 Notes — Please read before running

<div style="background:#FFF8E0;border:1px solid #FFE5A0;border-left:4px solid #FFB900;border-radius:8px;padding:14px 20px;margin:16px 0">

1. **Cell order matters** — Cells 1 → 2 → 3 → (Run Scan) → 4 → 5 → 6. Cell 2 builds the `run_scan()` function that Cell 3's button calls.
2. **Kernel restart after Cell 1** — the `%pip install` triggers a one-time kernel restart. If you see `ModuleNotFoundError`, re-run Cell 1 then continue.
3. **Service Principal recommended** — the Fabric runtime identity (`notebookutils.credentials.getToken`) does not always work for Graph API. Using SPN is faster (0.3s vs 60s+ timeout) and more reliable.
4. **Grant `Directory.Read.All` to the SPN** in Azure AD → App Registrations → API permissions, otherwise AAD group expansion in Security X-Ray will be skipped (RLS/OLS/workspace roles still work).
5. **AI descriptions need Fabric Copilot** enabled (F2+ capacity with Copilot tenant setting ON). If unavailable, the notebook falls back to template-based descriptions automatically.
6. **Security X-Ray requires TOM write access?** — No. The scan is **read-only**. Only Cell 6 (Apply & Verify) writes to the model, and only after you review the fix plan.
7. **Large models may take longer** — scan time scales with number of tables, measures, and relationships. The 9-tool default takes ~60–90s on a 30-measure model.

</div>

## ✅ Prerequisites

| Requirement | Detail |
|---|---|
| **Runtime** | Microsoft Fabric notebook (PySpark/Python), Runtime 1.3+ (Spark 3.5) |
| **Capacity** | F2+ or P1+ (any paid Fabric capacity); AI features require Copilot tenant switch enabled |
| **Semantic model** | Any Import / Direct Lake / DirectQuery model accessible via XMLA endpoint |
| **Service Principal** (recommended) | Required for full Security X-Ray. SPN needs: workspace Viewer+ role, `Directory.Read.All` in Azure AD, Power BI API access |
| **Python packages** | `semantic-link`, `semantic-link-labs`, `openai` — auto-installed by Cell 1 |

---

## What Problems Does This Solve?

Every Power BI developer and administrator has faced these situations. This suite catches them automatically &mdash; no manual checking, no guesswork.

<div style="margin:16px 0">

<details open style="margin:10px 0;border:2px solid #117865;border-radius:10px">
<summary style="padding:14px 18px;background:linear-gradient(135deg,#1B3A4B,#117865);color:#fff;font-weight:700;cursor:pointer;font-size:15px;border-radius:8px 8px 0 0">
&#128737; Security X-Ray &mdash; the game changer
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7;background:#F0F6F5">

<div style="background:#fff;border-left:4px solid #D13438;padding:10px 14px;border-radius:0 8px 8px 0;margin-bottom:12px">
<b style="color:#D13438">The problem nobody else solves:</b> In enterprise Power BI, security is managed through <b>Azure AD (Entra ID) groups</b> &mdash; not individual users. An AD group like <code>demo-ad-group-A</code> gets assigned to an RLS role, and that group might contain 200 users, nested sub-groups, and service principals. Today, there is <b>no tool in Power BI, no REST API, no sempy function, and no third-party solution</b> that answers the fundamental question:
<br/><br/>
<i>"Given all the RLS filters, OLS restrictions, AD group memberships, and workspace roles &mdash; what does each real human actually see when they open this model?"</i>
</div>

<b>Why this matters in practice:</b>
<ul style="margin:8px 0 12px 18px;padding:0">
<li><b>Workspace Members and Contributors bypass RLS entirely.</b> Most organizations don't realize this. You spend weeks building RLS filters, but anyone with Member or Contributor role on the workspace sees ALL data &mdash; unfiltered. This is a silent data leak that no standard audit catches.</li>
<li><b>AD groups are opaque.</b> The Power BI admin portal shows "demo-ad-group-B is assigned to role Sales_EU" but doesn't show who is IN that group. You have to go to Azure Portal, find the group, expand nested groups, and manually cross-reference. For 10 roles across 5 workspaces, this takes hours.</li>
<li><b>No single view exists.</b> To answer "can Bob see the Margin column?", you currently need to check: (1) which workspace role Bob has, (2) which AD groups Bob belongs to, (3) which model roles those groups are assigned to, (4) what RLS filter that role has, (5) what OLS restrictions apply. Our tool does this in one scan.</li>
<li><b>Individual access overrides AD group access.</b> If Alice is added to the workspace as a Contributor via her email address, but her AD group <code>demo-ad-group-A</code> is assigned to an RLS role as Viewer &mdash; Alice's <i>individual</i> Contributor access takes priority. Her RLS filter is bypassed even though her AD group has a restrictive role. This is the most common and hardest-to-detect security gap in enterprise Power BI deployments.</li>
</ul>

<b>What Security X-Ray does (5 steps):</b>
<ol style="margin:8px 0 12px 18px;padding:0">
<li><b>RLS + OLS extraction</b> via TOM &mdash; reads every role's FilterExpression and MetadataPermission. Flags dynamic filters (USERPRINCIPALNAME/USERNAME).</li>
<li><b>Role members</b> via DAX INFO.ROLES() + INFO.ROLEMEMBERSHIPS() &mdash; scale-out safe, no model mutations.</li>
<li><b>AAD group expansion</b> via Microsoft Graph /transitiveMembers &mdash; resolves nested groups to individual users. Graceful degradation if Directory.Read.All not granted.</li>
<li><b>Workspace role assignments</b> via Fabric REST /v1/workspaces/{id}/roleAssignments &mdash; maps every principal to Admin/Member/Contributor/Viewer.</li>
<li><b>Effective Access Map</b> &mdash; joins all four datasets into one row per user showing: workspace role, model role, RLS filter, OLS hidden columns, and the <b>Effective View</b> label.</li>
</ol>

<b>Example output:</b>
<div style="overflow-x:auto;margin:8px 0">
<table style="width:100%;border-collapse:collapse;font-size:12px;border:1px solid #EDEBE9;border-radius:6px">
<tr style="background:#F3F2F1"><th style="padding:6px 10px;text-align:left">User</th><th style="padding:6px 10px;text-align:left">Source</th><th style="padding:6px 10px;text-align:left">Workspace Role</th><th style="padding:6px 10px;text-align:left">Model Role</th><th style="padding:6px 10px;text-align:left">RLS Filter</th><th style="padding:6px 10px;text-align:left">Effective View</th></tr>
<tr style="background:#FDE7E8"><td style="padding:5px 10px">user1@example.com</td><td>AD Group: demo-ad-group-A</td><td style="color:#D13438;font-weight:700">Contributor</td><td>Sales_EU</td><td>[Region]="EMEA"</td><td style="color:#D13438;font-weight:700">FULL (Contributor bypass)</td></tr>
<tr style="background:#fff"><td style="padding:5px 10px" colspan="6" style="font-size:11px;color:#D13438;padding-left:30px"><em>&uarr; Alice is in AD group demo-ad-group-A (assigned to role Sales_EU with RLS filter [Region]="EMEA"), but she is also a workspace Contributor &mdash; so RLS is completely bypassed. She sees ALL data, not just EMEA. This is a silent security gap.</em></td></tr>
<tr style="background:#FDE7E8"><td style="padding:5px 10px">user2@example.com</td><td>Direct</td><td style="color:#D13438;font-weight:700">Member</td><td>&mdash;</td><td>&mdash;</td><td style="color:#D13438;font-weight:700">FULL (Member bypass)</td></tr>
<tr style="background:#E6F2E6"><td style="padding:5px 10px">user3@example.com</td><td>AD Group: demo-ad-group-A</td><td>Viewer</td><td>Sales_EU</td><td>[Region]="EMEA"</td><td style="color:#107C10;font-weight:600">Filtered</td></tr>
<tr style="background:#E6F2E6"><td style="padding:5px 10px">user4@example.com</td><td>AD Group: demo-ad-group-A</td><td>Viewer</td><td>Sales_EU</td><td>[Region]="EMEA"</td><td style="color:#107C10;font-weight:600">Filtered</td></tr>
<tr style="background:#FFF8E0"><td style="padding:5px 10px">user5@example.com</td><td>Direct</td><td>Viewer</td><td>Sales_Dynamic</td><td>USERPRINCIPALNAME()</td><td style="color:#FFB900;font-weight:600">Dynamic</td></tr>
</table>
</div>

<div style="background:#FFF8E0;padding:8px 12px;border-radius:6px;font-size:12px;color:#7A6400;margin-top:8px">
<b>This view does not exist anywhere else.</b> Not in Power BI Admin portal, not in the REST API, not in sempy/sempy_labs, not in any third-party tool. The combination of Graph API group expansion + workspace role cross-reference + RLS bypass detection is unique to this suite.
</div>

</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#129302; Copilot Readiness &mdash; "Why doesn't Copilot understand my model?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> You enable Copilot on your report, ask "What was total revenue last quarter?", and Copilot says "I don't have enough information" or gives a wrong answer. This happens because Copilot relies on model metadata &mdash; descriptions, synonyms, proper naming &mdash; to understand your data. Most models have none of this.<br/><br/>

<b>What we check (8 areas, 100 points):</b>
<ul style="margin:4px 0 8px 18px;padding:0">
<li><b>Table descriptions</b> &mdash; Does every table explain what it contains? Without this, Copilot doesn't know if "fact_sale" is about sales, shipping, or returns.</li>
<li><b>Column descriptions</b> &mdash; Does "SalesTerritory" mean a region, a team, or an office? Copilot needs to know.</li>
<li><b>Measure descriptions</b> &mdash; Is "Profit Margin" gross or net? Pre-tax or post-tax? The description tells Copilot.</li>
<li><b>Synonyms</b> &mdash; If a user asks about "income" but the measure is called "Total Revenue", Copilot needs a synonym mapping.</li>
<li><b>Naming conventions</b> &mdash; camelCase and underscores confuse Copilot. "total_excluding_tax" should be "Total Excluding Tax".</li>
<li><b>Hidden key columns</b> &mdash; SaleKey, CityKey, LineageKey should be hidden. Copilot shouldn't try to analyze surrogate keys.</li>
<li><b>Format strings</b> &mdash; "$#,##0" tells Copilot this is currency. Without it, revenue shows as "41614713983".</li>
<li><b>AI Prep</b> &mdash; Are Copilot AI Instructions configured? Is the schema simplified (technical columns excluded)?</li>
</ul>

<b>Example findings:</b><br/>
<code style="background:#FDE7E8;padding:2px 6px;border-radius:4px;color:#D13438">FAIL: 13 columns have no description. Copilot cannot explain these fields to users.</code><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: 20 key/ID columns are visible. Hide them so Copilot doesn't analyze surrogate keys.</code><br/>
<code style="background:#E6F2E6;padding:2px 6px;border-radius:4px;color:#107C10">PASS: AI Prep enabled, AI instructions configured, 20/75 columns excluded from Copilot.</code><br/><br/>

<b>Auto-fix (9 fix types):</b> Descriptions (AI-generated with schema context or template), synonyms (via TOM Culture ObjectTranslation), hidden columns, format strings, summarize-by, model description, AI instructions (via LinguisticMetadata CustomInstructions), schema reduction (isAvailableInMDX).
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#9989; Test Framework &mdash; "Are all my measures actually working?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> Your model has 30+ measures. Some were written months ago, some reference columns that were renamed, some have edge cases that return errors. The only way to find broken measures today is to click through every visual in every report page. With 10 reports and 50 pages, that's hours of manual testing.<br/><br/>

<b>What we do:</b> Evaluates <b>every single DAX measure</b> via <code>EVALUATE ROW("R", 'Table'[Measure])</code>. Also validates all relationships for many-to-many, bi-directional, and inactive patterns.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FDE7E8;padding:2px 6px;border-radius:4px;color:#D13438">FAIL: Measure 'Revenue PY' &mdash; error: "SAMEPERIODLASTYEAR requires a date column in a date table."</code><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: Relationship fact_sale[BillToCustomerKey] &rarr; dimension_customer[CustomerKey] is inactive.</code><br/>
<code style="background:#E6F2E6;padding:2px 6px;border-radius:4px;color:#107C10">PASS: Total Revenue = $41,614,713,983 (evaluates successfully)</code><br/><br/>

<b>Why it matters:</b> One broken measure in production = blank visuals, wrong executive decisions, and urgent support tickets. This tool catches them before deployment.
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128279; Lineage &mdash; "If I change this column, what breaks?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> The data team wants to rename <code>TotalIncludingTax</code> to <code>GrossAmount</code>. Before you do it, you need to know: which measures reference this column? Which measures reference THOSE measures? One column rename can cascade through 15 measures and break 5 reports.<br/><br/>

<b>What we do:</b> Traces every measure back to its source columns through the full DAX expression dependency chain. Computes <b>blast radius</b> &mdash; the number of measures that would break if a given measure or column changes. Finds orphan measures (defined but never used).<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: Blast Radius &mdash; Total Revenue is used by 12 other measures (Profit Margin, Revenue YoY %, Net Revenue, Revenue Per Customer, ...). Change with extreme caution.</code><br/>
<code style="background:#F3F2F1;padding:2px 6px;border-radius:4px">INFO: Orphan measure 'Old Metric v2' is not referenced by any other measure. Consider removing.</code><br/><br/>

<b>Why it matters:</b> Impact analysis before schema changes prevents production outages. "I renamed one column and 8 reports broke" is a preventable disaster.
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128202; Data Quality &mdash; "Why are my slicers showing blanks?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> Business users report: "The Region slicer shows (Blank) as an option" or "The total doesn't match when I filter by Category." These are symptoms of NULL values in slicer columns and broken referential integrity across relationships.<br/><br/>

<b>What we do:</b> Checks NULL rate per column (via DAX COUNTBLANK), validates referential integrity across all relationships, analyzes cardinality.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FDE7E8;padding:2px 6px;border-radius:4px;color:#D13438">FAIL: dimension_customer.BuyingGroup has 45% NULL rate. Slicers will show "(Blank)" and aggregations may be skewed.</code><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: 23 rows in fact_sale.CityKey have no match in dimension_city.CityKey. These rows are invisible in reports filtered by city.</code>
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128268; DAX Dependencies &mdash; "Which measures are fragile?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> Measure A calls Measure B which calls Measure C which calls Measure D. Deeply nested DAX chains are fragile &mdash; one change at any level cascades up. Without a dependency graph, you're changing code blind.<br/><br/>

<b>What we do:</b> Parses every DAX expression to build a full dependency graph. Flags high complexity (deep nesting), high fan-out (many dependents), and potential circular references.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: Blast Radius &mdash; Total Revenue is used by 12 other measures. High fan-out = fragile chain.</code><br/>
<code style="background:#E6F2E6;padding:2px 6px;border-radius:4px;color:#107C10">PASS: Avg Order Value &mdash; 2 dependencies (Total Revenue, Order Count). Clean and simple.</code>
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128196; Report Visuals &mdash; "Are all my report visuals using valid data?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> You renamed a column in the model last week. A report visual still references the old name. Result: blank visual in production and a confused VP. Also: your model has 30 measures, but only 18 are used in any report &mdash; the rest are dead code nobody notices.<br/><br/>

<b>What we do:</b> Uses <code>sempy_labs.report.ReportWrapper</code> to scan all reports connected to this semantic model. Lists every visual object (columns, measures, filters). Cross-references with the model to find broken references and unused measures.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FDE7E8;padding:2px 6px;border-radius:4px;color:#D13438">FAIL: Visual on page "Sales Overview" references column 'OldColumnName' which no longer exists in the model.</code><br/>
<code style="background:#F3F2F1;padding:2px 6px;border-radius:4px">INFO: Measure 'Weighted Qty Revenue' is defined in the model but not used in any report visual.</code>
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128260; Model Diff &mdash; "What changed between dev and prod?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> Before promoting a model from dev to production, the team asks "what exactly changed?" Comparing two models manually &mdash; checking every table, column, measure, relationship &mdash; is tedious and error-prone. One missed change can break production reports.<br/><br/>

<b>What we do:</b> Compares the current model against a baseline model (select in the "Compare Against" dropdown). Shows added, removed, and modified objects side by side.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#F3F2F1;padding:2px 6px;border-radius:4px">NEW: Table 'dimension_product' added (12 columns). REMOVED: Column fact_sale.OldMetric. MODIFIED: Measure 'Total Revenue' expression changed.</code><br/><br/>

<b>Setup:</b> Select a compare model in the "Compare Against" dropdown. Both models in the same workspace are supported. For cross-workspace comparison, the compare model must be accessible from the current workspace context.<br/><br/>
<b>Tip:</b> Create a copy of your model before making changes. Run the scan with the copy as the compare target to see exactly what you changed.
</div></details>

<details style="margin:10px 0;border:1px solid #EDEBE9;border-radius:10px">
<summary style="padding:14px 18px;background:#FAF9F8;font-weight:700;cursor:pointer;font-size:15px">
&#128274; Security Audit &mdash; "Is my security configuration complete?"
</summary>
<div style="padding:14px 18px;font-size:13px;line-height:1.7">
<b>The problem:</b> You defined 4 security roles, but one has no members, one has no description, and two tables have no RLS filter at all. Without a compliance checklist, these gaps go unnoticed until an audit or a data breach.<br/><br/>

<b>What we do:</b> Compliance score across: roles exist, have descriptions, have members assigned, RLS filters are defined per table, OLS (Object Level Security) is configured where needed.<br/><br/>

<b>Example findings:</b><br/>
<code style="background:#FDE7E8;padding:2px 6px;border-radius:4px;color:#D13438">FAIL: Role 'Executive' has no RLS filter and no members. Anyone in this role sees ALL data.</code><br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">WARN: Role 'Sales Manager' has no description. Document what this role grants access to.</code><br/><br/>

<b>Note:</b> Security Audit checks if the configuration is <i>complete</i>. Security X-Ray checks if the configuration is <i>effective</i> (who actually sees what). Both are shown in the unified Security X-Ray section of the report.
</div></details>

</div>

### Powered by Semantic Link Labs (optional &mdash; unchecked by default)

<div style="margin:12px 0">

<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px">
<summary style="padding:10px 14px;background:#F0F6F5;font-weight:600;cursor:pointer;font-size:13px;color:#117865">Best Practices &mdash; sempy_labs BPA rules</summary>
<div style="padding:10px 14px;font-size:12px;line-height:1.6">
Runs the full Best Practice Analyzer rule set from sempy_labs against your model. Catches anti-patterns like <code>FILTER(ALL(Table))</code> instead of <code>CALCULATE</code>, missing sort-by columns, <code>COUNTROWS(FILTER())</code> instead of <code>CALCULATE(COUNTROWS())</code>, and more.<br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">Example: "Measure 'Revenue All Items' uses FILTER(ALL()) anti-pattern. Use CALCULATE with REMOVEFILTERS instead."</code>
</div></details>

<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px">
<summary style="padding:10px 14px;background:#F0F6F5;font-weight:600;cursor:pointer;font-size:13px;color:#117865">Vertipaq Performance &mdash; memory analysis</summary>
<div style="padding:10px 14px;font-size:12px;line-height:1.6">
Analyzes memory consumption per table and column. Finds oversized dictionaries (a column with 10M unique values uses 500MB of RAM) and high-cardinality columns that slow queries. Critical for large models approaching capacity limits.<br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">Example: "Column dimension_city.Location (String) uses 45MB dictionary. Consider reducing cardinality."</code>
</div></details>

<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px">
<summary style="padding:10px 14px;background:#F0F6F5;font-weight:600;cursor:pointer;font-size:13px;color:#117865">Direct Lake &mdash; fallback detection</summary>
<div style="padding:10px 14px;font-size:12px;line-height:1.6">
For Fabric Direct Lake models: checks for calculated tables and calculated columns that force the engine to fall back from Direct Lake to import mode. Fallback means slower queries and higher memory usage.<br/>
<code style="background:#FFF8E0;padding:2px 6px;border-radius:4px;color:#7A6400">Example: "Calculated table 'Calendar' forces Direct Lake fallback. Use a lakehouse table instead."</code>
</div></details>

<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px">
<summary style="padding:10px 14px;background:#F0F6F5;font-weight:600;cursor:pointer;font-size:13px;color:#117865">Capacity &mdash; workspace inventory</summary>
<div style="padding:10px 14px;font-size:12px;line-height:1.6">
Scans the workspace for all semantic models. Shows names, sizes, refresh timestamps, and status. Useful for capacity planning and finding models that haven't been refreshed in weeks.<br/>
<code style="background:#F3F2F1;padding:2px 6px;border-radius:4px">Example: "Found 8 models. 'Legacy Report Model' last refreshed 45 days ago. Consider archiving."</code>
</div></details>

</div>

---

## Architecture

<div style="display:flex;gap:12px;margin:16px 0;flex-wrap:wrap">
  <div style="flex:1 1 200px;background:#F0F6F5;border:2px solid #117865;border-radius:10px;padding:14px;text-align:center">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:1px">Cell 1</div>
    <div style="font-size:16px;font-weight:700;color:#1B3A4B;margin:4px 0">Setup</div>
    <div style="font-size:12px;color:#605E5C">pip install<br/>semantic-link<br/>semantic-link-labs</div>
  </div>
  <div style="flex:2 1 300px;background:#F0F6F5;border:2px solid #117865;border-radius:10px;padding:14px;text-align:center">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:1px">Cell 2</div>
    <div style="font-size:16px;font-weight:700;color:#1B3A4B;margin:4px 0">Engine</div>
    <div style="font-size:12px;color:#605E5C">13 diagnostic tools &middot; Fix plan generator<br/>_render_widgets() &middot; _LAST_SCAN cache<br/>Refresh helpers &middot; Noise filters</div>
  </div>
  <div style="flex:1 1 220px;background:#F0F6F5;border:2px solid #117865;border-radius:10px;padding:14px;text-align:center">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:1px">Cell 3</div>
    <div style="font-size:16px;font-weight:700;color:#1B3A4B;margin:4px 0">Console</div>
    <div style="font-size:12px;color:#605E5C">Interactive UI &middot; Workspace/Model picker<br/>6-tab results &middot; 4 action buttons<br/>Restore on refresh</div>
  </div>
  <div style="flex:1 1 200px;background:#FFF8E0;border:2px solid #FFB900;border-radius:10px;padding:14px;text-align:center">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:1px">Cell 4</div>
    <div style="font-size:16px;font-weight:700;color:#1B3A4B;margin:4px 0">Apply & Verify</div>
    <div style="font-size:12px;color:#605E5C">Auto-fix via TOM &middot; Schema refresh<br/>Before/After comparison</div>
  </div>
</div>

<div style="background:#F3F2F1;border-radius:10px;padding:16px;margin:16px 0">
  <div style="font-weight:700;color:#1B3A4B;margin-bottom:10px;font-size:14px">Data Flow</div>
  <div style="display:flex;gap:8px;flex-wrap:wrap;justify-content:center">
    <div style="background:#117865;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">Console</div>
    <div style="color:#605E5C;font-size:18px;line-height:28px">&rarr;</div>
    <div style="background:#0078D4;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">sempy.fabric APIs</div>
    <div style="color:#605E5C;font-size:18px;line-height:28px">&rarr;</div>
    <div style="background:#1B3A4B;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">TOM Connection</div>
    <div style="color:#605E5C;font-size:18px;line-height:28px">&rarr;</div>
    <div style="background:#107C10;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">Results (6 tabs)</div>
  </div>
  <div style="display:flex;gap:8px;flex-wrap:wrap;justify-content:center;margin-top:8px">
    <div style="background:#605E5C;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">Microsoft Graph</div>
    <div style="color:#605E5C;font-size:12px;line-height:28px">AAD Groups &rarr; Users</div>
    <div style="background:#605E5C;color:#fff;padding:6px 14px;border-radius:20px;font-size:12px;font-weight:600">Fabric REST</div>
    <div style="color:#605E5C;font-size:12px;line-height:28px">Workspace Roles</div>
  </div>
</div>

---

## Quick Start

<div style="display:flex;gap:12px;margin:16px 0;flex-wrap:wrap">
  <div style="flex:1 1 200px;background:#fff;border-left:4px solid #117865;border-radius:0 8px 8px 0;padding:12px 16px;box-shadow:0 1px 3px rgba(0,0,0,0.08)">
    <div style="font-size:20px;font-weight:700;color:#117865">1</div>
    <div style="font-weight:600;color:#323130">Setup</div>
    <div style="font-size:12px;color:#605E5C">Run Cell 1. Installs dependencies. May restart kernel.</div>
  </div>
  <div style="flex:1 1 200px;background:#fff;border-left:4px solid #0078D4;border-radius:0 8px 8px 0;padding:12px 16px;box-shadow:0 1px 3px rgba(0,0,0,0.08)">
    <div style="font-size:20px;font-weight:700;color:#0078D4">2</div>
    <div style="font-weight:600;color:#323130">Engine</div>
    <div style="font-size:12px;color:#605E5C">Run Cell 2. Loads 13 tools. You'll see "Engine loaded: 13 tools ready."</div>
  </div>
  <div style="flex:1 1 200px;background:#fff;border-left:4px solid #FFB900;border-radius:0 8px 8px 0;padding:12px 16px;box-shadow:0 1px 3px rgba(0,0,0,0.08)">
    <div style="font-size:20px;font-weight:700;color:#FFB900">3</div>
    <div style="font-weight:600;color:#323130">Console</div>
    <div style="font-size:12px;color:#605E5C">Run Cell 3. Select workspace, model, tools. Click <b>Run Scan</b>.</div>
  </div>
  <div style="flex:1 1 200px;background:#fff;border-left:4px solid #D13438;border-radius:0 8px 8px 0;padding:12px 16px;box-shadow:0 1px 3px rgba(0,0,0,0.08)">
    <div style="font-size:20px;font-weight:700;color:#D13438">4</div>
    <div style="font-weight:600;color:#323130">Apply</div>
    <div style="font-size:12px;color:#605E5C">Review Fix Plan tab. Run Cell 6 to auto-fix + verify improvements.</div>
  </div>
</div>

> **Tip:** If the console disappears after browser refresh, just re-run Cell 3. Click **Restore Last Scan** to bring back results instantly.

---

## Security X-Ray Flow

<div style="background:#F3F2F1;border-radius:10px;padding:16px;margin:16px 0">
  <div style="display:flex;gap:0;flex-wrap:wrap;justify-content:center;align-items:stretch">
    <div style="flex:1 1 150px;background:#1B3A4B;color:#fff;padding:10px 14px;border-radius:8px 0 0 8px;text-align:center;min-width:140px">
      <div style="font-size:10px;color:#7DBDAB;text-transform:uppercase">Step 1</div>
      <div style="font-weight:700;font-size:13px;margin:4px 0">RLS + OLS</div>
      <div style="font-size:11px;color:#B3D9D2">via TOM</div>
    </div>
    <div style="flex:1 1 150px;background:#117865;color:#fff;padding:10px 14px;text-align:center;min-width:140px">
      <div style="font-size:10px;color:#B3D9D2;text-transform:uppercase">Step 2</div>
      <div style="font-weight:700;font-size:13px;margin:4px 0">Role Members</div>
      <div style="font-size:11px;color:#D2ECE8">via DAX INFO</div>
    </div>
    <div style="flex:1 1 150px;background:#0078D4;color:#fff;padding:10px 14px;text-align:center;min-width:140px">
      <div style="font-size:10px;color:#B3D9FF;text-transform:uppercase">Step 3</div>
      <div style="font-weight:700;font-size:13px;margin:4px 0">AAD Groups</div>
      <div style="font-size:11px;color:#D2E4FF">via Graph API</div>
    </div>
    <div style="flex:1 1 150px;background:#605E5C;color:#fff;padding:10px 14px;text-align:center;min-width:140px">
      <div style="font-size:10px;color:#C8C6C4;text-transform:uppercase">Step 4</div>
      <div style="font-weight:700;font-size:13px;margin:4px 0">Workspace Roles</div>
      <div style="font-size:11px;color:#E1DFDD">via Fabric REST</div>
    </div>
    <div style="flex:1 1 150px;background:#107C10;color:#fff;padding:10px 14px;border-radius:0 8px 8px 0;text-align:center;min-width:140px">
      <div style="font-size:10px;color:#A7E3A5;text-transform:uppercase">Step 5</div>
      <div style="font-weight:700;font-size:13px;margin:4px 0">Access Map</div>
      <div style="font-size:11px;color:#C8F0C8">join + classify</div>
    </div>
  </div>
  <div style="margin-top:12px;font-size:12px;color:#605E5C;text-align:center">
    Outputs: <b>Effective Access Map</b> &middot; RLS Filters &middot; OLS Restrictions &middot; Workspace Assignments &middot; Raw Members
  </div>
</div>

### Required Permissions

<div style="display:flex;gap:10px;margin:12px 0;flex-wrap:wrap">
  <div style="flex:1 1 200px;background:#fff;border:1px solid #EDEBE9;border-radius:8px;padding:12px">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase">Microsoft Graph</div>
    <div style="font-weight:600;color:#323130;margin:4px 0">Directory.Read.All</div>
    <div style="font-size:11px;color:#605E5C">Expand AAD groups to real users</div>
  </div>
  <div style="flex:1 1 200px;background:#fff;border:1px solid #EDEBE9;border-radius:8px;padding:12px">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase">Fabric REST</div>
    <div style="font-weight:600;color:#323130;margin:4px 0">Workspace Reader+</div>
    <div style="font-size:11px;color:#605E5C">Read workspace role assignments</div>
  </div>
  <div style="flex:1 1 200px;background:#fff;border:1px solid #EDEBE9;border-radius:8px;padding:12px">
    <div style="font-size:11px;color:#605E5C;text-transform:uppercase">XMLA / TOM</div>
    <div style="font-weight:600;color:#323130;margin:4px 0">Model Read</div>
    <div style="font-size:11px;color:#605E5C">Read roles, RLS, OLS</div>
  </div>
</div>

---

## What Gets Fixed (Cell 6)

<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(200px,1fr));gap:10px;margin:12px 0">
  <div style="background:#E6F2E6;border-radius:8px;padding:10px 14px;border-left:3px solid #107C10">
    <div style="font-weight:600;color:#107C10;font-size:13px">Descriptions</div>
    <div style="font-size:11px;color:#605E5C">Table, column, measure — AI or template</div>
  </div>
  <div style="background:#E6F2E6;border-radius:8px;padding:10px 14px;border-left:3px solid #107C10">
    <div style="font-weight:600;color:#107C10;font-size:13px">Synonyms</div>
    <div style="font-size:11px;color:#605E5C">AI-generated Copilot synonyms via TOM Culture</div>
  </div>
  <div style="background:#E6F2E6;border-radius:8px;padding:10px 14px;border-left:3px solid #107C10">
    <div style="font-weight:600;color:#107C10;font-size:13px">Hidden Columns</div>
    <div style="font-size:11px;color:#605E5C">Key/ID/FK columns hidden from report builders</div>
  </div>
  <div style="background:#E6F2E6;border-radius:8px;padding:10px 14px;border-left:3px solid #107C10">
    <div style="font-weight:600;color:#107C10;font-size:13px">Format Strings</div>
    <div style="font-size:11px;color:#605E5C">Currency, percentage, decimal formatting</div>
  </div>
  <div style="background:#E6F2E6;border-radius:8px;padding:10px 14px;border-left:3px solid #107C10">
    <div style="font-weight:600;color:#107C10;font-size:13px">Summarize By</div>
    <div style="font-size:11px;color:#605E5C">Prevents accidental SUM on ID columns</div>
  </div>
  <div style="background:#FFF8E0;border-radius:8px;padding:10px 14px;border-left:3px solid #FFB900">
    <div style="font-weight:600;color:#7A6400;font-size:13px">Schema Refresh</div>
    <div style="font-size:11px;color:#605E5C">Auto-syncs metadata after fixes applied</div>
  </div>
</div>

---

## Scoring

<div style="display:flex;gap:8px;margin:12px 0;flex-wrap:wrap;justify-content:center">
  <div style="background:#107C10;color:#fff;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">A+ 95-100</div>
  <div style="background:#107C10;color:#fff;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">A 90-94</div>
  <div style="background:#0078D4;color:#fff;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">B 75-89</div>
  <div style="background:#FFB900;color:#323130;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">C 60-74</div>
  <div style="background:#E87400;color:#fff;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">D 50-59</div>
  <div style="background:#D13438;color:#fff;padding:6px 16px;border-radius:20px;font-size:13px;font-weight:700">F 0-49</div>
</div>

---

## Requirements

- **Environment:** Microsoft Fabric Notebook (PySpark / Python)
- **Libraries:** `semantic-link`, `semantic-link-labs` (installed by Cell 1)
- **Semantic model:** Any model accessible from the workspace
- **Optional:** Service Principal credentials (for cross-workspace access)
- **Optional:** Microsoft Graph permissions (for Security X-Ray AAD group expansion)

---

---

## 🔧 Troubleshooting

<div style="background:#FFF8E0;border:1px solid #FFE5A0;border-left:4px solid #FFB900;border-radius:8px;padding:14px 20px;margin:16px 0">

**Graph API 403 Forbidden** — Your SPN doesn't have directory read permission.
1. Go to **Azure Portal → App Registrations → {your SPN app} → API permissions**
2. Click **Add permission → Microsoft Graph → Application permissions**
3. Add: `Directory.Read.All` (or `GroupMember.Read.All` as minimum)
4. Click **Grant admin consent for {tenant}**
5. Wait ~2 minutes for propagation, then re-run the scan

**AI Functions not available** — Your Fabric capacity doesn't have Copilot enabled.
1. Ensure you're on **Fabric F2+ or P1+ capacity** (AI features require paid capacity)
2. Check **Fabric Admin Portal → Tenant settings → Copilot and Azure OpenAI Service**
3. Enable: **Copilot and other features powered by Azure OpenAI**
4. Re-run Cell 2 — should print `AI: Azure OpenAI (Fabric Copilot) ready`
5. **Fallback**: If AI unavailable, the notebook uses template-based descriptions (still works, just less context-aware)

**Workspace not found / permission denied** — SPN needs workspace access.
1. Power BI Service → your workspace → **Manage access**
2. Add your SPN as **Viewer** (minimum) or **Contributor**
3. Wait 1-2 minutes, refresh Cell 3 dropdown

**Security Audit score = 0%** — Usually means no roles defined or TOM read failed.
- Check model has at least one security role in Power BI Desktop
- Verify SPN has access to the semantic model
- Check Verbose Logs for the actual error

**`notebookutils.credentials.getToken` hangs 60+ seconds** — Known Fabric bug.
- Fix: Enable **"Use Service Principal"** in the console and provide SPN credentials
- SPN via MSAL takes 0.3s vs the broken 60s+ hang

**Cell 5 (View Report) shows nothing** — Run Cell 3 first with Run Scan button.
- Check for `_LAST_SCAN` error — means scan didn't complete or engine not loaded
- Re-run Cell 2 to reload the engine if kernel restarted

</div>

---

## ⚡ Performance

Tested on various model sizes:

| Model Size | Tables | Measures | Scan Time | Notes |
|-----------|--------|----------|-----------|-------|
| Small | 6 | 30 | ~30s | msft_sample_mod_sm test model |
| Medium | 20 | 100 | ~60s | Typical Power BI semantic model |
| Large | 50+ | 200+ | ~90-120s | Enterprise model with many reports |

**Performance optimizations applied:**
- Data Quality: batched DAX UNION query (1 call for N columns vs N calls)
- Security X-Ray: SPN-based Graph token via MSAL (0.3s vs 60s+ `notebookutils` timeout)
- Report Visuals: filters to only reports connected to this model
- AI descriptions: parallelized via Azure OpenAI Fabric Copilot auth
- HTML rendering: chunked into 50KB blocks for Fabric's widget pipeline

---

**Built for the 2026 Semantic Link Developer Experience Challenge by Natarajan Manivasagan**

- LinkedIn: [www.linkedin.com/in/natarajan-manivasagan](https://www.linkedin.com/in/natarajan-manivasagan)
- Power BI / Fabric Community: [community.fabric.microsoft.com — user 1345926](https://community.fabric.microsoft.com/t5/user/viewprofilepage/user-id/1345926)


In [1]:
# ══════════════════════════════════════════════════════════════
#  SETUP — Install dependencies (may restart kernel once)
# ══════════════════════════════════════════════════════════════
%pip install semantic-link semantic-link-labs openai -q


StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 8, Finished, Available, Finished, True)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
nni 3.0 requires filelock<3.12, but you have filelock 3.13.1 which is incompatible.
datasets 2.19.1 requires fsspec[http]<=2024.3.1,>=2023.1.0, but you have fsspec 2024.6.1 which is incompatible.
fsspec-wrapper 0.1.15 requires PyJWT>=2.6.0, but you have pyjwt 2.4.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
# ══════════════════════════════════════════════════════════════
#  SCAN ENGINE — 13 diagnostic tools + Security X-Ray + auto-fix
#  Just click Run. Takes 1-3 minutes depending on model size.
#  See Cell 0 documentation for tool descriptions and architecture.
# ══════════════════════════════════════════════════════════════

import sempy.fabric as fabric
from sempy.fabric import set_service_principal
import sempy_labs as labs
from sempy_labs.tom import connect_semantic_model
from sempy_labs import directlake
import pandas as pd
import numpy as np
import json
import re
import warnings
warnings.filterwarnings('ignore')
from collections import Counter
import sys as _xsys
import contextlib as _xcontextlib
import warnings as _xwarnings

# Defaults if Cell 1 wasn't run or variables missing
try: SPN_TENANT_ID
except NameError: SPN_TENANT_ID = None
try: SPN_CLIENT_ID
except NameError: SPN_CLIENT_ID = None
try: SPN_CLIENT_SECRET
except NameError: SPN_CLIENT_SECRET = None
try: COMPARE_DATASET
except NameError: COMPARE_DATASET = None
try: USE_AI_DESCRIPTIONS
except NameError: USE_AI_DESCRIPTIONS = True
try: DATASET
except NameError: DATASET = None
try: WORKSPACE
except NameError: WORKSPACE = None
USE_SPN = SPN_TENANT_ID is not None and SPN_CLIENT_ID is not None
VERBOSE_LOG = False

# ── PII Masking (for public screenshots / demos) ──
MASK_PII = False  # toggled by console checkbox
_PII_USER_MAP = {}   # original → (email_pseudo, name_pseudo)
_PII_GROUP_MAP = {}  # original → "Group X"

def _reset_pii_maps():
    """Reset pseudonym maps — call at start of each scan."""
    _PII_USER_MAP.clear()
    _PII_GROUP_MAP.clear()

def _pseudo_user(original, as_email=None):
    """Generate consistent pseudonym: 'User N' and 'user6@example.com' for same input."""
    if not original:
        return ""
    s = str(original).strip()
    key = s.lower()
    if key not in _PII_USER_MAP:
        idx = len(_PII_USER_MAP) + 1
        _PII_USER_MAP[key] = (f"user{idx}@example.com", f"User {idx}")
    email_form, name_form = _PII_USER_MAP[key]
    if as_email is True:
        return email_form
    if as_email is False:
        return name_form
    return email_form if "@" in s else name_form

def _pseudo_group(original):
    """Generate consistent 'Group A', 'Group B', ... pseudonym."""
    if not original:
        return ""
    s = str(original).strip()
    key = s.lower()
    if key in _PII_GROUP_MAP:
        return _PII_GROUP_MAP[key]
    idx = len(_PII_GROUP_MAP)
    if idx < 26:
        label = chr(ord("A") + idx)
    else:
        label = chr(ord("A") + (idx // 26) - 1) + chr(ord("A") + (idx % 26))
    pseudo = f"Group {label}"
    _PII_GROUP_MAP[key] = pseudo
    return pseudo

def _mask_pii(value, kind="auto"):
    """Mask PII if MASK_PII=True, else return original unchanged.
    kind: 'email' | 'name' | 'group' | 'auto' (autodetect)
    """
    if not MASK_PII or not value:
        return value
    s = str(value).strip()
    if not s:
        return value
    # Handle prefixed strings like "AD Group: demo-ad-group-C" or "Direct" or "AD Group (unresolved): xxx"
    # Strip common source-column prefixes to get the actual identifier
    _prefix_strip = s
    _is_group_prefix = False
    for _pref in ("AD Group: ", "AD Group (unresolved): ", "Group: ", "SPN/External: "):
        if s.startswith(_pref):
            _prefix_strip = s[len(_pref):].strip()
            _is_group_prefix = True
            break
    # "Direct" source = user added directly; keep as-is (no masking needed)
    if s == "Direct" or s.startswith("Direct"):
        return s

    if kind == "group":
        return _pseudo_group(s)
    if kind == "email":
        return _pseudo_user(s, as_email=True)
    if kind == "name":
        return _pseudo_user(s, as_email=False)

    # Auto-detect
    if _is_group_prefix:
        # Return "AD Group: {pseudonym}" to preserve the source label context
        return f"AD Group: {_pseudo_group(_prefix_strip)}"
    _l = s.lower()
    if any(_l.startswith(p) for p in ("pbi-","pbi_","entp-","entp_","enterprise-","enterprise_","grp-","grp_","ad-","ad_","corp-","corp_","azure-","azure_")):
        return _pseudo_group(s)
    if s.count("-") >= 3 and " " not in s and "@" not in s:
        return _pseudo_group(s)
    if "@" in s:
        return _pseudo_user(s, as_email=True)
    return _pseudo_user(s, as_email=False)

def _vlog(tool, msg):
    """Print verbose log line if VERBOSE_LOG is enabled."""
    if VERBOSE_LOG:
        print(f"        [{tool}] {msg}")
try: _WIDGET_MODE
except NameError: _WIDGET_MODE = False
try: _tab_scores
except NameError: _tab_scores = None
try: _tab_findings
except NameError: _tab_findings = None
try: _tab_fixplan
except NameError: _tab_fixplan = None
try: _tab_security
except NameError: _tab_security = None
try: _tab_export
except NameError: _tab_export = None
try: _tab_security_audit
except NameError: _tab_security_audit = None
try: _tab_security_xray
except NameError: _tab_security_xray = None
try: _report_html
except NameError: _report_html = None
try: _EXTRACT_MEMBERS
except NameError: _EXTRACT_MEMBERS = False
try: TOOLS
except NameError: TOOLS = {k:True for k in ["copilot_readiness","lineage","test_framework","model_diff","report_visuals","security_audit","security_xray","data_quality","dax_dependencies","bpa","vertipaq","direct_lake","capacity"]}

# ── AI: OpenAI SDK with Fabric Copilot auth (no API key needed) ──
# Runs here (Cell 2) because Cell 1's %pip install restarts the kernel.
try:
    from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
    import openai as _openai_mod
    _AI_CLIENT = _openai_mod.AzureOpenAI(
        http_client=get_openai_httpx_sync_client(),
        api_version="2025-04-01-preview",
    )
    _AI_AVAILABLE = True
    print("  AI: Azure OpenAI (Fabric Copilot) ready")
except Exception as _ai_init_err:
    _AI_AVAILABLE = False
    _AI_CLIENT = None
    print(f"  AI: Not available ({type(_ai_init_err).__name__})")
    print(f"     Fallback: template-based descriptions (still work, less context-aware)")
    print(f"     To enable AI: ensure Fabric F2+ capacity with Copilot tenant switch ON")
try: AI_USER_CONTEXT
except NameError: AI_USER_CONTEXT = ""

def _ai_system_prompt():
    """Build AI system prompt optimized for Power BI semantic model documentation."""
    base = (
        "You are a Power BI semantic model documentation expert. "
        "Write SHORT 2-4 word business labels (not sentences) for Power BI tables, columns, and measures. "
        "Format: noun phrases only. Examples: Customer expiry date, Total revenue, Sales transactions, Customer billing ID. "
        "Focus on what the object represents in business terms. "
        "Synonyms: 1-3 words a business user would type in Power BI Q&A. "
        "Return ONLY the label text — no sentences, no verbs like stores or represents, no prefixes, no quotes, no JSON."
    )
    ctx = (AI_USER_CONTEXT or "").strip()
    if ctx:
        return f"{base}\n\nBusiness domain context provided by the user:\n{ctx}"
    return base

def _ai_generate(prompts_list, label=""):
    """Generate text via Azure OpenAI (Fabric Copilot auth). No API key needed.
    Returns list of response strings, or None on failure."""
    if not _AI_AVAILABLE or _AI_CLIENT is None or not prompts_list:
        return None
    import time as _t
    _t0 = _t.time()
    _vlog("AI", f"Generating {len(prompts_list)} {label} via Azure OpenAI...")
    results = []
    try:
        for idx, prompt in enumerate(prompts_list):
            try:
                resp = _AI_CLIENT.chat.completions.create(
                    model="gpt-4.1-mini",
                    messages=[
                        {"role": "system", "content": _ai_system_prompt()},
                        {"role": "user", "content": prompt}
                    ],
                    max_tokens=30,
                    temperature=0.3,
                    timeout=15,
                )
                text = resp.choices[0].message.content.strip()
                results.append(text)
                _vlog("AI", f"  [{idx+1}/{len(prompts_list)}] {text[:60]}...")
            except Exception as call_err:
                _vlog("AI", f"  [{idx+1}/{len(prompts_list)}] Failed: {str(call_err)[:60]}")
                results.append(None)
        _vlog("AI", f"Done — {len([r for r in results if r])} {label} [{_t.time()-_t0:.1f}s]")
        return results if any(r is not None for r in results) else None
    except Exception as e:
        print(f"  AI generation failed ({label}): {type(e).__name__}: {str(e)[:120]}")
        return None

# ── Weights ──
WEIGHTS = {"copilot_readiness":0.10,"lineage":0.12,"test_framework":0.13,"model_diff":0.05,
           "bpa":0.10,"vertipaq":0.08,"dax_dependencies":0.05,"data_quality":0.09,
           "direct_lake":0.05,"capacity":0.03,"report_visuals":0.08,"security_audit":0.12}

# ── Helpers ──
def score_to_grade(s):
    for threshold, grade in [(95,"A+"),(90,"A"),(85,"A-"),(80,"B+"),(75,"B"),(70,"B-"),(65,"C+"),(60,"C"),(55,"C-"),(50,"D")]:
        if s >= threshold: return grade
    return "F"

def score_color(s):
    if s >= 80: return "#107C10"
    if s >= 60: return "#C87000"
    return "#A4262C"

def make_result(name, score, details, recs=None, extra=None):
    return {"name":name,"score":round(score,1),"grade":score_to_grade(score),
            "color":score_color(score),"details":details,"recommendations":recs or [],"extra":extra or {}}

def safe_dax(n): return n.replace("'","''")

def eval_dax(ds, dax, ws=None):
    try: return fabric.evaluate_dax(ds, dax, workspace=ws)
    except: return pd.DataFrame()

def safe_int(val, default=0):
    try:
        if val is None or (hasattr(pd,'isna') and pd.isna(val)): return default
        return int(val)
    except: return default

def pct_pass(s, v="PASS"):
    return round((s==v).mean()*100,1) if len(s)>0 else 100.0

def norm_cols(df, renames):
    for old, new in renames.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old:new})
    return df

TABLE_RENAMES = {"Name":"Table Name"}


# Module-level cache of last scan — used by Cell 3 auto-restore
_LAST_SCAN = None

def _render_widgets(results, h, _all_detail_dfs, _html_path):
    """Render scan results into widget tabs using VBox.children assignment.

    Uses direct widget creation (widgets.HTML) instead of display() calls,
    because Fabric notebooks don't honor Output widget context managers.
    VBox.children assignment is synchronous and Fabric-safe.
    """
    if VERBOSE_LOG: print(f"  [RENDER] _SKIP_WIDGET_RENDER={globals().get('_SKIP_WIDGET_RENDER', '?')} _WIDGET_MODE={globals().get('_WIDGET_MODE', '?')} _tab_scores={type(globals().get('_tab_scores'))}")
    if globals().get('_SKIP_WIDGET_RENDER', False):
        return True

    import ipywidgets as _w
    if VERBOSE_LOG: print(f"  [RENDER] Starting: h={len(h)} chars, {len(results)} result keys")

    try:
        # ── Score Cards tab ──
        _scores_html = h.split('Detailed Findings')[0] if 'Detailed Findings' in h else h
        _tab_scores.children = [_w.HTML(value='<div style="font-family:Segoe UI,sans-serif">' + _scores_html + '</div>')]
        if VERBOSE_LOG: print(f"  [RENDER] Score Cards populated: {len(_scores_html)} chars")

        # ── Detailed Findings tab ──
        _findings_start = h.find('Detailed Findings')
        _fixplan_start = h.find('Fix Plan')
        if _findings_start > 0 and _fixplan_start > 0:
            _tab_findings.children = [_w.HTML(value='<div style="font-family:Segoe UI,sans-serif">' + h[_findings_start:_fixplan_start] + '</div>')]
        elif _findings_start > 0:
            _tab_findings.children = [_w.HTML(value='<div style="font-family:Segoe UI,sans-serif">' + h[_findings_start:] + '</div>')]

        # ── Fix Plan tab ──
        if _fixplan_start > 0:
            _tab_fixplan.children = [_w.HTML(value='<div style="font-family:Segoe UI,sans-serif">' + h[_fixplan_start:] + '</div>')]

        # ── Security Audit tab ──
        _sec_items = []
        if "security_audit" in results:
            _sec = results["security_audit"]
            _sec_dets = _sec.get("details", [])
            if _sec_dets:
                _sec_score = _sec["score"]
                _sec_grade = _sec["grade"]
                _sec_color = "#107C10" if _sec_score >= 80 else ("#FFB900" if _sec_score >= 60 else "#D13438")
                _sec_items.append(_w.HTML(value=(
                    '<div style="background:linear-gradient(135deg,' + _sec_color + '15,' + _sec_color + '05);'
                    'border:1px solid ' + _sec_color + '44;border-radius:10px;padding:16px;margin-bottom:12px">'
                    '<div style="font-size:13px;color:#605E5C;margin-bottom:4px">Security Audit \u2014 compliance score</div>'
                    '<div style="font-size:28px;font-weight:700;color:' + _sec_color + '">' + f"{_sec_score:.0f}% ({_sec_grade})" + '</div>'
                    '<div style="font-size:12px;color:#605E5C;margin-top:4px">RLS coverage, OLS presence, role hygiene, member assignment</div>'
                    '</div>'
                )))
                _sec_df = pd.DataFrame([{"Status":d["status"],"Category":d.get("category",""),"Item":d["item"],"Info":d.get("info","")} for d in _sec_dets])
                _sec_items.append(_w.HTML(value='<h4 style="margin:12px 0 6px 0">Findings</h4>' + _sec_df.to_html(index=False, classes='table', border=0)))
                _sec_recs = _sec.get("recommendations", []) or []
                if _sec_recs:
                    _recs_html = '<h4 style="margin:16px 0 6px 0">Recommendations</h4><ul style="margin:4px 0 0 18px;padding:0;font-size:13px;line-height:1.6">'
                    for _rec in _sec_recs:
                        _recs_html += '<li>' + str(_rec) + '</li>'
                    _recs_html += '</ul>'
                    _sec_items.append(_w.HTML(value=_recs_html))
            else:
                _sec_items.append(_w.HTML(value='<p style="color:#605E5C">Security Audit ran but produced no findings.</p>'))
        else:
            _sec_items.append(_w.HTML(value='<p style="color:#605E5C">Security Audit not run. Enable the checkbox and re-run.</p>'))
        _tab_security_audit.children = _sec_items

        # ── Security X-Ray tab ──
        _xr_items = []
        if "security_xray" in results:
            _xr = results["security_xray"]
            if not _xr.get("extra",{}).get("skipped",False):
                _xray = _xr.get("extra",{}).get("xray",{}) or {}
                _xr_score = _xr["score"]
                _xr_grade = _xr["grade"]
                _xr_color = "#107C10" if _xr_score >= 80 else ("#FFB900" if _xr_score >= 60 else "#D13438")
                _eff = _xray.get("effective", []) or []
                _rls = _xray.get("rls", []) or []
                _ols = _xray.get("ols", []) or []
                _mem = _xray.get("members", []) or []
                _wsa = _xray.get("workspace_access", []) or []
                _n_bypass = sum(1 for _r in _eff if "bypass" in str(_r.get("Effective_View","")).lower())
                _n_dyn = len(_xray.get("dynamic_roles", []) or [])
                _graph_ok = _xray.get("graph_available", False)
                _rest_ok = _xray.get("fabric_rest_available", False)
                # Header card
                _xr_items.append(_w.HTML(value=(
                    '<div style="background:linear-gradient(135deg,' + _xr_color + '15,' + _xr_color + '05);'
                    'border:1px solid ' + _xr_color + '44;border-radius:10px;padding:16px;margin-bottom:12px">'
                    '<div style="font-size:13px;color:#605E5C;margin-bottom:4px">Security X-Ray \u2014 effective access map</div>'
                    '<div style="font-size:28px;font-weight:700;color:' + _xr_color + '">' + f"{_xr_score:.0f}% ({_xr_grade})" + '</div>'
                    '<div style="font-size:12px;color:#605E5C;margin-top:4px">Who can actually see what \u2014 RLS + OLS + AAD group expansion + workspace roles</div>'
                    '</div>'
                )))
                # Stat cards
                _card = lambda lbl, val, color="#1B3A4B": (
                    '<div style="flex:1 1 0;background:#fff;border:1px solid #EDEBE9;border-radius:8px;padding:10px 14px;margin:0 4px">'
                    '<div style="font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:0.5px">' + lbl + '</div>'
                    '<div style="font-size:20px;font-weight:700;color:' + color + ';margin-top:2px">' + str(val) + '</div></div>'
                )
                _cards = '<div style="display:flex;margin:0 -4px 12px -4px;flex-wrap:wrap">'
                _cards += _card("Effective Users", len({_r.get("User","") for _r in _eff}))
                _cards += _card("RLS Filters", len(_rls))
                _cards += _card("OLS Restrictions", len(_ols))
                _cards += _card("Workspace Assignments", len(_wsa))
                _cards += _card("Admin Bypass", _n_bypass, "#D13438" if _n_bypass > 0 else "#107C10")
                _cards += _card("Dynamic Roles", _n_dyn, "#FFB900" if _n_dyn > 0 else "#605E5C")
                _cards += '</div>'
                _xr_items.append(_w.HTML(value=_cards))
                # Availability
                _av = '<div style="font-size:12px;color:#605E5C;margin-bottom:12px;padding:6px 10px;background:#FAF9F8;border-radius:6px;border-left:3px solid #0078D4">'
                _av += "Graph API: " + ("<span style='color:#107C10;font-weight:600'>available</span>" if _graph_ok else "<span style='color:#FFB900;font-weight:600'>unavailable</span>")
                _av += " &middot; Fabric REST: " + ("<span style='color:#107C10;font-weight:600'>available</span>" if _rest_ok else "<span style='color:#FFB900;font-weight:600'>unavailable</span>")
                if _xray.get("dynamic_roles"):
                    _av += " &middot; Dynamic RLS: " + ", ".join(_xray["dynamic_roles"])
                _av += "</div>"
                _xr_items.append(_w.HTML(value=_av))
                # DataFrames as HTML tables
                if _eff:
                    _xr_items.append(_w.HTML(value='<h4 style="margin:16px 0 6px 0">Effective Access Map \u2014 who sees what</h4>' + pd.DataFrame(_eff).to_html(index=False, border=0)))
                if _rls:
                    _xr_items.append(_w.HTML(value='<h4 style="margin:16px 0 6px 0">RLS Filters</h4>' + pd.DataFrame(_rls).to_html(index=False, border=0)))
                if _ols:
                    _xr_items.append(_w.HTML(value='<h4 style="margin:16px 0 6px 0">OLS Restrictions</h4>' + pd.DataFrame(_ols).to_html(index=False, border=0)))
                if _wsa:
                    _xr_items.append(_w.HTML(value='<h4 style="margin:16px 0 6px 0">Workspace Role Assignments</h4>' + pd.DataFrame(_wsa).to_html(index=False, border=0)))
                if _mem:
                    _xr_items.append(_w.HTML(value='<h4 style="margin:16px 0 6px 0">Raw Role Members</h4>' + pd.DataFrame(_mem).to_html(index=False, border=0)))
                if not any([_eff,_rls,_ols,_mem,_wsa]):
                    _xr_items.append(_w.HTML(value='<p style="color:#605E5C">No Security X-Ray data captured.</p>'))
            else:
                _xr_items.append(_w.HTML(value='<p style="color:#605E5C">Security X-Ray was skipped.</p>'))
        else:
            _xr_items.append(_w.HTML(value='<p style="color:#605E5C">Security X-Ray not run. Enable the checkbox and re-run.</p>'))
        _tab_security_xray.children = _xr_items
        if VERBOSE_LOG: print(f"  [RENDER] X-Ray tab: {len(_xr_items)} items")

        # ── Export tab ──
        _exp_items = []
        if _html_path:
            _exp_items.append(_w.HTML(value=f'<p>\u2705 HTML report saved: <code>{_html_path}</code></p>'))
        _exp_items.append(_w.HTML(value='<p style="color:#605E5C">Open the HTML file in a browser for collapsible sections + print to PDF.</p>'))
        _tab_export.children = _exp_items
        if VERBOSE_LOG: print(f"  [RENDER] All tabs populated OK")

    except Exception as _we:
        import traceback as _tb
        # If rendering fails, show error in the scores tab
        print(_tb.format_exc()[:500])
        print(f"  [RENDER] EXCEPTION: {_we}")
        _tab_scores.children = [_w.HTML(value=f'<pre style="color:#D13438">Widget rendering error:\n{_tb.format_exc()[:1000]}</pre>')]

    # Render HTML via base64 iframe — bypasses all Fabric widget quirks.
    # The widget only receives a tiny iframe tag. Browser loads the full 181KB
    # content independently from the data URL, avoiding widget size/re-render issues.
    try:
        import base64 as _b64_mod
        _b64 = _b64_mod.b64encode(h.encode('utf-8')).decode()
        _iframe = (
            f'<iframe src="data:text/html;charset=utf-8;base64,{_b64}" '
            f'style="width:100%;height:800px;border:1px solid #EDEBE9;'
            f'border-radius:8px;background:#fff"></iframe>'
        )
        try:
            _report_html.value = _iframe
        except NameError:
            # Cell 3 not loaded — fall back to displayHTML with the raw HTML
            try:
                displayHTML(h)
            except Exception:
                from IPython.display import display as _d, HTML as _H
                _d(_H(h))
    except Exception as _re_err:
        # Last resort: show error in widget or stdout
        try:
            _report_html.value = f'<pre style="color:#D13438;padding:12px">Render error: {_re_err}</pre>'
        except NameError:
            print(f"Render error: {_re_err}")

    return True


def _install_logging_filters():
    try:
        _root = _xlogging.getLogger()
        # Attach to root logger itself
        if _NOISE_LOGGING_FILTER not in _root.filters:
            _root.addFilter(_NOISE_LOGGING_FILTER)
        # Attach to every existing handler on root + sempy loggers
        for _lg_name in ["", "sempy", "sempy.fabric", "sempy.fabric._xmla",
                         "sempy_labs", "Microsoft.AnalysisServices",
                         "Microsoft.AnalysisServices.AdomdClient"]:
            _lg = _xlogging.getLogger(_lg_name)
            if _NOISE_LOGGING_FILTER not in _lg.filters:
                _lg.addFilter(_NOISE_LOGGING_FILTER)
            for _h in list(_lg.handlers):
                if _NOISE_LOGGING_FILTER not in _h.filters:
                    _h.addFilter(_NOISE_LOGGING_FILTER)
        # Also suppress via the `warnings` module
        for _pat in ["is serialized as string", "cast to desired type"]:
            _xwarnings.filterwarnings("ignore", message=f".*{_pat}.*")
    except Exception:
        pass

# Fire now so filters apply immediately (not just inside the context manager).
_install_logging_filters()

@_xcontextlib.contextmanager
def _silence_noisy_warnings():
    """Context manager: filter DAX/ADOMD spam from stdout+stderr during run_scan.

    Note: logging/warnings filters are already installed at module load by
    _install_logging_filters() — this context manager only adds the extra
    stdout/stderr stream filter for any direct writes that bypass logging.
    """
    _orig_out, _orig_err = _xsys.stdout, _xsys.stderr
    _xsys.stdout = _FilteredStream(_orig_out)
    _xsys.stderr = _FilteredStream(_orig_err)
    # Re-install logging filters in case new handlers were added by sempy between
    # calls (idempotent)
    _install_logging_filters()
    # OS-level: redirect fd 2 (real stderr) to devnull
    import os as _os
    _saved_fd2 = None
    _devnull_fd = None
    try:
        _saved_fd2 = _os.dup(2)
        _devnull_fd = _os.open(_os.devnull, _os.O_WRONLY)
        _os.dup2(_devnull_fd, 2)
    except Exception:
        _saved_fd2 = None
    try:
        yield
    finally:
        # Restore OS stderr first
        if _saved_fd2 is not None:
            try:
                _os.dup2(_saved_fd2, 2)
                _os.close(_saved_fd2)
            except Exception:
                pass
        if _devnull_fd is not None:
            try:
                _os.close(_devnull_fd)
            except Exception:
                pass
        try:
            _xsys.stdout.flush()
            _xsys.stderr.flush()
        except Exception:
            pass
        _xsys.stdout, _xsys.stderr = _orig_out, _orig_err

# ── Preflight ──
def preflight(dataset, workspace=None):
    datasets = fabric.list_datasets(workspace=workspace)
    ds_col = "Dataset Name" if "Dataset Name" in datasets.columns else "Name"
    if dataset not in datasets[ds_col].values:
        print(f"ERROR: '{dataset}' not found. Check DATASET in Cell 1.")
        return None
    tables = norm_cols(fabric.list_tables(dataset, extended=True, workspace=workspace), TABLE_RENAMES)
    columns = fabric.list_columns(dataset, extended=True, workspace=workspace)
    measures = fabric.list_measures(dataset, workspace=workspace)
    relationships = fabric.list_relationships(dataset, extended=True, workspace=workspace)
    try: hierarchies = fabric.list_hierarchies(dataset, workspace=workspace)
    except: hierarchies = pd.DataFrame()
    try:
        roles = fabric.get_roles(dataset, workspace=workspace)
        rls = fabric.get_row_level_security_permissions(dataset, workspace=workspace)
    except:
        roles = pd.DataFrame()
        rls = pd.DataFrame()
    return {"tables":tables,"columns":columns,"measures":measures,
            "relationships":relationships,"hierarchies":hierarchies,"roles":roles,"rls":rls}

# ── Tool 1: Test Framework ──
def tool_test_framework(ds, meta, ws=None):
    """Validate that every DAX measure evaluates without error.

    Also validates relationships: cardinality (one-to-many, many-to-one,
    many-to-many), cross-filter direction (single/both), and active status.
    Broken measures and many-to-many relationships are flagged as WARN/FAIL.

    Returns: make_result with pass-rate score. Each measure row has its
    actual sample output value in the info field.
    """
    _vlog("Test", "Starting Test Framework scan...")
    details, recs = [], []
    mdf = meta["measures"]
    # Measure tests
    mresults = []
    total_m = len(mdf)
    for idx, (_, m) in enumerate(mdf.iterrows()):
        if total_m > 20 and idx % 50 == 0 and idx > 0:
            print(f"\n        Measures tested: {idx}/{total_m}...", end="", flush=True)
        t, n = m["Table Name"], m["Measure Name"]
        dax = f"EVALUATE ROW(\"R\", '{safe_dax(t)}'[{safe_dax(n)}])"
        row = {"table":t,"measure":n}
        try:
            df = fabric.evaluate_dax(ds, dax, workspace=ws)
            v = df.iloc[0,0] if len(df)>0 else None
            row.update({"value":str(v) if v is not None else "NULL","status":"PASS"})
        except Exception as e:
            emsg = str(e)[:200]
            # AdomdCommand errors on Direct Lake = needs refresh, not a real measure error
            if "AdomdCommand" in emsg or "connection" in emsg.lower():
                row.update({"value":None,"status":"WARN","error":"Direct Lake — query failed (try refreshing model)"})
            else:
                row.update({"value":None,"status":"FAIL","error":emsg})
        mresults.append(row)
    mr = pd.DataFrame(mresults)
    _vlog("Test", "Validating relationships...")
    # Relationships
    rr = []
    for _, rel in meta["relationships"].iterrows():
        _mult = str(rel.get("Multiplicity","")).strip() or "?"
        _xfilter = str(rel.get("Cross Filtering Behavior","")).strip() or "single"
        _active = rel.get("Is Active", True)
        # Normalize multiplicity to human-readable
        _mult_lower = _mult.lower()
        if "manytoone" in _mult_lower.replace("-","").replace(" ",""):
            _mult_display = "many-to-one"
        elif "onetomany" in _mult_lower.replace("-","").replace(" ",""):
            _mult_display = "one-to-many"
        elif "onetoone" in _mult_lower.replace("-","").replace(" ",""):
            _mult_display = "one-to-one"
        elif "manytomany" in _mult_lower.replace("-","").replace(" ",""):
            _mult_display = "many-to-many"
        else:
            _mult_display = _mult
        _xfilter_display = "both-directions" if "both" in _xfilter.lower() else "single-direction"
        _active_display = "active" if _active else "INACTIVE"
        _info_parts = [_mult_display, _xfilter_display, _active_display]
        row = {"item":f"{rel['From Table']}[{rel['From Column']}] -> {rel['To Table']}[{rel['To Column']}]","status":"PASS","info":" | ".join(_info_parts)}
        issues = []
        if "many-to-many" == _mult_display: issues.append("Many-to-many (consider star schema)")
        if "both" in _xfilter.lower(): issues.append("Bi-directional filter (performance risk)")
        if not _active: issues.append("Inactive (must be activated via USERELATIONSHIP)")
        if issues:
            row["status"] = "WARN"
            row["info"] = " | ".join(_info_parts) + " | Issues: " + "; ".join(issues)
        rr.append(row)
    rd = pd.DataFrame(rr)
    # Score
    scores = []
    if len(mr)>0: scores.append(pct_pass(mr["status"]))
    if len(rd)>0: scores.append(pct_pass(rd["status"]))
    score = np.mean(scores) if scores else 100.0
    for _,r in mr.iterrows():
        info = r.get("value","") if r["status"]=="PASS" else r.get("error","Eval failed")
        details.append({"category":"Measure","item":f"{r['table']}.{r['measure']}","status":r["status"],"info":str(info)[:80]})
    for _,r in rd.iterrows():
        details.append({"category":"Relationship","item":r["item"],"status":r["status"],"info":r.get("info","")})
    if len(mr)>0:
        fails = mr[mr["status"]=="FAIL"]
        if len(fails)>0: recs.append(f"Fix {len(fails)} erroring measure(s)")
    warns = rd[rd["status"]=="WARN"] if len(rd)>0 else pd.DataFrame()
    if len(warns)>0: recs.append(f"Review {len(warns)} relationship warning(s)")
    return make_result("Test Framework",score,details,recs)

# ── Tool 2: Copilot Readiness ──
def tool_copilot(ds, meta, ws=None):
    """Scan for Copilot/Q&A readiness.

    Checks: table/column/measure descriptions, synonyms (en-US culture),
    naming conventions, hidden key columns, format strings on measures,
    and AI Prep configuration (CopilotTooling, CustomInstructions, schema
    simplified via isAvailableInMDX).

    Returns: make_result with weighted score (0-100), details per check
    category, and recs for the top missing items.
    """
    _vlog("Copilot", "Starting Copilot Readiness scan...")
    tdf, cdf, mdf = meta["tables"], meta["columns"], meta["measures"]
    checks, details, recs = [], [], []
    def _chk(name, df, col, max_pts, missing_fn):
        if col in df.columns:
            has = df[col].notna() & (df[col]!="")
            pct = has.mean()*100
            missing = missing_fn(df[~has])
            checks.append({"check":name,"score":pct,"max":max_pts,"missing":missing})
            if pct<100: recs.append(f"Add {col.lower()} to {len(missing)} item(s) ({name})")
    _chk("Table Descriptions", tdf, "Description", 10, lambda d: d["Table Name"].tolist())
    _cdf_clean = cdf[~cdf["Column Name"].str.contains("RowNumber-|rowNumber-",na=False)] if "Column Name" in cdf.columns else cdf
    _chk("Column Descriptions", _cdf_clean, "Description", 15, lambda d: d[["Table Name","Column Name"]].values.tolist()[:20])
    _chk("Measure Descriptions", mdf, "Description", 15, lambda d: d["Measure Name"].tolist() if "Measure Name" in d.columns else [])
    _vlog("Copilot", "Checking naming conventions...")
    # Naming

    tnames = list(tdf["Table Name"]) if "Table Name" in tdf.columns else []
    cnames = list(_cdf_clean["Column Name"]) if "Column Name" in _cdf_clean.columns else []
    mnames = list(mdf["Measure Name"]) if "Measure Name" in mdf.columns else []
    all_n = tnames+cnames+mnames
    bad = [n for n in all_n if re.match(r'^[a-z].*[A-Z]',n) or '_' in n]
    checks.append({"check":"Naming","score":max(0,(1-len(bad)/max(len(all_n),1))*100),"max":10,"missing":bad[:10]})
    _vlog("Copilot", "Checking hidden key columns...")
    if "Is Hidden" in cdf.columns:
        keys = cdf[cdf["Column Name"].str.contains(r'(?i)(key|sk|fk|id)$',regex=True)]
        if len(keys)>0:
            hp = keys["Is Hidden"].mean()*100
            nh = keys[~keys["Is Hidden"]][["Table Name","Column Name"]].values.tolist()
            checks.append({"check":"Key Cols Hidden","score":hp,"max":10,"missing":nh[:10]})
            if hp<100: recs.append(f"Hide {len(nh)} key/ID column(s)")
        else:
            checks.append({"check":"Key Cols Hidden","score":100,"max":10,"missing":[]})
    _vlog("Copilot", "Checking format strings...")
    if "Format String" in mdf.columns:
        has = mdf["Format String"].notna() & (mdf["Format String"]!="")
        checks.append({"check":"Format Strings","score":has.mean()*100,"max":10,"missing":mdf[~has]["Measure Name"].tolist()})
    # ── Prep data for AI checks ──
    _copilot_enabled = False
    _has_ai_instructions = False
    _schema_simplified = False
    try:
        with connect_semantic_model(ds, readonly=True, workspace=ws) as _tom_ai:
            # Check 1: CopilotTooling enabled?
            for ann in _tom_ai.model.Annotations:
                if str(ann.Name) == "demo-ad-group-D":
                    _copilot_enabled = "CopilotTooling" in str(ann.Value)
                if str(ann.Name) in ("__PBI_CopilotInstructions", "QueryInsightsInstructions"):
                    _has_ai_instructions = bool(str(ann.Value).strip())
            # Check 2: Schema simplified? (any columns with isAvailableInMDX=False)
            _mdx_excluded = 0
            _total_visible = 0
            for table in _tom_ai.model.Tables:
                if table.IsHidden: continue
                for col in table.Columns:
                    if str(col.Type) == "RowNumber": continue
                    if col.IsHidden: continue
                    _total_visible += 1
                    try:
                        if not col.IsAvailableInMDX:
                            _mdx_excluded += 1
                    except: pass
            _schema_simplified = _mdx_excluded > 0
    except Exception:
        pass

    # Score the AI prep
    _ai_prep_score = 0
    _ai_prep_max = 15
    if _copilot_enabled: _ai_prep_score += 5
    else: recs.append("Enable 'Prep data for AI' in semantic model settings")
    if _has_ai_instructions: _ai_prep_score += 5
    else: recs.append("Add AI instructions via Prep for AI UI (Settings > Prep data for AI > Add AI Instructions). Cannot be set programmatically.")
    if _schema_simplified: _ai_prep_score += 5
    else: recs.append("Simplify schema: exclude technical columns (keys, lineage) from Copilot via isAvailableInMDX")
    checks.append({"check":"AI Prep","score":(_ai_prep_score/_ai_prep_max)*100,"max":_ai_prep_max,
        "missing":[x for x in [
            None if _copilot_enabled else "Copilot not enabled",
            None if _has_ai_instructions else "No AI instructions",
            None if _schema_simplified else "Schema not simplified"
        ] if x]})
    # Show current AI instructions if they exist
    _current_instructions = ""
    try:
        with connect_semantic_model(ds, readonly=True, workspace=ws) as _tom_instr:
            for _c in _tom_instr.model.Cultures:
                if str(_c.Name) == "en-US" and _c.LinguisticMetadata and _c.LinguisticMetadata.Content:
                    _lm = json.loads(str(_c.LinguisticMetadata.Content))
                    _current_instructions = _lm.get("CustomInstructions", "")
                    break
    except Exception:
        pass
    if _current_instructions:
        details.append({"category":"AI Prep","item":"Current AI Instructions","status":"PASS",
            "info":_current_instructions[:200]})
    details.append({"category":"AI Prep - Copilot Config","item":f"Copilot tenant setting",
                    "status":"PASS" if _copilot_enabled else "WARN",
                    "info":f"CopilotTooling enabled: {'Yes' if _copilot_enabled else 'No — enable in tenant settings'}"})
    details.append({"category":"AI Prep - Instructions","item":"Model-level AI instructions",
                    "status":"PASS" if _has_ai_instructions else "WARN",
                    "info":f"CustomInstructions configured: {'Yes' if _has_ai_instructions else 'No — auto-generate in fix plan'}"})
    details.append({"category":"AI Prep - Schema Simplification","item":"Technical columns hidden from Copilot",
                    "status":"PASS" if _mdx_excluded > 0 else "INFO",
                    "info":f"{_mdx_excluded}/{_total_visible} cols excluded via isAvailableInMDX (prevents Copilot from analyzing technical keys)"})

    # Keep legacy summary line for backward compat (but with simpler text)
    details.append({"category":"AI Prep","item":f"Overall AI Prep status",
        "status":"PASS" if _ai_prep_score >= 10 else ("WARN" if _ai_prep_score >= 5 else "FAIL"),
        "info":f"{_ai_prep_score}/{_ai_prep_max} pts"})

    # Synonym coverage via TOM
    syn = 0
    try:
        with connect_semantic_model(ds, readonly=True, workspace=ws) as tom:
            tot, ws2 = 0, 0
            for table in tom.model.Tables:
                if table.IsHidden: continue
                tot += 1
                for culture in tom.model.Cultures:
                    if culture.Name=="en-US" and culture.ObjectTranslations.Count>0: ws2+=1; break
            syn = (ws2/max(tot,1))*100
    except: pass
    checks.append({"check":"Synonyms","score":syn,"max":15,"missing":[]})
    if syn<50: recs.append("Add synonyms for Copilot")
    # Score
    tp = sum(c["score"]/100*c["max"] for c in checks)
    mp = sum(c["max"] for c in checks)
    score = (tp/mp*100) if mp>0 else 0
    for c in checks:
        st = "PASS" if c["score"]>=80 else ("WARN" if c["score"]>=50 else "FAIL")
        # Build meaningful info: show missing items for FAIL/WARN, summary for PASS
        if c["missing"]:
            # Show first 2-3 missing items, truncated
            _miss = c["missing"]
            if isinstance(_miss, list) and len(_miss) > 0:
                _first = _miss[0] if not isinstance(_miss[0], (list, tuple)) else ".".join(str(x) for x in _miss[0])
                _info = f"{len(_miss)} missing: {str(_first)[:40]}"
                if len(_miss) > 1:
                    _info += f", +{len(_miss)-1} more"
            else:
                _info = "Items missing"
        else:
            # For PASS — show what's present
            if c["score"] >= 100:
                _info = "All items covered"
            elif c["score"] > 0:
                _info = f"{c['score']:.0f}% coverage (no missing items tracked)"
            else:
                _info = "No items to check"
        details.append({"category":c["check"],"item":f"{c['score']:.0f}% ({c['score']/100*c['max']:.1f}/{c['max']} pts)",
                       "status":st,"info":_info})
        # Expand missing items so they show in detailed findings
        if c["missing"]:
            for mi in c["missing"][:15]:
                if isinstance(mi, list):
                    lbl = ".".join(str(x) for x in mi)
                else:
                    lbl = str(mi)
                details.append({"category":c["check"],"item":lbl,"status":"FAIL","info":"Missing"})
    return make_result("Copilot Readiness",score,details,recs)

# ── Tool 3: Model Diff ──
def tool_diff(ds, compare_ds, meta, ws=None, cws=None):
    _vlog("Diff", "Starting Model Diff...")
    if not compare_ds:
        return make_result("Model Diff",100,[],["Set COMPARE_DATASET to enable"],extra={"skipped":True})
    mb = {"tables":norm_cols(fabric.list_tables(compare_ds,extended=True,workspace=cws),TABLE_RENAMES),
          "columns":fabric.list_columns(compare_ds,extended=True,workspace=cws),
          "measures":fabric.list_measures(compare_ds,workspace=cws)}
    def _diff(a,b,key,cols=None):
        diffs=[]
        ka=set(a[key].tolist()) if key in a.columns else set()
        kb=set(b[key].tolist()) if key in b.columns else set()
        for k in kb-ka: diffs.append({"name":k,"change":"Only in compare"})
        for k in ka-kb: diffs.append({"name":k,"change":"New (not in compare)"})
        if cols:
            for k in ka&kb:
                ra,rb=a[a[key]==k].iloc[0],b[b[key]==k].iloc[0]
                ch=[c for c in cols if c in a.columns and c in b.columns and str(ra.get(c,""))!=str(rb.get(c,""))]
                if ch: diffs.append({"name":k,"change":"Modified","fields":ch})
        return diffs
    ad={}
    ad["tables"]=_diff(meta["tables"],mb["tables"],"Table Name",["Description"])
    ad["columns"]=_diff(meta["columns"],mb["columns"],"Column Name",["Data Type"])
    ad["measures"]=_diff(meta["measures"],mb["measures"],"Measure Name",["Measure Expression"])
    tc=sum(len(d) for d in ad.values())
    to=len(meta["tables"])+len(meta["columns"])+len(meta["measures"])
    score=max(0,round((1-tc/max(to,1))*100,1))
    details=[{"category":ot.title(),"item":d["name"],
             "status":"INFO" if "New" in d["change"] else "WARN",
             "info":d["change"]+(" ("+", ".join(d["fields"])+")" if "fields" in d else "")}
             for ot,diffs in ad.items() for d in diffs]
    return make_result("Model Diff",score,details,[f"{tc} difference(s)"] if tc>0 else [])

# ── Tool 4: BPA ──
def tool_bpa(ds, ws=None):
    _vlog("BPA", "Running Best Practice Analyzer (sempy_labs)...")
    details, recs = [], []
    # Get available rules
    try:
        rules_df = labs.model_bpa_rules()
        rule_count = len(rules_df)
        rc_col = next((c for c in ["Rule Name","rule_name","Rule"] if c in rules_df.columns), rules_df.columns[0])
        cat_col = next((c for c in ["Category","category"] if c in rules_df.columns), None)
        rule_names = rules_df[rc_col].tolist()
    except Exception:
        rules_df = None
        rule_count = 0
        rule_names = []
    try:
        bpa = labs.run_model_bpa(dataset=ds, workspace=ws, return_dataframe=True)
        rc = next((c for c in ["Rule Name","rule_name","Rule"] if c in bpa.columns),bpa.columns[0])
        sc = next((c for c in ["Severity","severity","Category"] if c in bpa.columns),None)
        oc = next((c for c in ["Object Name","object_name","Object"] if c in bpa.columns),None)
        violation_names = set(bpa[rc].tolist()) if len(bpa) > 0 else set()
        # Show violations
        for _,r in bpa.iterrows():
            sev=str(r.get(sc,"Info")) if sc else "Info"
            st="FAIL" if sev.lower() in ["error","high","critical"] else ("WARN" if sev.lower() in ["warning","medium"] else "INFO")
            details.append({"category":sev,"item":str(r[rc]),"status":st,"info":str(r[oc])[:80] if oc else ""})
        # Show passed rules
        for rn in rule_names:
            if rn not in violation_names:
                details.append({"category":"Passed","item":rn,"status":"PASS","info":"No violations"})
        e=len([d for d in details if d["status"]=="FAIL"])
        w=len([d for d in details if d["status"]=="WARN"])
        p=len([d for d in details if d["status"]=="PASS"])
        score=max(0,100-e*5-w*2)
        if e>0: recs.append(f"Fix {e} error(s)")
        if w>0: recs.append(f"Address {w} warning(s)")
        recs.append(f"{rule_count} rules evaluated, {p} passed, {len(violation_names)} violated")
        return make_result("Best Practices",score,details,recs)
    except Exception as ex:
        msg = str(ex)[:200]
        if "vectorize" in msg.lower() or "size 0" in msg.lower():
            # BPA ran but returned empty — means all rules passed
            for rn in rule_names:
                details.append({"category":"Passed","item":rn,"status":"PASS","info":"No violations"})
            recs.append(f"{rule_count} rules evaluated, all passed")
            return make_result("Best Practices",100,details,recs)
        return make_result("Best Practices",0,[],[f"BPA failed: {msg}"])

# ── Tool 5: Vertipaq ──
def tool_vertipaq(ds, ws=None):
    _vlog("Vertipaq", "Running Vertipaq Analyzer (sempy_labs)...")
    try:
        import io as _io, contextlib as _ctx
        _buf = _io.StringIO()
        with _ctx.redirect_stdout(_buf):
            vpax = labs.vertipaq_analyzer(dataset=ds, workspace=ws)
        tdf = None
        if isinstance(vpax,dict):
            for k in ["Tables","tables","Table"]:
                if k in vpax: tdf=vpax[k]; break
            if tdf is None and len(vpax)>0: tdf=list(vpax.values())[0]
        elif isinstance(vpax,pd.DataFrame): tdf=vpax
        if tdf is None or len(tdf)==0: return make_result("Vertipaq Performance",80,[],["No data"])
        sc2=next((c for c in tdf.columns if "size" in c.lower() or "byte" in c.lower()),None)
        nc=next((c for c in tdf.columns if "table" in c.lower() or "name" in c.lower()),tdf.columns[0])
        details=[]
        if sc2:
            total=tdf[sc2].sum()
            for _,r in tdf.sort_values(sc2,ascending=False).head(15).iterrows():
                pct=(r[sc2]/total*100) if total>0 else 0
                st="WARN" if pct>30 else ("INFO" if pct>15 else "PASS")
                sz=r[sc2]/(1024*1024) if r[sc2]>1024*1024 else r[sc2]/1024
                u="MB" if r[sc2]>1024*1024 else "KB"
                details.append({"category":"Table","item":str(r[nc]),"status":st,"info":f"{sz:.1f} {u} ({pct:.1f}%)"})
            top_pct=(tdf[sc2].max()/total*100) if total>0 else 0
            score=max(0,100-max(0,top_pct-30)*2)
        else: score=80
        recs=[f"Top table uses {top_pct:.0f}% of memory"] if sc2 and top_pct>40 else []
        return make_result("Vertipaq Performance",score,details,recs)
    except Exception as ex:
        return make_result("Vertipaq Performance",0,[],[f"Failed: {str(ex)[:150]}"])

# ── Tool 6: DAX Dependencies ──
def tool_dax_deps(ds, meta, ws=None):
    """DAX measure dependency graph analysis.

    Parses measure expressions for 'Table'[Column] and [Measure] refs.
    Emits two categories:
    - Complexity: measures sorted by number of dependencies (top 15)
    - Blast Radius: most-referenced objects (top 10, fan-out risk)

    High complexity (>10 deps) and high fan-out (>5 dependents) are WARN —
    change impact is wide. Low deps are PASS.

    Returns: make_result with score penalized for complex and high-fan-out measures.
    """
    _vlog("DAXDeps", "Parsing DAX dependency graph...")
    mdf = meta["measures"]
    if len(mdf)==0: return make_result("DAX Dependencies",100,[],["No measures"])
    details, recs = [], []
    deps, rdeps = {}, {}
    ec = next((c for c in ["Measure Expression","Expression"] if c in mdf.columns),None)
    if ec:
        for _,m in mdf.iterrows():
            mn=m["Measure Name"]; expr=str(m.get(ec,""))
            refs=set()
            for match in re.finditer(r"'([^']+)'\[([^\]]+)\]",expr): refs.add(f"{match.group(1)}.{match.group(2)}")
            for match in re.finditer(r"(?<!')\[([^\]]+)\]",expr):
                rn=match.group(1)
                if rn in mdf["Measure Name"].values: refs.add(f"(m).{rn}")
            deps[mn]=refs
            for ref in refs: rdeps.setdefault(ref,set()).add(mn)
    for mn,refs in sorted(deps.items(),key=lambda x:len(x[1]),reverse=True)[:15]:
        dc=len(refs); st="WARN" if dc>10 else ("INFO" if dc>5 else "PASS")
        # Show actual dep names (first 3, truncated)
        _dep_list = sorted(list(refs))[:3]
        _dep_str = ", ".join(_dep_list)
        if dc > 3:
            _dep_str += f" (+{dc-3} more)"
        _info = f"{dc} deps: {_dep_str}" if dc > 0 else "No dependencies (constant)"
        details.append({"category":"Complexity","item":mn,"status":st,"info":_info})
    for obj,ms in sorted(rdeps.items(),key=lambda x:len(x[1]),reverse=True)[:10]:
        dc=len(ms); st="WARN" if dc>5 else ("INFO" if dc>2 else "PASS")
        # Show actual measures using this object
        _user_list = sorted(list(ms))[:3]
        _user_str = ", ".join(_user_list)
        if dc > 3:
            _user_str += f" (+{dc-3} more)"
        _info = f"Used by {dc}: {_user_str}"
        details.append({"category":"Blast Radius","item":obj,"status":st,"info":_info})
    hc=len([m for m,r in deps.items() if len(r)>10])
    hb=len([o for o,d in rdeps.items() if len(d)>5])
    score=max(0,100-hc*3-hb*2)
    if hc>0: recs.append(f"{hc} measure(s) >10 deps")
    if hb>0: recs.append(f"{hb} high blast radius object(s)")
    return make_result("DAX Dependencies",score,details,recs)

# ── Tool 7: Data Quality ──
def tool_data_quality(ds, meta, ws=None):
    """Referential integrity + null rate analysis.

    1. fabric.list_relationship_violations for ref integrity issues
    2. Batched DAX UNION query for null counts on 10 sampled columns
       (RowNumber system columns filtered out)

    Uses ONE DAX call per scan (batched) rather than N per-column calls
    for ~10x faster performance.

    Returns: make_result with composite score across integrity + null + cardinality.
    """
    _vlog("DQ", "Starting Data Quality scan...")
    import time as _time
    details, recs, sub = [], [], []

    # ── Step 1: Referential integrity ──
    print(f"        [DQ] Step 1/2: Checking referential integrity...")
    _t0 = _time.time()
    try:
        try:
            viol = fabric.list_relationship_violations(ds, workspace=ws)
        except TypeError:
            # Older sempy versions don't accept workspace param
            viol = fabric.list_relationship_violations(ds)
        if len(viol) > 0:
            for _, v in viol.head(15).iterrows():
                details.append({"category": "Ref Integrity", "item": str(v.iloc[0]), "status": "WARN", "info": ""})
            sub.append(max(0, 100 - len(viol) * 10))
            recs.append(f"{len(viol)} violation(s)")
            print(f"        [DQ] Step 1/2: {len(viol)} violation(s) [{_time.time()-_t0:.1f}s]")
        else:
            sub.append(100)
            details.append({"category": "Ref Integrity", "item": "All", "status": "PASS", "info": "No violations"})
            print(f"        [DQ] Step 1/2: No violations [{_time.time()-_t0:.1f}s]")
    except Exception as e:
        sub.append(90)
        print(f"        [DQ] Step 1/2: Skipped — {str(e)[:60]} [{_time.time()-_t0:.1f}s]")

    # ── Step 2: Null rates — single batched DAX query ──
    cdf = meta["columns"]
    # Filter out system columns (RowNumber, internal) that can't be queried via DAX
    cdf = cdf[~cdf["Column Name"].str.contains("RowNumber|^__", regex=True, na=False)]
    _vlog("DQ", f"Eligible columns: {len(cdf)} (after filtering system columns)")
    n_sample = min(10, len(cdf))
    sample = cdf.sample(n_sample, random_state=42) if len(cdf) > n_sample else cdf
    print(f"        [DQ] Step 2/2: Checking null rates ({len(sample)} columns, batched)...")
    _t1 = _time.time()

    # Build one UNION DAX query instead of N separate calls
    union_parts = []
    col_list = []
    for _, c in sample.iterrows():
        t = safe_dax(c["Table Name"])
        cn = safe_dax(c["Column Name"])
        col_list.append((c["Table Name"], c["Column Name"]))
        _row_expr = 'ROW("T","' + t + '","C","' + cn + '","B",COUNTBLANK(' + "'" + t + "'" + '[' + cn + ']),"R",COUNTROWS(' + "'" + t + "'" + '))'
        union_parts.append(_row_expr)

    nok, ntot = 0, 0
    if union_parts:
        batch_ok = False
        try:
            if len(union_parts) == 1:
                batch_dax = "EVALUATE " + union_parts[0]
            else:
                batch_dax = "EVALUATE\nUNION(\n" + ",\n".join(union_parts) + "\n)"
            _vlog("DQ", f"Executing batched DAX ({len(union_parts)} columns)...")
            df_batch = eval_dax(ds, batch_dax, ws)
            if len(df_batch) > 0 and len(df_batch.columns) >= 4:
                batch_ok = True
                for _, row in df_batch.iterrows():
                    tname = str(row.iloc[0])
                    cname = str(row.iloc[1])
                    b = safe_int(row.iloc[2])
                    tot = safe_int(row.iloc[3], 1)
                    pct = b / max(tot, 1) * 100
                    ntot += 1
                    if pct < 50: nok += 1
                    if pct > 20:
                        details.append({"category": "Null Rate", "item": f"{tname}.{cname}",
                                       "status": "WARN" if pct > 50 else "INFO", "info": f"{pct:.1f}%"})
                    _vlog("DQ", f"  {tname}.{cname}: {pct:.1f}% null")
        except Exception as be:
            _vlog("DQ", f"  Batch failed: {str(be)[:80]}, falling back...")

        # Fallback: individual queries if batch failed
        if not batch_ok:
            _vlog("DQ", "Falling back to individual queries...")
            for idx, (tname, cname) in enumerate(col_list):
                t, cn = safe_dax(tname), safe_dax(cname)
                _vlog("DQ", f"  [{idx+1}/{len(col_list)}] {tname}.{cname}...")
                dax_q = 'EVALUATE ROW("B",COUNTBLANK(' + "'" + t + "'" + "[" + cn + ']),"T",COUNTROWS(' + "'" + t + "'" + "))"
                df = eval_dax(ds, dax_q, ws)
                if len(df) > 0 and len(df.columns) >= 2 and df.iloc[0, 1] is not None:
                    b, tot = safe_int(df.iloc[0, 0]), safe_int(df.iloc[0, 1], 1)
                    pct = b / max(tot, 1) * 100
                    ntot += 1
                    if pct < 50: nok += 1
                    if pct > 20:
                        details.append({"category": "Null Rate", "item": f"{tname}.{cname}",
                                       "status": "WARN" if pct > 50 else "INFO", "info": f"{pct:.1f}%"})

    sub.append((nok / max(ntot, 1)) * 100)
    sub.append(90)
    score = np.mean(sub) if sub else 0
    if (nok / max(ntot, 1)) * 100 < 80: recs.append("Review high null rate columns")
    print(f"        [DQ] Step 2/2: Done [{_time.time()-_t1:.1f}s]")
    return make_result("Data Quality", score, details, recs)

# ── Tool 8: Direct Lake ──
def tool_direct_lake(ds, meta, ws=None):
    _vlog("DLake", "Checking Direct Lake compatibility...")
    details, recs = [], []
    tdf,cdf = meta["tables"],meta["columns"]
    passed,total = 0, 0
    # Calc tables
    total+=1
    ct = tdf[tdf.get("Type",pd.Series(dtype=str)).str.contains("Calculated",case=False,na=False)]["Table Name"].tolist() if "Type" in tdf.columns else []
    if not ct: details.append({"category":"Calc Tables","item":"None","status":"PASS","info":"OK"}); passed+=1
    else:
        for t in ct: details.append({"category":"Calc Tables","item":t,"status":"FAIL","info":"Replace"})
        recs.append(f"Replace {len(ct)} calculated table(s)")
    # Calc columns
    total+=1
    cc = cdf[cdf.get("Type",pd.Series(dtype=str)).str.contains("Calculated",case=False,na=False)] if "Type" in cdf.columns else pd.DataFrame()
    if len(cc)==0: details.append({"category":"Calc Columns","item":"None","status":"PASS","info":"OK"}); passed+=1
    else:
        for _,c in cc.head(5).iterrows(): details.append({"category":"Calc Columns","item":f"{c['Table Name']}.{c['Column Name']}","status":"FAIL","info":"Convert"})
        recs.append(f"Convert {len(cc)} calculated column(s)")
    # Data types
    total+=1
    bad=[f"{c['Table Name']}.{c['Column Name']}" for _,c in cdf.iterrows() if str(c.get("Data Type","")).lower() in ["binary","variant"]] if "Data Type" in cdf.columns else []
    if not bad: details.append({"category":"Types","item":"All","status":"PASS","info":"OK"}); passed+=1
    else:
        for b in bad: details.append({"category":"Types","item":b,"status":"FAIL","info":"Unsupported"})
    # DL fallback
    total+=1
    try:
        fb = directlake.check_fallback_reason(dataset=ds,workspace=ws)
        if fb is not None and len(fb)>0:
            details.append({"category":"DL Fallback","item":"Active","status":"WARN","info":"Falling back"})
            recs.append("DL falling back to import")
        else:
            details.append({"category":"DL Fallback","item":"None","status":"PASS","info":"OK"}); passed+=1
    except:
        details.append({"category":"DL Fallback","item":"N/A","status":"INFO","info":"Not DL"}); passed+=1
    return make_result("Direct Lake",round(passed/max(total,1)*100,1),details,recs)

# ── Tool 9: Capacity ──
def tool_capacity(ds, ws=None):
    _vlog("Capacity", "Listing workspace model inventory...")
    details, recs = [], []
    try:
        datasets = fabric.list_datasets(workspace=ws)
        dc = "Dataset Name" if "Dataset Name" in datasets.columns else "Name"
        for _,d in datasets.iterrows():
            dn=d[dc]; is_t=dn==ds
            try:
                tbls=norm_cols(fabric.list_tables(dn,workspace=ws),TABLE_RENAMES)
                info=f"{len(tbls)} tables"
                st="PASS" if len(tbls)>0 else "WARN"
                if len(tbls)==0 and not is_t: recs.append(f"'{dn}' is empty")
            except: info="Cannot inspect"; st="INFO"
            details.append({"category":"Model","item":dn+(" (TARGET)" if is_t else ""),"status":st,"info":info})
        empty=len([d for d in details if d["status"]=="WARN"])
        return make_result("Capacity",max(0,100-empty*15),details,recs)
    except Exception as ex:
        return make_result("Capacity",0,[],[f"Failed: {str(ex)[:150]}"])

# ── Tool 10: Lineage ──
def tool_lineage(ds, meta, ws=None):
    """Trace data flow: tables -> columns -> measures.

    Four sections:
    1. Table Source — partition mode (Import/DirectLake/DirectQuery/M)
    2. Relationship Path — fact->dimension joins
    3. Measure Dependencies — parses DAX to build full dependency chain
    4. Unused Column + Orphan Measure + Circular dependency detection

    Returns: make_result with complexity-based score. Blast radius (fan-out)
    is deliberately NOT emitted here to avoid redundancy with DAX Dependencies.
    """
    _vlog("Lineage", "Building dependency graph...")
    details, recs = [], []
    mdf = meta["measures"]
    cdf = meta["columns"]
    tdf = meta["tables"]
    rdf = meta["relationships"]

    # ── 1. Table lineage: where does each table get its data? ──
    try:
        partitions = fabric.list_partitions(ds, workspace=ws)
        pc_name = next((c for c in ["Table Name","table_name"] if c in partitions.columns), None)
        pc_mode = next((c for c in ["Mode","mode","Source Type","source_type"] if c in partitions.columns), None)
        pc_expr = next((c for c in ["Query","Expression","Source Expression","query"] if c in partitions.columns), None)
        if pc_name:
            for _, p in partitions.iterrows():
                tn = str(p[pc_name])
                mode = str(p.get(pc_mode, "")) if pc_mode else ""
                expr = str(p.get(pc_expr, ""))[:100] if pc_expr else ""
                source = "Direct Lake" if "directlake" in mode.lower() or "entity" in mode.lower() else mode
                if not source and expr:
                    source = "M Expression"
                info = f"Source: {source}"
                if expr and len(expr) > 5:
                    info += f" | {expr[:60]}..."
                details.append({"category":"Table Source","item":tn,"status":"PASS","info":info})
    except Exception as e:
        details.append({"category":"Table Source","item":"(partitions)","status":"INFO","info":f"Could not read: {str(e)[:60]}"})

    # ── 2. Relationship lineage: fact → dimension paths ──
    if len(rdf) > 0:
        fc = next((c for c in ["From Table","fromTable"] if c in rdf.columns), None)
        tc = next((c for c in ["To Table","toTable"] if c in rdf.columns), None)
        fcc = next((c for c in ["From Column","fromColumn"] if c in rdf.columns), None)
        tcc = next((c for c in ["To Column","toColumn"] if c in rdf.columns), None)
        ac = next((c for c in ["Is Active","isActive","Active"] if c in rdf.columns), None)
        mc = next((c for c in ["Multiplicity","multiplicity","Cross Filtering Behavior"] if c in rdf.columns), None)
        if fc and tc:
            for _, rel in rdf.iterrows():
                ft, tt = str(rel[fc]), str(rel[tc])
                fc_col = str(rel.get(fcc, "")) if fcc else ""
                tc_col = str(rel.get(tcc, "")) if tcc else ""
                active = rel.get(ac, True) if ac else True
                mult = str(rel.get(mc, "")) if mc else ""
                st = "PASS" if active else "WARN"
                info = f"{ft}[{fc_col}] -> {tt}[{tc_col}]"
                if mult: info += f" ({mult})"
                if not active: info += " [INACTIVE]"
                details.append({"category":"Relationship Path","item":f"{ft} -> {tt}","status":st,"info":info})

    # ── 3. Measure lineage: DAX parsing + dependency chain ──
    ec = next((c for c in ["Measure Expression","Expression"] if c in mdf.columns), None)
    dep_map = {}   # measure -> direct refs
    rdep_map = {}  # object -> measures that use it

    if ec:
        _vlog("Lineage", "Tracing measure dependencies...")
    for _, m in mdf.iterrows():
            mn = m["Measure Name"]
            expr = str(m.get(ec, ""))
            refs = set()
            # 'Table'[Column] references
            for match in re.finditer(r"'([^']+)'\[([^\]]+)\]", expr):
                refs.add(f"{match.group(1)}[{match.group(2)}]")
            # Table[Column] references (no quotes)
            for match in re.finditer(r"(?<!')(\w+)\[([^\]]+)\]", expr):
                tbl = match.group(1)
                if tbl.upper() not in ("VAR","RETURN","TRUE","FALSE","BLANK","IF","NOT","AND","OR","IN","CALCULATE","FILTER","ALL","VALUES","SELECTEDVALUE","RELATED","LOOKUPVALUE","EARLIER","EARLIEST"):
                    refs.add(f"{tbl}[{match.group(2)}]")
            # [Measure] references
            for match in re.finditer(r"(?<!')\[([^\]]+)\]", expr):
                rn = match.group(1)
                if rn in mdf["Measure Name"].values and rn != mn:
                    refs.add(f"[{rn}]")
            dep_map[mn] = refs
            for ref in refs:
                rdep_map.setdefault(ref, set()).add(mn)

    # Build full dependency chain (transitive closure)
    def _get_chain(mn, visited=None):
        if visited is None: visited = set()
        if mn in visited: return visited
        visited.add(mn)
        for ref in dep_map.get(mn, []):
            if ref.startswith("["):
                child = ref.strip("[]")
                if child in dep_map:
                    _get_chain(child, visited)
        return visited

    # Measure details with full chain
    for mn, refs in sorted(dep_map.items(), key=lambda x: len(x[1]), reverse=True):
        col_refs = sorted([r for r in refs if not r.startswith("[")])
        msr_refs = sorted([r for r in refs if r.startswith("[")])
        chain = _get_chain(mn) - {mn}
        chain_msrs = [c for c in chain if c in dep_map]

        info_parts = []
        if col_refs:
            info_parts.append("Sources: " + ", ".join(col_refs[:4]))
            if len(col_refs) > 4:
                info_parts[-1] += f" +{len(col_refs)-4}"
        if msr_refs:
            info_parts.append("Refs: " + ", ".join(msr_refs[:3]))
        if chain_msrs:
            info_parts.append(f"Chain depth: {len(chain_msrs)}")
        info = " | ".join(info_parts) if info_parts else "Standalone (no dependencies)"

        dc = len(refs)
        st = "WARN" if dc > 8 else ("INFO" if dc > 3 else "PASS")
        details.append({"category":"Measure Dependencies","item":mn,"status":st,"info":info})

    # ── 4. Blast radius removed — kept in DAX Dependencies tool (avoids redundancy) ──

    # ── 5. Column usage: which columns are used by measures ──
    all_col_refs = set()
    for refs in dep_map.values():
        for r in refs:
            if not r.startswith("["):
                all_col_refs.add(r)

    # Find unused columns (not referenced by any measure or relationship)
    rel_cols = set()
    if len(rdf) > 0 and fc and tc and fcc and tcc:
        for _, rel in rdf.iterrows():
            rel_cols.add(f"{rel[fc]}[{rel.get(fcc,'')}]")
            rel_cols.add(f"{rel[tc]}[{rel.get(tcc,'')}]")

    used_cols = all_col_refs | rel_cols
    if "Table Name" in cdf.columns and "Column Name" in cdf.columns:
        unused_cols = []
        for _, c in cdf.iterrows():
            cn = c["Column Name"]
            tn = c["Table Name"]
            if "RowNumber" in cn: continue
            col_ref = f"{tn}[{cn}]"
            if col_ref not in used_cols:
                unused_cols.append(col_ref)
        if unused_cols:
            for uc in unused_cols[:10]:
                details.append({"category":"Unused Column","item":uc,"status":"INFO","info":"Not referenced by any measure or relationship"})
            if len(unused_cols) > 10:
                details.append({"category":"Unused Column","item":f"... +{len(unused_cols)-10} more","status":"INFO","info":""})
            recs.append(f"{len(unused_cols)} column(s) not used by any measure or relationship")

    # ── 6. Orphan measures (no column refs = derived only) ──
    orphans = [mn for mn, refs in dep_map.items() if len(refs) == 0]
    if orphans:
        for o in orphans:
            details.append({"category":"Orphan Measure","item":o,"status":"INFO","info":"No table/column refs (may use only other measures)"})

    # ── 7. Circular dependencies ──
    for mn in dep_map:
        chain = _get_chain(mn)
        msr_deps = [r.strip("[]") for r in dep_map.get(mn,[]) if r.startswith("[")]
        for md in msr_deps:
            if mn in _get_chain(md, set()):
                details.append({"category":"Circular","item":f"{mn} <-> {md}","status":"FAIL","info":"Circular dependency detected"})

    # Score
    complex_m = len([mn for mn, r in dep_map.items() if len(r) > 8])
    high_blast = len([o for o, d in rdep_map.items() if len(d) > 5])
    circulars = len([d for d in details if d["category"] == "Circular"])
    score = max(0, 100 - complex_m * 3 - high_blast * 2 - circulars * 10)

    if complex_m > 0:
        recs.append(f"{complex_m} measure(s) with >8 dependencies")
    if high_blast > 0:
        recs.append(f"{high_blast} high blast radius object(s)")
    if orphans:
        recs.append(f"{len(orphans)} orphan measure(s) with no column refs")
    if circulars > 0:
        recs.append(f"{circulars} circular dependency(ies) detected")
    total_tables = len(tdf) if "Table Name" in tdf.columns else 0
    recs.append(f"{total_tables} tables, {len(dep_map)} measures, {len(rdep_map)} unique dependencies traced")
    return make_result("Lineage",score,details,recs)


# ── Tool 12: Security Audit ──
def tool_security_audit(ds, meta, ws=None):
    """Top-level compliance check for the model's security posture.

    Evaluates whether roles are defined, have descriptions, have assigned
    members, and whether RLS filters cover the tables. This is the COMPLIANCE
    view — contrast with Security X-Ray which shows EFFECTIVE access (who
    actually sees what, including admin/member/contributor bypass).

    Returns a roles_pivot in extra={"roles_pivot": [...]} for the renderer.
    """
    _vlog("SecAudit", "Running Security Audit...")
    details, recs = [], []
    try:
        with connect_semantic_model(dataset=ds, workspace=ws, readonly=True) as tom:
            roles = list(tom.model.Roles)
            if not roles:
                details.append({"category":"Roles","item":"No roles defined","status":"FAIL","info":"Model has no RLS/OLS security"})
                recs.append("Define at least one security role with RLS filters")
                return make_result("Security Audit", 20, details, recs, extra={"roles_pivot": []})

            total_tables = len(list(tom.model.Tables))
            tables_with_rls = set()
            tables_with_ols = set()
            roles_no_desc = []
            roles_no_members = []
            total_members = 0
            roles_pivot = []  # role-centric pivot data for HTML render

            _vlog("SecAudit", f"Found {len(roles)} role(s), scanning...")
            for role in roles:
                rn = str(role.Name)
                _vlog("SecAudit", f"  Role: {rn}")
                rdesc = str(role.Description) if role.Description else ""
                rperm = str(role.ModelPermission) if hasattr(role, "ModelPermission") else "Read"

                # Role metadata
                st = "PASS"
                info_parts = [f"Permission: {rperm}"]
                if not rdesc:
                    roles_no_desc.append(rn)
                    st = "WARN"
                    info_parts.append("No description")

                _vlog("SecAudit", "Checking role members...")
    # Members
                members = []
                try:
                    for m in role.Members:
                        members.append(str(m.MemberName) if hasattr(m, "MemberName") else str(m.Name))
                except:
                    pass
                total_members += len(members)
                if not members:
                    roles_no_members.append(rn)
                    info_parts.append("No members assigned")
                    st = "WARN"
                else:
                    info_parts.append(f"{len(members)} member(s)")

                details.append({"category":"Role","item":rn,"status":st,"info":" | ".join(info_parts)})

                # Show members if extracted
                if members:
                    for mi in members[:5]:
                        details.append({"category":"Member","item":f"{rn}: {mi}","status":"PASS","info":"Assigned"})
                    if len(members) > 5:
                        details.append({"category":"Member","item":f"{rn}: +{len(members)-5} more","status":"PASS","info":""})

                # RLS filters
                try:
                    for tp in role.TablePermissions:
                        tn = str(tp.Table.Name)
                        filt = str(tp.FilterExpression).strip() if tp.FilterExpression else ""
                        if filt:
                            tables_with_rls.add(tn)
                            complexity = "Simple" if "USERNAME()" in filt.upper() or "USERPRINCIPALNAME()" in filt.upper() else "Complex"
                            details.append({"category":"RLS","item":f"{rn} -> {tn}","status":"PASS",
                                          "info":f"{complexity}: {filt[:80]}"})

                        _vlog("SecAudit", "Checking OLS configuration...")
    # OLS
                        meta_perm = str(tp.MetadataPermission) if hasattr(tp, "MetadataPermission") else "Default"
                        if meta_perm == "None":
                            tables_with_ols.add(tn)
                            details.append({"category":"OLS","item":f"{rn} -> {tn} (all columns)","status":"INFO",
                                          "info":"Table hidden from role"})

                        try:
                            for cp in tp.ColumnPermissions:
                                cn = str(cp.Column.Name)
                                cp_perm = str(cp.MetadataPermission)
                                if cp_perm == "None":
                                    tables_with_ols.add(tn)
                                    details.append({"category":"OLS","item":f"{rn} -> {tn}.{cn}","status":"INFO",
                                                  "info":"Column hidden from role"})
                        except:
                            pass
                except:
                    pass

            # Build role-centric pivot from collected details
            _rp_data = {}
            for _d in details:
                _cat = _d.get("category", "")
                _item = str(_d.get("item", ""))
                _info = str(_d.get("info", ""))
                if _cat == "Role":
                    _parts = _info.split(" | ")
                    _perm = "Read"
                    _desc = "(none)"
                    _mcount = ""
                    for _p in _parts:
                        _p = _p.strip()
                        if _p.startswith("Permission:"):
                            _perm = _p.replace("Permission:", "").strip()
                        elif _p == "No description":
                            _desc = "(none)"
                        elif "member(s)" in _p:
                            _mcount = _p
                    _rp_data.setdefault(_item, {"permission":_perm, "description":_desc, "member_count":_mcount, "members":[], "rls":[], "ols":[]})
                    _rp_data[_item]["permission"] = _perm
                    _rp_data[_item]["description"] = _desc
                    _rp_data[_item]["member_count"] = _mcount
                elif _cat == "Member":
                    if ":" in _item:
                        _rn, _mn = _item.split(":", 1)
                        _rn = _rn.strip()
                        _mn = _mn.strip()
                        _rp_data.setdefault(_rn, {"permission":"Read", "description":"(none)", "member_count":"", "members":[], "rls":[], "ols":[]})
                        _rp_data[_rn]["members"].append(_mn)
                elif _cat == "RLS":
                    if " -> " in _item:
                        _rn, _tbl = _item.split(" -> ", 1)
                        _rp_data.setdefault(_rn, {"permission":"Read", "description":"(none)", "member_count":"", "members":[], "rls":[], "ols":[]})
                        _rp_data[_rn]["rls"].append(f"{_tbl}: {_info}")
                elif _cat == "OLS":
                    if " -> " in _item:
                        _rn, _rest = _item.split(" -> ", 1)
                        _rp_data.setdefault(_rn, {"permission":"Read", "description":"(none)", "member_count":"", "members":[], "rls":[], "ols":[]})
                        _rp_data[_rn]["ols"].append(f"{_rest}: {_info}")
            for _rn, _rd in _rp_data.items():
                _members_str = ", ".join(_rd["members"][:5])
                if len(_rd["members"]) > 5:
                    _members_str += f" (+{len(_rd['members'])-5} more)"
                if not _members_str:
                    _members_str = "(none)"
                _rls_str = "; ".join(_rd["rls"]) if _rd["rls"] else "No filter condition applied"
                _ols_str = "; ".join(_rd["ols"]) if _rd["ols"] else "(none)"
                roles_pivot.append({
                    "Role": _rn,
                    "Permission": _rd["permission"],
                    "Description": _rd["description"],
                    "Members": _members_str,
                    "RLS": _rls_str,
                    "OLS": _ols_str,
                })

            # Scoring
            role_count = len(roles)
            rls_coverage = len(tables_with_rls) / max(total_tables, 1) * 100
            has_descriptions = (role_count - len(roles_no_desc)) / max(role_count, 1) * 100
            has_members = (role_count - len(roles_no_members)) / max(role_count, 1) * 100

            score_parts = []
            score_parts.append(min(100, rls_coverage * 2) * 0.30)  # RLS coverage (30%)
            score_parts.append((100 if tables_with_ols else 70) * 0.15)  # OLS (15%, base 70)
            score_parts.append(has_descriptions * 0.20)  # Role hygiene (20%)
            score_parts.append(80 * 0.15)  # Filter complexity baseline (15%)
            score_parts.append(has_members * 0.20)  # Members (20%)
            score = min(100, round(sum(score_parts), 1))

            # Summary
            details.insert(0, {"category":"Summary","item":f"{role_count} roles, {len(tables_with_rls)} tables with RLS, {len(tables_with_ols)} with OLS",
                              "status":"PASS" if role_count > 0 else "FAIL","info":f"Score: {score:.0f}%"})

            if roles_no_desc:
                recs.append(f"{len(roles_no_desc)} role(s) without description")
            if roles_no_members:
                recs.append(f"{len(roles_no_members)} role(s) with no members assigned")
            if not tables_with_rls:
                recs.append("No tables have RLS filters defined")
            recs.append(f"{role_count} roles, {total_members} total members, {len(tables_with_rls)}/{total_tables} tables secured")

            return make_result("Security Audit", score, details, recs, extra={"roles_pivot": roles_pivot})

    except Exception as ex:
        details.append({"category":"Error","item":"Security scan failed","status":"FAIL","info":str(ex)[:120]})
        return make_result("Security Audit", 0, details, [f"Failed: {str(ex)[:100]}"], extra={"roles_pivot": []})


# ── Tool 13: Security X-Ray ──
def _role_priority(r):
    return {"admin": 4, "member": 3, "contributor": 2, "viewer": 1}.get((r or "").lower(), 0)

def _xray_get_token(scope_host, timeout_sec=10):
    """Best-effort token acquisition. SPN first (instant), notebookutils fallback (with timeout)."""
    # ── Fast path: SPN via MSAL (no thread needed, completes in <1s) ──
    if USE_SPN and SPN_TENANT_ID and SPN_CLIENT_ID and SPN_CLIENT_SECRET:
        try:
            import msal
            _app = msal.ConfidentialClientApplication(
                SPN_CLIENT_ID,
                authority=f"https://login.microsoftonline.com/{SPN_TENANT_ID}",
                client_credential=SPN_CLIENT_SECRET,
            )
            _r = _app.acquire_token_for_client(scopes=[f"https://{scope_host}/.default"])
            if "access_token" in _r:
                print(f"        [X-Ray] Graph token acquired via SPN")
                return _r["access_token"]
            else:
                print(f"        [X-Ray] WARN: SPN token failed: {_r.get('error_description', _r.get('error', '?'))[:80]}")
        except ImportError:
            print(f"        [X-Ray] WARN: msal not installed — run %pip install msal")
        except Exception as e:
            print(f"        [X-Ray] WARN: SPN token error: {str(e)[:80]}")

    # ── Slow path: notebookutils (can hang — wrap in thread with timeout) ──
    import threading
    _result = [None]

    def _acquire_nu():
        try:
            import notebookutils
            _result[0] = notebookutils.credentials.getToken(f"https://{scope_host}")
        except Exception:
            pass

    t = threading.Thread(target=_acquire_nu, daemon=True)
    t.start()
    t.join(timeout=timeout_sec)
    if t.is_alive():
        print(f"        [X-Ray] WARN: notebookutils.getToken timed out after {timeout_sec}s — skipping")
        return None
    if _result[0]:
        print(f"        [X-Ray] Graph token acquired via notebookutils")
    return _result[0]

_group_name_cache = {}

def _xray_resolve_group_name(gid, hdr):
    """Fetch the display name for an AAD group. Returns name string or None."""
    if gid in _group_name_cache:
        return _group_name_cache[gid]
    import requests
    try:
        url = f"https://graph.microsoft.com/v1.0/groups/{gid}?$select=id,displayName,mail"
        resp = requests.get(url, headers=hdr, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            name = data.get("displayName") or data.get("mail") or None
            _group_name_cache[gid] = name
            return name
        # Might be a service principal — try that endpoint too
        if resp.status_code == 404:
            url = f"https://graph.microsoft.com/v1.0/servicePrincipals/{gid}?$select=id,displayName,appId"
            resp = requests.get(url, headers=hdr, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                name = data.get("displayName") or f"SPN:{data.get('appId', gid)[:8]}"
                _group_name_cache[gid] = name
                return name
    except Exception:
        pass
    _group_name_cache[gid] = None
    return None

def _xray_expand_group(gid, hdr):
    """Expand a single AAD group via Graph /transitiveMembers. Returns list[dict] or None on 403.
    Also resolves and caches the group's own display name for later lookup."""
    import requests
    users = []
    # Resolve group name proactively
    _xray_resolve_group_name(gid, hdr)
    url = f"https://graph.microsoft.com/v1.0/groups/{gid}/transitiveMembers?$select=id,displayName,userPrincipalName,mail"
    while url:
        try:
            resp = requests.get(url, headers=hdr, timeout=30)
        except Exception:
            return None
        if resp.status_code == 403:
            return None
        if resp.status_code >= 400:
            return None
        data = resp.json()
        for it in data.get("value", []):
            if it.get("@odata.type") == "#microsoft.graph.user":
                users.append({
                    "id": it.get("id", ""),
                    "displayName": it.get("displayName", ""),
                    "upn": it.get("userPrincipalName") or it.get("mail", ""),
                })
        url = data.get("@odata.nextLink")
    return users

def tool_security_xray(ds, meta, ws=None):
    """
    Security X-Ray — unified effective access map.

    Combines RLS filters + OLS restrictions + role members (DAX INFO) + AAD group
    expansion (Graph /transitiveMembers) + workspace role assignments (Fabric REST)
    into a single "who can actually see what" view. Flags admin/member/contributor
    bypass of RLS and dynamic RLS (USERPRINCIPALNAME/USERNAME).
    """
    import pandas as _pd
    import time as _time
    details, recs = [], []
    rls_rows, ols_rows, members_raw, ws_roles, effective = [], [], [], [], []
    expanded_groups = {}
    graph_available = False
    fabric_rest_available = False

    def _checkpoint(step, msg, sub=False):
        if sub and not VERBOSE_LOG:
            return
        prefix = "          " if sub else "        "
        print(f"{prefix}[X-Ray] Step {step}/6: {msg}")

    # ── Step 1: RLS + OLS via TOM ──
    _checkpoint(1, "Reading RLS + OLS via TOM...")
    _t1 = _time.time()
    try:
        with connect_semantic_model(dataset=ds, workspace=ws, readonly=True) as tom:
            roles = list(tom.model.Roles)
            if not roles:
                details.append({"category": "Roles", "item": "No roles defined", "status": "WARN",
                                "info": "Model has no roles. All users with workspace access see everything."})
                _checkpoint(1, f"Done — no roles found [{_time.time()-_t1:.1f}s]")
                return make_result("Security X-Ray", 50, details,
                                   ["Define at least one security role with RLS filters"],
                                   extra={"xray": {"rls": [], "ols": [], "members": [],
                                                   "workspace_access": [], "effective": [],
                                                   "graph_available": False, "fabric_rest_available": False,
                                                   "dynamic_roles": []}})
            _checkpoint(1, f"Found {len(roles)} role(s), scanning...", sub=True)
            for role in roles:
                rn = str(role.Name)
                _checkpoint(1, f"  Role: {rn}", sub=True)
                try:
                    role_ols_map = {}
                    for tp in role.TablePermissions:
                        tn = str(tp.Table.Name)
                        meta_perm = str(tp.MetadataPermission) if hasattr(tp, "MetadataPermission") else "Default"
                        col_perms = {}
                        try:
                            for cp in tp.ColumnPermissions:
                                col_perms[str(cp.Column.Name)] = str(cp.MetadataPermission)
                        except Exception:
                            pass
                        role_ols_map[tn] = {"table_perm": meta_perm, "cols": col_perms}
                        filt = None
                        try:
                            raw = tp.FilterExpression
                            filt = str(raw).strip() if raw is not None else None
                        except Exception:
                            pass
                        if filt:
                            upper = filt.upper()
                            is_dynamic = ("USERPRINCIPALNAME" in upper) or ("USERNAME()" in upper) or ("CUSTOMDATA" in upper)
                            rls_rows.append({
                                "Role": rn, "Table": tn,
                                "Filter": filt[:300],
                                "Dynamic": "Yes" if is_dynamic else "No",
                            })
                    for table in tom.model.Tables:
                        tn = str(table.Name)
                        sec = role_ols_map.get(tn, {"table_perm": "Default", "cols": {}})
                        tperm = sec["table_perm"]
                        if tperm == "None":
                            ols_rows.append({"Role": rn, "Table": tn, "Column": "*",
                                             "Level": "Table", "Permission": "None", "Visibility": "Hidden"})
                            continue
                        for col in table.Columns:
                            try:
                                if str(col.Type) == "RowNumber":
                                    continue
                            except Exception:
                                pass
                            cn = str(col.Name)
                            cperm = sec["cols"].get(cn)
                            if cperm == "None":
                                ols_rows.append({"Role": rn, "Table": tn, "Column": cn,
                                                 "Level": "Column", "Permission": "None", "Visibility": "Hidden"})
                except Exception as e:
                    details.append({"category": "RLS/OLS", "item": rn, "status": "FAIL", "info": str(e)[:120]})
        _checkpoint(1, f"Done — {len(rls_rows)} RLS, {len(ols_rows)} OLS [{_time.time()-_t1:.1f}s]")
    except Exception as e:
        _checkpoint(1, f"FAILED — {str(e)[:80]} [{_time.time()-_t1:.1f}s]")
        details.append({"category": "TOM", "item": "Could not read model", "status": "FAIL", "info": str(e)[:120]})
        return make_result("Security X-Ray", 0, details, [f"TOM read failed: {str(e)[:100]}"],
                           extra={"xray": {"rls": [], "ols": [], "members": [],
                                           "workspace_access": [], "effective": [],
                                           "graph_available": False, "fabric_rest_available": False,
                                           "dynamic_roles": []}})

    # ── Step 2: Role members via DAX INFO functions (scale-out safe) ──
    _checkpoint(2, "Querying role members via DAX INFO...")
    _t2 = _time.time()
    _dax = (
        "EVALUATE\n"
        "VAR _R = SELECTCOLUMNS(INFO.ROLES(), \"RoleID\",[ID], \"Role_Name\",[Name])\n"
        "VAR _M = SELECTCOLUMNS(INFO.ROLEMEMBERSHIPS(), \"RoleID\",[RoleID], "
        "\"Member_Name\",[MemberName], \"Member_Type\",[MemberType], \"Member_ID\",[MemberID])\n"
        "RETURN NATURALLEFTOUTERJOIN(_R, _M)"
    )
    try:
        df_m = fabric.evaluate_dax(ds, _dax, workspace=ws)
        def _col(df, needle):
            for c in df.columns:
                if needle.lower() in str(c).lower():
                    return c
            return None
        c_role = _col(df_m, "Role_Name")
        c_mname = _col(df_m, "Member_Name")
        c_mid = _col(df_m, "Member_ID")
        c_mtype = _col(df_m, "Member_Type")
        for _, r in df_m.iterrows():
            rn_v = r[c_role] if c_role else None
            if rn_v is None or (hasattr(_pd, "isna") and _pd.isna(rn_v)):
                continue
            mn_v = r[c_mname] if c_mname else None
            if mn_v is None or (hasattr(_pd, "isna") and _pd.isna(mn_v)):
                continue
            mid_v = r[c_mid] if c_mid else ""
            mtype_v = r[c_mtype] if c_mtype else ""
            members_raw.append({
                "Role": str(rn_v),
                "Member_Name": str(mn_v),
                "Member_ID": "" if mid_v is None or (hasattr(_pd, "isna") and _pd.isna(mid_v)) else str(mid_v),
                "Member_Type": "Unknown" if mtype_v is None or (hasattr(_pd, "isna") and _pd.isna(mtype_v)) else str(mtype_v),
            })
        for mr in members_raw:
            _checkpoint(2, f"  {mr['Role']}: {mr['Member_Name']} ({mr['Member_Type']})", sub=True)
        _checkpoint(2, f"Done — {len(members_raw)} member(s) [{_time.time()-_t2:.1f}s]")
    except Exception as e:
        _checkpoint(2, f"WARN — {str(e)[:80]} [{_time.time()-_t2:.1f}s]")
        details.append({"category": "Members", "item": "DAX INFO query failed", "status": "WARN", "info": str(e)[:120]})

    # ── Step 3: Graph API — expand AAD groups to users ──
    _checkpoint(3, "Acquiring Graph token + expanding AAD groups...")
    _t3 = _time.time()
    graph_token = _xray_get_token("graph.microsoft.com")
    def _is_expandable(m):
        """Check if a role member should be expanded via Graph.
        INFO.ROLEMEMBERSHIPS() MemberType: 1=User, 2=WindowsGroup, 3=External(SPN/Group).
        obj:GUID@tenant pattern = AAD object added by ID (could be group or SPN)."""
        mt = str(m.get("Member_Type") or "").strip().lower()
        mid = str(m.get("Member_ID") or "").strip()
        mn = str(m.get("Member_Name") or "").strip()
        if not mid:
            return False
        # Explicit group type
        if mt in ("2", "group", "windowsgroup"):
            return True
        # obj:GUID@tenant pattern — could be group or SPN, try expansion
        if mn.startswith("obj:") or mt in ("3", "external"):
            return True
        if "group" in mt:
            return True
        return False

    if graph_token:
        hdr = {"Authorization": f"Bearer {graph_token}"}
        model_group_ids = set()
        for m in members_raw:
            mid = str(m.get("Member_ID") or "").strip()
            mn = str(m.get("Member_Name") or "").strip()
            if _is_expandable(m):
                # Use Member_ID if it looks like a GUID, else try to extract from obj: pattern
                gid = mid
                if mn.startswith("obj:"):
                    # Extract GUID from obj:GUID@tenant
                    gid_part = mn.split("obj:")[-1].split("@")[0]
                    if len(gid_part) > 8:
                        gid = gid_part
                if gid:
                    model_group_ids.add(gid)
        _checkpoint(3, f"Expanding {len(model_group_ids)} group(s)...")
        for gid in model_group_ids:
            _checkpoint(3, f"  Expanding group {gid[:8]}...", sub=True)
            users = _xray_expand_group(gid, hdr)
            expanded_groups[gid] = users
            _checkpoint(3, f"  → {len(users) if users else 0} user(s)", sub=True)
            if users is None:
                details.append({"category": "Graph API", "item": f"Group {gid[:8]}... could not be expanded",
                                "status": "WARN", "info": "Graph call failed (403 Forbidden). Grant Directory.Read.All in Azure AD \u2192 App Registrations \u2192 API permissions to the SPN. See Troubleshooting section in Cell 0."})
        graph_available = True
        # Prettify obj:GUID names in members_raw using resolved group names
        for _mr in members_raw:
            _mn = str(_mr.get('Member_Name', ''))
            if _mn.startswith('obj:'):
                _guid = _mn.split('obj:')[-1].split('@')[0]
                _name = _group_name_cache.get(_guid)
                if _name:
                    _mr['Member_Name'] = _name
                    _mr['Member_ID'] = _guid
        _checkpoint(3, f"Done — {len(expanded_groups)} group(s) expanded [{_time.time()-_t3:.1f}s]")
    else:
        _checkpoint(3, f"Skipped — no Graph token [{_time.time()-_t3:.1f}s]")
        details.append({"category": "Graph API", "item": "No token", "status": "WARN",
                        "info": "Runtime identity needs Directory.Read.All (or GroupMember.Read.All) to expand AAD groups"})

    # ── Step 4: Workspace role assignments via Fabric REST ──
    _checkpoint(4, "Fetching workspace role assignments...")
    _t4 = _time.time()
    try:
        from sempy.fabric import FabricRestClient
        _client = FabricRestClient()
        ws_id = None
        try:
            if ws is None:
                ws_id = fabric.get_workspace_id()
            else:
                try:
                    ws_id = fabric.resolve_workspace_id(ws)
                except Exception:
                    _checkpoint(4, "resolve_workspace_id failed, trying get_workspace_id fallback...")
                    # Avoid list_workspaces() — it scans ALL workspaces and can hang in large tenants.
                    # Instead, try fabric.get_workspace_id() if we're already in the right workspace,
                    # or use a targeted REST call.
                    try:
                        # Try REST API to find workspace by name (much faster than list_workspaces)
                        _ws_resp = _client.get(f"v1/workspaces?name={ws}&type=Workspace")
                        _ws_data = _ws_resp.json() if hasattr(_ws_resp, "json") else {}
                        _ws_list = _ws_data.get("value", [])
                        if _ws_list:
                            ws_id = _ws_list[0].get("id")
                            _checkpoint(4, f"Resolved via REST filter: {ws_id}")
                    except Exception:
                        pass
                    # Last resort: try current workspace
                    if not ws_id:
                        try:
                            ws_id = fabric.get_workspace_id()
                            _checkpoint(4, f"Using current workspace ID: {ws_id}")
                        except Exception:
                            pass
        except Exception as we:
            details.append({"category": "Fabric REST", "item": "Workspace ID lookup",
                            "status": "WARN", "info": str(we)[:120]})
        if ws_id:
            _checkpoint(4, f"Workspace ID: {ws_id} — fetching role assignments...")
            try:
                _resp = _client.get(f"v1/workspaces/{ws_id}/roleAssignments")
                _data = _resp.json() if hasattr(_resp, "json") else {}
                for ra in _data.get("value", []):
                    p = ra.get("principal", {}) or {}
                    ptype = p.get("type", "")
                    pname = p.get("displayName", "")
                    pid = p.get("id", "")
                    email = ""
                    if ptype == "User":
                        email = (p.get("userDetails") or {}).get("userPrincipalName", "")
                    elif ptype == "Group":
                        email = (p.get("groupDetails") or {}).get("email", "")
                    elif ptype == "ServicePrincipal":
                        email = (p.get("servicePrincipalDetails") or {}).get("appId", "")
                    ws_roles.append({
                        "Principal_Name": pname, "Principal_ID": pid,
                        "Principal_Type": ptype, "Email": email,
                        "Workspace_Role": ra.get("role", ""),
                    })
                fabric_rest_available = True
                for wr in ws_roles:
                    _checkpoint(4, f"  {wr['Principal_Name']} ({wr['Principal_Type']}) → {wr['Workspace_Role']}", sub=True)
                # Also expand any workspace-level groups not yet seen
                if graph_available:
                    hdr2 = {"Authorization": f"Bearer {graph_token}"}
                    ws_groups = [wr for wr in ws_roles
                                 if "group" in (wr.get("Principal_Type") or "").lower()
                                 and wr.get("Principal_ID", "") not in expanded_groups]
                    if ws_groups:
                        _checkpoint(4, f"Expanding {len(ws_groups)} workspace group(s) via Graph...")
                    for wr in ws_groups:
                        gid = wr.get("Principal_ID", "")
                        if gid:
                            expanded_groups[gid] = _xray_expand_group(gid, hdr2)
                _checkpoint(4, f"Done — {len(ws_roles)} assignment(s) [{_time.time()-_t4:.1f}s]")
            except Exception as re:
                _checkpoint(4, f"WARN — roleAssignments call failed [{_time.time()-_t4:.1f}s]")
                details.append({"category": "Fabric REST", "item": "roleAssignments call",
                                "status": "WARN", "info": str(re)[:120]})
        else:
            _checkpoint(4, f"Skipped — workspace ID not resolved [{_time.time()-_t4:.1f}s]")
            details.append({"category": "Fabric REST", "item": "Workspace not resolved",
                            "status": "WARN", "info": "Workspace ID could not be determined"})
    except Exception as e:
        _checkpoint(4, f"WARN — FabricRestClient unavailable [{_time.time()-_t4:.1f}s]")
        details.append({"category": "Fabric REST", "item": "FabricRestClient unavailable",
                        "status": "WARN", "info": str(e)[:120]})

    # ── Step 5: Build effective access map ──
    _checkpoint(5, "Building effective access map...")
    _t5 = _time.time()
    role_filters = {}
    role_ols_map2 = {}
    for r in rls_rows:
        label = f'{r["Table"]}: {r["Filter"][:120]}'
        if r["Dynamic"] == "Yes":
            label += " [DYN]"
        role_filters.setdefault(r["Role"], []).append(label)
    for o in ols_rows:
        lbl = f'{o["Table"]} (all)' if o["Column"] == "*" else f'{o["Table"]}.{o["Column"]}'
        role_ols_map2.setdefault(o["Role"], []).append(lbl)

    user_model_roles = {}
    for m in members_raw:
        rn = m["Role"]
        mn = m["Member_Name"]
        mid = m["Member_ID"]
        mtype = str(m["Member_Type"] or "").strip().lower()
        mn_str = str(mn or "")
        _is_grp = mtype in ("2", "group", "windowsgroup") or "group" in mtype
        _is_ext = mtype in ("3", "external") or mn_str.startswith("obj:")
        _mid_key = mid
        if mn_str.startswith("obj:"):
            _mid_key = mn_str.split("obj:")[-1].split("@")[0] if len(mn_str.split("obj:")[-1].split("@")[0]) > 8 else mid
        if (_is_grp or _is_ext) and _mid_key in expanded_groups and expanded_groups[_mid_key] is not None:
            for u in expanded_groups[_mid_key]:
                upn = u.get("upn") or u.get("displayName") or ""
                if upn:
                    user_model_roles.setdefault(upn, set()).add((rn, f"AD Group: {mn}"))
        else:
            key = mn or mid
            if not key:
                continue
            if mtype in ("1", "user") or "user" in mtype:
                src = "Direct"
            elif mtype in ("2", "group", "windowsgroup") or "group" in mtype:
                # Try to resolve name from cache
                _grp_name = _group_name_cache.get(_mid_key) if _mid_key in _group_name_cache else None
                if _grp_name:
                    src = f"AD Group: {_grp_name}"
                else:
                    src = f"AD Group (unresolved): {mn}" if mn else "AD Group (unresolved)"
            elif mtype in ("3", "external") or str(mn or "").startswith("obj:"):
                _grp_name = _group_name_cache.get(_mid_key) if _mid_key in _group_name_cache else None
                if _grp_name:
                    src = f"AD Group: {_grp_name}"
                else:
                    src = f"SPN/External: {mn}" if mn else "External"
            else:
                src = mtype.title() or "Unknown"
            user_model_roles.setdefault(key, set()).add((rn, src))

    user_ws_roles = {}
    for wr in ws_roles:
        ptype = (wr.get("Principal_Type") or "").lower()
        pid = wr.get("Principal_ID", "")
        pname = wr.get("Principal_Name", "")
        email = wr.get("Email", "")
        role = wr.get("Workspace_Role", "")
        if "group" in ptype and pid in expanded_groups and expanded_groups[pid] is not None:
            for u in expanded_groups[pid]:
                upn = u.get("upn") or u.get("displayName") or ""
                if upn:
                    ex = user_ws_roles.get(upn)
                    if ex is None or _role_priority(role) > _role_priority(ex[0]):
                        user_ws_roles[upn] = (role, f"AD Group: {pname}")
        else:
            key = email or pname or pid
            if key:
                ex = user_ws_roles.get(key)
                if ex is None or _role_priority(role) > _role_priority(ex[0]):
                    user_ws_roles[key] = (role, ptype.title() or "Direct")

    all_users = set(user_model_roles.keys()) | set(user_ws_roles.keys())
    for u in sorted(all_users):
        ws_role, ws_src = user_ws_roles.get(u, (None, None))
        assignments = user_model_roles.get(u, set())
        if not assignments:
            assignments = {(None, None)}
        for mrn, msrc in assignments:
            filters = "; ".join(role_filters.get(mrn, [])) if mrn else ""
            ols_list = "; ".join(role_ols_map2.get(mrn, [])) if mrn else ""
            if ws_role and ws_role.lower() in ("admin", "member", "contributor"):
                eff = f"FULL ({ws_role} bypass)"
            elif mrn and any("[DYN]" in f for f in role_filters.get(mrn, [])):
                eff = "Dynamic (evaluated at query time)"
            elif mrn and filters:
                eff = "Filtered"
            elif mrn and ols_list:
                eff = "Full rows, columns hidden"
            elif mrn:
                eff = "Role has no restrictions"
            elif ws_role:
                eff = f"{ws_role} (no model role)"
            else:
                eff = "Unknown"
            effective.append({
                "User": u,
                "Source": msrc or ws_src or "Unknown",
                "Workspace_Role": ws_role or "—",
                "Model_Role": mrn or "—",
                "RLS_Filter": filters[:200] if filters else "—",
                "OLS_Hidden": ols_list[:200] if ols_list else "—",
                "Effective_View": eff,
            })
    for ea in effective:
        _checkpoint(5, f"  {ea['User']}: {ea['Effective_View']}", sub=True)
    _checkpoint(5, f"Done — {len(effective)} effective access row(s) [{_time.time()-_t5:.1f}s]")

    # ── Step 6: Summary details + score ──
    _checkpoint(6, "Computing score + summary...")
    _t6 = _time.time()
    n_roles = len({r["Role"] for r in rls_rows} | {o["Role"] for o in ols_rows} | {m["Role"] for m in members_raw})
    n_rls_tables = len({r["Table"] for r in rls_rows})
    n_ols_restrictions = len(ols_rows)
    n_members = len(members_raw)
    n_users = len(all_users)
    n_bypass = sum(1 for r in effective if "bypass" in r["Effective_View"].lower())
    dyn_roles = sorted({r["Role"] for r in rls_rows if r["Dynamic"] == "Yes"})
    n_groups_resolved = sum(1 for v in expanded_groups.values() if v is not None)

    details.insert(0, {
        "category": "Summary",
        "item": f"{n_roles} roles | {n_rls_tables} RLS tables | {n_ols_restrictions} OLS restrictions | {n_users} effective users",
        "status": "PASS",
        "info": f"{n_bypass} user(s) bypass RLS via workspace privilege | Graph={'yes' if graph_available else 'no'} | REST={'yes' if fabric_rest_available else 'no'}",
    })
    details.append({"category": "Inventory", "item": f"{len(rls_rows)} RLS filter(s)", "status": "PASS", "info": ""})
    details.append({"category": "Inventory", "item": f"{len(ols_rows)} OLS restriction(s)", "status": "PASS", "info": ""})
    details.append({"category": "Inventory", "item": f"{n_members} raw role member(s)", "status": "PASS", "info": ""})
    details.append({"category": "Inventory", "item": f"{len(ws_roles)} workspace role assignment(s)",
                    "status": "PASS" if fabric_rest_available else "WARN", "info": ""})
    details.append({"category": "Inventory", "item": f"{len(expanded_groups)} AAD group(s) probed",
                    "status": "PASS" if graph_available else "WARN", "info": f"{n_groups_resolved} resolved"})
    if dyn_roles:
        details.append({"category": "Note", "item": f"Dynamic RLS in: {', '.join(dyn_roles)}", "status": "INFO",
                        "info": "Uses USERPRINCIPALNAME/USERNAME — effective rows vary per-user"})
    if n_bypass > 0:
        recs.append(f"{n_bypass} user(s) bypass RLS via workspace Admin/Member/Contributor role")
    if not graph_available:
        recs.append("Grant Directory.Read.All or GroupMember.Read.All to runtime identity for AAD group expansion")
    if not fabric_rest_available:
        recs.append("Fabric REST workspace role lookup failed — check identity permissions on workspace")
    if not rls_rows:
        recs.append("No RLS filters defined — consider adding row-level security")

    score_parts = [
        100 if rls_rows else 40,
        100 if members_raw else 50,
        50 if (n_bypass > 0 and n_users > 0 and n_bypass >= n_users / 2) else 90,
    ]
    score = round(sum(score_parts) / len(score_parts), 1)
    _checkpoint(6, f"Done — score {score} [{_time.time()-_t6:.1f}s]")

    return make_result("Security X-Ray", score, details, recs, extra={"xray": {
        "rls": rls_rows,
        "ols": ols_rows,
        "members": members_raw,
        "workspace_access": ws_roles,
        "effective": effective,
        "graph_available": graph_available,
        "fabric_rest_available": fabric_rest_available,
        "dynamic_roles": dyn_roles,
        "expanded_groups": expanded_groups,
    }})

# ── Tool 11: Report Visual Analysis ──
def tool_report_visuals(ds, meta, ws=None):
    """Scan reports connected to this semantic model.

    Uses sempy_labs.ReportWrapper to list visuals and their object
    references. Detects broken references (visual points at measure/column
    that no longer exists) and builds a Report -> Measures Used mapping.
    Also flags measures defined in the model but not used in any report.

    Returns: make_result with 100 - (broken/total * 100). Strictly filters
    to reports using the current model (by dataset ID).
    """
    _vlog("Visuals", "Scanning report visuals...")
    details, recs = [], []
    _vlog("Visuals", "Listing connected reports...")
    try:
        import sempy_labs.report as _rep
        # Find reports connected to this dataset
        reports = fabric.list_reports(workspace=ws)
        rc = next((c for c in ["Report Name","Name","name"] if c in reports.columns), reports.columns[0] if len(reports.columns)>0 else None)
        dc = next((c for c in ["Dataset Name","Dataset Id","datasetId","dataset_id"] if c in reports.columns), None)
        if rc is None:
            return make_result("Report Visuals",100,[],["No reports found"])

        # Resolve current dataset ID so we can filter reports properly
        _ds_id = None
        try:
            _ds_list = fabric.list_datasets(workspace=ws)
            _name_col = next((c for c in ["Dataset Name","Name","name"] if c in _ds_list.columns), None)
            _id_col = next((c for c in ["Dataset Id","Id","id","datasetId"] if c in _ds_list.columns), None)
            if _name_col and _id_col:
                _match = _ds_list[_ds_list[_name_col].astype(str).str.lower() == str(ds).lower()]
                if len(_match) > 0:
                    _ds_id = str(_match.iloc[0][_id_col])
        except Exception:
            pass

        # Build dataset ID/name column candidates on the reports DataFrame
        _rep_ds_id_col = next((c for c in ["Dataset Id","DatasetId","datasetId","dataset_id"] if c in reports.columns), None)
        _rep_ds_name_col = next((c for c in ["Dataset Name","Dataset","dataset_name"] if c in reports.columns), None)

        connected = []
        for _, r in reports.iterrows():
            rn = r[rc]
            # Primary match: by dataset ID (most reliable)
            if _ds_id and _rep_ds_id_col:
                if str(r.get(_rep_ds_id_col,"")).lower() == _ds_id.lower():
                    connected.append(rn)
                    continue
            # Secondary match: by dataset name
            if _rep_ds_name_col:
                if str(r.get(_rep_ds_name_col,"")).lower() == str(ds).lower():
                    connected.append(rn)
                    continue
        # NO fallback — only reports actually connected to this model

        if not connected:
            return make_result("Report Visuals",100,[],["No reports connected to this model"],extra={"skipped":True})

        all_objects = []
        for rn in connected[:3]:  # max 3 reports
            try:
                rep = _rep.ReportWrapper(report=rn, workspace=ws)

                # Pages
                try:
                    pages = rep.list_pages()
                    details.append({"category":"Report","item":rn,"status":"PASS",
                                  "info":f"{len(pages)} pages"})
                except:
                    details.append({"category":"Report","item":rn,"status":"PASS","info":"Connected"})

                # Semantic model objects used in report
                try:
                    objs = rep.list_semantic_model_objects(extended=True)
                    oc = next((c for c in ["Object Name","object_name","Object"] if c in objs.columns), None)
                    tc = next((c for c in ["Object Type","object_type","Type"] if c in objs.columns), None)
                    vc = next((c for c in ["Visual Id","visual_id","Visual"] if c in objs.columns), None)
                    pc = next((c for c in ["Page Name","page_name","Page"] if c in objs.columns), None)
                    val_c = next((c for c in ["Valid Semantic Model Object","valid"] if c in objs.columns), None)

                    if oc:
                        for _, o in objs.iterrows():
                            obj_name = str(o[oc])
                            obj_type = str(o.get(tc,"")) if tc else ""
                            page = str(o.get(pc,"")) if pc else ""
                            valid = o.get(val_c, True) if val_c else True
                            st = "PASS" if valid else "FAIL"
                            info = f"{obj_type} on {page}" if page else obj_type
                            if not valid:
                                info += " (BROKEN - not in model)"
                            all_objects.append({"obj":obj_name,"type":obj_type,"page":page,"valid":valid,"report":rn})
                            details.append({"category":"Visual Object","item":obj_name,"status":st,"info":info})
                except Exception as e:
                    details.append({"category":"Report","item":rn,"status":"INFO","info":f"Cannot read visuals: {str(e)[:60]}"})

                # Filters
                try:
                    filters = rep.list_report_filters()
                    if len(filters) > 0:
                        details.append({"category":"Filters","item":f"{rn} report filters","status":"PASS","info":f"{len(filters)} filters"})
                except:
                    pass

            except Exception as e:
                details.append({"category":"Report","item":rn,"status":"WARN","info":f"Error: {str(e)[:80]}"})

        # Score
        broken = len([o for o in all_objects if not o["valid"]])
        total_obj = len(all_objects)
        if total_obj > 0:
            score = max(0, round((1 - broken / total_obj) * 100, 1))
        else:
            score = 100
        if broken > 0:
            recs.append(f"{broken} broken reference(s) — measures/columns not found in model")
        recs.append(f"{len(connected)} report(s) analyzed, {total_obj} visual objects found")

        # Report → Measures Used mapping (built from all_objects)
        _report_measures = {}
        for _o in all_objects:
            _rn = _o.get("report","")
            _obj = _o.get("obj","")
            _typ = _o.get("type","")
            if _rn and _obj and ("measure" in _typ.lower() or _obj in set(m.strip() for m in meta.get("measures", pd.DataFrame()).get("Measure Name", []))):
                _report_measures.setdefault(_rn, set()).add(_obj)
        for _rn, _measures in _report_measures.items():
            _m_list = sorted(_measures)[:5]
            _info = f"{len(_measures)} measure(s): " + ", ".join(_m_list)
            if len(_measures) > 5:
                _info += f" +{len(_measures)-5} more"
            details.append({"category":"Report Usage","item":_rn,"status":"INFO","info":_info})
        if _report_measures:
            recs.append(f"{len(_report_measures)} report(s) mapped to measures used")

        # Unused measures (in model but not in any report)
        used_objs = set(o["obj"] for o in all_objects)
        try:
            mdf = meta.get("measures", pd.DataFrame())
            if "Measure Name" in mdf.columns:
                unused = [mn for mn in mdf["Measure Name"] if mn not in used_objs]
                if unused:
                    for u in unused[:10]:
                        details.append({"category":"Unused","item":u,"status":"INFO","info":"Not used in any report visual"})
                    if len(unused) > 10:
                        details.append({"category":"Unused","item":f"... +{len(unused)-10} more","status":"INFO","info":""})
                    recs.append(f"{len(unused)} measure(s) not used in any report")
        except:
            pass

        return make_result("Report Visuals",score,details,recs)
    except ImportError:
        return make_result("Report Visuals",0,[],["sempy_labs.report not available"])
    except Exception as ex:
        return make_result("Report Visuals",0,[],[f"Failed: {str(ex)[:150]}"])

# ── Fix Plan Generator ──
def generate_fix_plan(results_data, meta):
    """Build the auto-fix plan from scan results.

    Generates up to 9 fix types: descriptions (AI or template), synonyms
    (via Culture ObjectTranslation), hidden key columns, format strings,
    summarize-by, model description (CustomInstructions), AI instructions,
    schema reduction (isAvailableInMDX + LinguisticMetadata Entity Visibility),
    and role descriptions.

    AI fixes use Azure OpenAI (Fabric Copilot auth) with rich context
    (table columns, relationships, DAX expressions, user business context).
    Template fixes use rule-based smart defaults.

    Returns: list of dicts with {category, item, current, proposed, source,
    type, name, value, risk}. Cell 5 applies selected fixes via TOM writes.
    """
    fixes = []
    tdf,cdf,mdf = meta["tables"],meta["columns"],meta["measures"]
    def _cur(val):
        if val is None or (hasattr(pd,'isna') and pd.isna(val)): return "(empty)"
        s=str(val).strip(); return s[:60] if s else "(empty)"
    def _clean(name):
        import re
        n = re.sub(r'^(?:fact|dim|dimension)[_ ]', '', name, flags=re.IGNORECASE)
        return n.replace('_', ' ').strip().title()

    def _smart_table_desc(tn):
        # Generate meaningful table description from name.
        clean = _clean(tn).lower()
        # Detect table type from original name
        tl = tn.lower()
        if "date" in tl or "calendar" in tl:
            return f"Calendar dimension with date hierarchies and fiscal periods"
        elif tl.startswith("fact"):
            return f"Transactional data recording {clean} events"
        elif tl.startswith(("dim", "dimension")):
            return f"Descriptive attributes for {clean} entities"
        elif "bridge" in tl or "map" in tl:
            return f"Bridge table linking many-to-many {clean} relationships"
        else:
            return f"Contains {clean} data"

    def _smart_col_desc(cn, tn, dtype=None):
        # Generate meaningful column description from name, table, and data type.
        cl = cn.lower().replace("_", " ")
        tclean = _clean(tn).lower()
        # Key columns
        if cl.endswith("key") or cl.endswith("sk"):
            if tn.lower() in cn.lower().replace("key","").replace("_","").lower():
                return f"Primary surrogate key for the {tclean} table"
            return f"Foreign key linking to the {_clean(cl.replace(' key','').replace(' sk','').strip())} dimension"
        if cl.endswith("id") or cl.startswith("wwi"):
            ref = cl.replace("id","").replace("wwi","").strip()
            return f"Business identifier for {ref if ref else tclean} records"
        # Date columns
        if dtype and "date" in str(dtype).lower():
            return f"Date when the {cl.replace('date','').strip() or tclean} event occurred"
        if "date" in cl or "valid" in cl:
            return f"Date for {cl} tracking"
        # Boolean
        if cl.startswith("is ") or cl.startswith("has "):
            return f"Flag indicating whether the {tclean} {cl}"
        # Descriptive
        if cl in ("name", "description", "label"):
            return f"Display name of the {tclean}"
        if "email" in cl: return f"Email address for the {tclean}"
        if "phone" in cl: return f"Phone number for the {tclean}"
        if "address" in cl or "city" in cl or "state" in cl or "country" in cl or "postal" in cl:
            return f"Geographic {cl} of the {tclean}"
        if "latitude" in cl: return f"Geographic latitude coordinate"
        if "longitude" in cl: return f"Geographic longitude coordinate"
        # Numeric
        if any(k in cl for k in ["amount", "price", "cost", "revenue", "tax", "profit"]):
            return f"Monetary {cl} value for the {tclean} transaction"
        if any(k in cl for k in ["quantity", "count", "number", "qty"]):
            return f"Numeric {cl} for the {tclean}"
        if any(k in cl for k in ["rate", "percent", "ratio"]):
            return f"Calculated {cl} metric"
        # Category/type
        if any(k in cl for k in ["category", "type", "status", "segment", "tier", "group", "class"]):
            return f"Classification {cl} for the {tclean}"
        if any(k in cl for k in ["color", "size", "brand", "package"]):
            return f"Product attribute: {cl}"
        # Fallback
        return f"{cn.replace('_', ' ')} attribute of the {tclean}"

    def _smart_measure_desc(mn, expr=None):
        # Generate meaningful measure description from name and DAX expression.
        ml = mn.lower().replace("_", " ")
        el = (expr or "").upper()
        # Detect aggregation type from expression
        agg = ""
        if "SUM(" in el: agg = "sum of"
        elif "AVERAGE(" in el or "AVG" in ml: agg = "average"
        elif "COUNT(" in el or "COUNTROWS(" in el: agg = "count of"
        elif "DISTINCTCOUNT(" in el: agg = "distinct count of"
        elif "MIN(" in el: agg = "minimum"
        elif "MAX(" in el: agg = "maximum"
        elif "DIVIDE(" in el: agg = "ratio of"
        # Detect time intelligence
        time_fn = ""
        if "TOTALYTD(" in el: time_fn = "year-to-date "
        elif "TOTALMTD(" in el: time_fn = "month-to-date "
        elif "SAMEPERIODLASTYEAR(" in el: time_fn = "prior year "
        elif "DATEADD(" in el or "EDATE(" in el: time_fn = "rolling period "
        # Detect anti-patterns for description
        if "FILTER(ALL(" in el:
            return f"Calculates {ml} using filtered context override (FILTER/ALL pattern)"
        if "COUNTROWS(FILTER(" in el:
            return f"Counts rows matching a filter condition for {ml}"
        if "HASONEVALUE(" in el:
            return f"Returns selected {ml} value when single selection exists"
        if "SUMX(" in el:
            return f"Row-by-row calculation of {ml} across the table"
        if "USERELATIONSHIP(" in el:
            return f"Calculates {ml} using an alternate relationship path"
        # Build description
        if agg and time_fn:
            return f"The {time_fn}{agg} {ml}"
        elif agg:
            return f"The {agg} {ml} across all transactions"
        elif time_fn:
            return f"The {time_fn}{ml}"
        # Composite measures (referencing other measures)
        if "[" in (expr or "") and not any(f in el for f in ["SUM(","COUNT(","MIN(","MAX(","AVERAGE("]):
            return f"Derived metric calculating {ml} from other measures"
        return f"Calculates the {ml}"


    # ── AI-powered descriptions & synonyms ──
    # Passes rich metadata (columns, relationships, types, expressions) to the LLM
    # for context-aware generation. Falls back to templates if AI is unavailable.
    ai_descs = {"tables": None, "columns": None, "measures": None}
    _ai_used = False

    # Build metadata context for AI prompts
    _rels = meta.get("relationships", pd.DataFrame())
    def _table_context(tn):
        """Build rich context string for a table."""
        cols = cdf[cdf["Table Name"] == tn]["Column Name"].tolist() if "Table Name" in cdf.columns else []
        cols = [c for c in cols if "RowNumber" not in c]
        col_str = ", ".join(cols[:12])
        if len(cols) > 12:
            col_str += f", ... (+{len(cols)-12} more)"
        # Related tables
        related = set()
        if len(_rels) > 0:
            for _, r in _rels.iterrows():
                ft, tt = r.get("From Table", ""), r.get("To Table", "")
                if ft == tn: related.add(tt)
                if tt == tn: related.add(ft)
        rel_str = ", ".join(sorted(related)) if related else "none"
        # Table type
        tl = tn.lower()
        ttype = "fact" if tl.startswith("fact") else ("dimension" if tl.startswith(("dim","dimension")) else "other")
        return f"Table '{tn}' (type: {ttype}, {len(cols)} columns: {col_str}). Related to: {rel_str}."

    def _column_context(cn, tn, dtype=None):
        """Build rich context for a column."""
        ctx = f"Column '{cn}'"
        if dtype: ctx += f" (type: {dtype})"
        ctx += f" in table '{tn}'"
        # Check if it's a key column
        cl = cn.lower()
        if cl.endswith(("key", "sk", "fk", "id")):
            # Find what it relates to
            for _, r in _rels.iterrows() if len(_rels) > 0 else []:
                if r.get("From Column", "") == cn and r.get("From Table", "") == tn:
                    ctx += f" (FK to {r.get('To Table','')}.{r.get('To Column','')})"
                    break
                if r.get("To Column", "") == cn and r.get("To Table", "") == tn:
                    ctx += f" (PK, referenced by {r.get('From Table','')}.{r.get('From Column','')})"
                    break
        return ctx

    if USE_AI_DESCRIPTIONS:
        try:
            # ── Table descriptions ──
            items = []
            if "Description" in tdf.columns:
                no_desc = tdf[~(tdf["Description"].notna() & (tdf["Description"] != ""))]
                for _, t in no_desc.iterrows():
                    tn = t["Table Name"]
                    ctx = _table_context(tn)
                    items.append(f"{ctx} Generate a 2-4 word label for this table (noun phrase only, no verbs). Examples: 'Sales transactions', 'Customer master', 'Date dimension'.")
            if items:
                ai_descs["tables"] = _ai_generate(items, "table descriptions")
                _ai_used = True
                print(f"  AI: {len(ai_descs['tables'])} table descriptions (with schema context)")

            # ── Column descriptions ──
            col_items = []
            if "Description" in cdf.columns:
                no_desc_c = cdf[~(cdf["Description"].notna() & (cdf["Description"] != ""))]
                no_desc_c = no_desc_c[~no_desc_c["Column Name"].str.contains("RowNumber-|rowNumber-", na=False)]
                for _, c in no_desc_c.head(20).iterrows():
                    cn, tn = c["Column Name"], c["Table Name"]
                    dtype = c.get("Data Type", "") if hasattr(c, "get") else ""
                    ctx = _column_context(cn, tn, dtype)
                    col_items.append(f"{ctx}. Generate a 2-4 word label for this column (noun phrase only). Examples: 'Customer ID', 'Sale amount', 'Product category', 'Order date'.")
            if col_items:
                ai_descs["columns"] = _ai_generate(col_items, "column descriptions")
                _ai_used = True
                print(f"  AI: {len(ai_descs['columns'])} column descriptions (with type + relationship context)")

            # ── Measure descriptions ──
            msr_items = []
            if "Description" in mdf.columns:
                for _, m in mdf.iterrows():
                    if not (pd.notna(m["Description"]) and str(m["Description"]).strip()):
                        mn = m["Measure Name"]
                        expr = str(m.get("Measure Expression", ""))[:200]
                        folder = str(m.get("Display Folder", "")) if "Display Folder" in m.index else ""
                        ctx = f"DAX measure '{mn}'"
                        if folder: ctx += f" (folder: {folder})"
                        ctx += f" with expression: {expr}"
                        msr_items.append(f"{ctx}. Generate a 2-4 word label for this measure (noun phrase only). Examples: 'Total revenue', 'Average order size', 'YoY growth %'.")
            if msr_items:
                ai_descs["measures"] = _ai_generate(msr_items, "measure descriptions")
                _ai_used = True
                print(f"  AI: {len(ai_descs['measures'])} measure descriptions (with DAX expression context)")

            if not _ai_used:
                print("  AI: No items needed AI generation.")
        except Exception as _ai_err:
            print(f"  AI generation error: {type(_ai_err).__name__}: {str(_ai_err)[:150]}")
            print(f"     Falling back to template-based descriptions")

    # Table descriptions
    if "Description" in tdf.columns:
        ai_idx=0
        for _,t in tdf.iterrows():
            tn=t["Table Name"]; cur=_cur(t["Description"])
            if cur=="(empty)":
                if ai_descs["tables"] and ai_idx<len(ai_descs["tables"]):
                    proposed=str(ai_descs["tables"][ai_idx])[:60]; src="AI"; ai_idx+=1
                else:
                    proposed=_smart_table_desc(tn); src="Template"
                fixes.append({"category":"Description","item":f"Table: {tn}","current":cur,"proposed":proposed,
                             "source":src,"type":"description","obj_type":"table","name":tn,"risk":"Low"})
    # Column descriptions (skip system columns like RowNumber-...)
    if "Description" in cdf.columns:
        no_desc=cdf[~(cdf["Description"].notna()&(cdf["Description"]!=""))]
        no_desc=no_desc[~no_desc["Column Name"].str.contains("RowNumber-|rowNumber-",na=False)]
        ai_cidx=0
        for _,c in no_desc.head(20).iterrows():
            tn,cn=c["Table Name"],c["Column Name"]
            if ai_descs["columns"] and ai_cidx<len(ai_descs["columns"]):
                proposed=str(ai_descs["columns"][ai_cidx])[:60]; src="AI"; ai_cidx+=1
            else:
                dtype=c.get("Data Type","") if hasattr(c,'get') else ""; proposed=_smart_col_desc(cn,tn,dtype); src="Template"
            fixes.append({"category":"Description","item":f"Column: {tn}.{cn}","current":"(empty)",
                         "proposed":proposed,"source":src,
                         "type":"description","obj_type":"column","table":tn,"name":cn,"risk":"Low"})
    # Measure descriptions
    if "Description" in mdf.columns:
        ai_midx=0
        for _,m in mdf.iterrows():
            mn=m["Measure Name"]; cur=_cur(m["Description"])
            if cur=="(empty)":
                if ai_descs["measures"] and ai_midx<len(ai_descs["measures"]):
                    proposed=str(ai_descs["measures"][ai_midx])[:60]; src="AI"; ai_midx+=1
                else:
                    expr=m.get("Measure Expression","") if hasattr(m,'get') else ""; proposed=_smart_measure_desc(mn,expr); src="Template"
                fixes.append({"category":"Description","item":f"Measure: {mn}","current":cur,
                             "proposed":proposed,"source":src,
                             "type":"measure_description","name":mn,"risk":"Low"})
    # Hidden keys
    if "Is Hidden" in cdf.columns:
        keys=cdf[(cdf["Column Name"].str.contains(r'(?i)(key|sk|fk|id)$',regex=True)) & (~cdf["Column Name"].str.contains("RowNumber-",na=False))]
        for _,c in keys[~keys["Is Hidden"]].iterrows():
            tn,cn=c["Table Name"],c["Column Name"]
            fixes.append({"category":"Visibility","item":f"Column: {tn}.{cn}","current":"Visible","proposed":"Hidden",
                         "source":"Rule","type":"hide_column","table":tn,"name":cn,"risk":"Low"})
    # Format strings
    if "Format String" in mdf.columns:
        for _,m in mdf.iterrows():
            mn=m["Measure Name"]; cur=_cur(m.get("Format String"))
            if cur=="(empty)":
                fmt="#,##0"; ml=mn.lower()
                if any(k in ml for k in ["%","pct","percent","ratio","rate","margin"]): fmt="0.0%"
                elif any(k in ml for k in ["$","revenue","sales","cost","price","amount"]): fmt="$#,##0.00"
                elif any(k in ml for k in ["avg","average"]): fmt="#,##0.00"
                fixes.append({"category":"Format","item":f"Measure: {mn}","current":cur,"proposed":fmt,
                             "source":"Rule","type":"format_string","name":mn,"value":fmt,"risk":"Low"})
    # Summarize By
    if "Summarize By" in cdf.columns and "Data Type" in cdf.columns:
        bad=cdf[(cdf["Data Type"].isin(["String","DateTime"]))&(cdf["Summarize By"].isin(["Sum","Average","Min","Max"]))]
        for _,c in bad.iterrows():
            fixes.append({"category":"Summarize By","item":f"Column: {c['Table Name']}.{c['Column Name']}",
                         "current":str(c["Summarize By"]),"proposed":"None","source":"Rule",
                         "type":"summarize_by","table":c["Table Name"],"name":c["Column Name"],"risk":"Low"})
    # ── Synonyms (ObjectTranslation Captions for Copilot) ──
    # Generate synonym suggestions for tables, columns, and measures.
    # These become ObjectTranslation Caption entries in the en-US Culture,
    # giving Copilot alternative names to recognize in natural language queries.
    def _smart_synonym(name, obj_type="table"):
        """Generate a concise synonym/alias from a table/column/measure name."""
        import re
        n = name.strip()
        # Remove common prefixes (fact_, dim_, dimension_, dbo_, wwi, stg_, raw_)
        clean = re.sub(r'^(?:dimension|fact|dim|dbo|stg|raw|wwi)[_ .]?', '', n, flags=re.IGNORECASE)
        if not clean:
            clean = n  # Don't empty the name
        # Convert camelCase/PascalCase to spaces
        clean = re.sub(r'([a-z])([A-Z])', r'\1 \2', clean)
        # Convert underscores/dots to spaces
        clean = clean.replace('_', ' ').replace('.', ' ').strip()
        # Title case with special handling for ID, WWI, etc.
        words = clean.split()
        titled = []
        for w in words:
            wup = w.upper()
            if wup in ('ID', 'SK', 'FK', 'PK', 'WWI', 'KPI', 'YTD', 'MTD', 'PY', 'QTD'):
                titled.append(wup)
            else:
                titled.append(w.capitalize())
        synonym = ' '.join(titled)
        # Don't return if it's the same as the original (case-insensitive)
        if synonym.lower() == n.lower() or synonym.lower() == name.lower():
            return None  # No useful synonym — skip this item entirely
        # Don't return very short synonyms (< 2 chars)
        if len(synonym) < 2:
            return None
        return synonym

    try:
        # Check if en-US culture exists and what translations already exist
        _existing_syns = set()
        try:
            with connect_semantic_model(DATASET, readonly=True, workspace=WORKSPACE) as _tom_syn:
                for culture in _tom_syn.model.Cultures:
                    if culture.Name == "en-US":
                        try:
                            for ot in culture.ObjectTranslations:
                                _prop = str(ot.Property) if hasattr(ot, 'Property') else ""
                                if "Caption" in _prop:
                                    _obj = ot.Object
                                    _existing_syns.add(str(_obj.Name) if hasattr(_obj, 'Name') else "")
                        except Exception:
                            pass
                        break
        except Exception:
            pass

        # Generate AI synonyms if available
        syn_ai = {"tables": None, "columns": None, "measures": None}
        if USE_AI_DESCRIPTIONS:
            try:
                syn_items = []
                for _, t in tdf.iterrows():
                    tn = t["Table Name"]
                    if tn not in _existing_syns:
                        syn_items.append(f"Table '{tn}' ({_table_context(tn)[:80]}). Suggest a 1-3 word business-friendly synonym. Return ONLY the synonym.")
                if syn_items:
                    syn_ai["tables"] = _ai_generate(syn_items, "table synonyms")
                    print(f"  AI: {len(syn_ai['tables'])} table synonyms generated")

                col_syn_items = []
                _cdf_clean2 = cdf[~cdf["Column Name"].str.contains("RowNumber-|rowNumber-", na=False)] if "Column Name" in cdf.columns else cdf
                for _, c in _cdf_clean2.head(30).iterrows():
                    cn = c["Column Name"]
                    if cn not in _existing_syns:
                        col_syn_items.append(f"Column '{cn}' ({c.get('Data Type','')}) in table '{c['Table Name']}'. Suggest a 1-3 word business-friendly synonym. Return ONLY the synonym.")
                if col_syn_items:
                    syn_ai["columns"] = _ai_generate(col_syn_items, "column synonyms")
                    print(f"  AI: {len(syn_ai['columns'])} column synonyms generated")

                msr_syn_items = []
                for _, m in mdf.iterrows():
                    mn = m["Measure Name"]
                    if mn not in _existing_syns:
                        msr_syn_items.append(f"Measure '{mn}' (DAX: {str(m.get('Measure Expression',''))[:60]}). Suggest a 1-3 word business-friendly synonym. Return ONLY the synonym.")
                if msr_syn_items:
                    syn_ai["measures"] = _ai_generate(msr_syn_items, "measure synonyms")
                    print(f"  AI: {len(syn_ai['measures'])} measure synonyms generated")
            except Exception:
                pass

        # Build synonym fix entries
        _syn_tidx = 0
        for _, t in tdf.iterrows():
            tn = t["Table Name"]
            if tn in _existing_syns:
                continue
            if syn_ai["tables"] and _syn_tidx < len(syn_ai["tables"]):
                proposed = str(syn_ai["tables"][_syn_tidx])[:60].strip()
                _syn_tidx += 1
                src = "AI"
            else:
                proposed = _smart_synonym(tn, "table")
                src = "Template"
            if proposed and proposed.lower() != tn.lower():
                fixes.append({"category": "Synonym", "item": f"Table: {tn}",
                              "current": "(none)", "proposed": proposed,
                              "source": src, "type": "synonym", "obj_type": "table",
                              "table": tn, "name": tn, "value": proposed, "risk": "Low"})

        _syn_cidx = 0
        _cdf_clean3 = cdf[~cdf["Column Name"].str.contains("RowNumber-|rowNumber-", na=False)] if "Column Name" in cdf.columns else cdf
        for _, c in _cdf_clean3.head(30).iterrows():
            cn = c["Column Name"]
            tn = c["Table Name"]
            if cn in _existing_syns:
                continue
            if syn_ai["columns"] and _syn_cidx < len(syn_ai["columns"]):
                proposed = str(syn_ai["columns"][_syn_cidx])[:60].strip()
                _syn_cidx += 1
                src = "AI"
            else:
                proposed = _smart_synonym(cn, "column")
                src = "Template"
            if proposed and proposed.lower() != cn.lower():
                fixes.append({"category": "Synonym", "item": f"Column: {tn}.{cn}",
                              "current": "(none)", "proposed": proposed,
                              "source": src, "type": "synonym", "obj_type": "column",
                              "table": tn, "name": cn, "value": proposed, "risk": "Low"})

        _syn_midx = 0
        for _, m in mdf.iterrows():
            mn = m["Measure Name"]
            if mn in _existing_syns:
                continue
            if syn_ai["measures"] and _syn_midx < len(syn_ai["measures"]):
                proposed = str(syn_ai["measures"][_syn_midx])[:60].strip()
                _syn_midx += 1
                src = "AI"
            else:
                proposed = _smart_synonym(mn, "measure")
                src = "Template"
            if proposed and proposed.lower() != mn.lower():
                fixes.append({"category": "Synonym", "item": f"Measure: {mn}",
                              "current": "(none)", "proposed": proposed,
                              "source": src, "type": "synonym", "obj_type": "measure",
                              "table": m.get("Table Name", ""), "name": mn,
                              "value": proposed, "risk": "Low"})
    except Exception as _syn_e:
        print(f"  Synonym generation skipped: {str(_syn_e)[:80]}")

    # ── AI Instructions (Prep data for AI) ──
    # Generate model-level AI instructions from schema metadata
    try:
        _has_instr = False
        with connect_semantic_model(DATASET, readonly=True, workspace=WORKSPACE) as _tom_instr:
            for ann in _tom_instr.model.Annotations:
                if str(ann.Name) in ("__PBI_CopilotInstructions", "QueryInsightsInstructions"):
                    if str(ann.Value).strip():
                        _has_instr = True
                        break
        if not _has_instr:
            # Build AI instructions from model metadata
            _tnames = tdf["Table Name"].tolist() if "Table Name" in tdf.columns else []
            _mnames = mdf["Measure Name"].tolist() if "Measure Name" in mdf.columns else []
            _fact_tables = [t for t in _tnames if t.lower().startswith("fact")]
            _dim_tables = [t for t in _tnames if t.lower().startswith(("dim","dimension"))]

            _instr_parts = []
            _instr_parts.append(f"This semantic model contains {len(_tnames)} tables and {len(_mnames)} measures.")
            if _fact_tables:
                _instr_parts.append(f"Fact tables: {', '.join(_fact_tables[:5])}.")
            if _dim_tables:
                _instr_parts.append(f"Dimension tables: {', '.join(_dim_tables[:5])}.")
            if _mnames:
                _key_measures = [m for m in _mnames if any(k in m.lower() for k in ["revenue","profit","total","count","margin"])]
                if _key_measures:
                    _instr_parts.append(f"Key business measures: {', '.join(_key_measures[:8])}.")
            _instr_parts.append("When answering questions, prefer measures over raw column aggregations.")
            _instr_parts.append("Use dimension tables for filtering and grouping, fact tables for metrics.")

            _ai_instr_text = " ".join(_instr_parts)

            # Read CURRENT AI instructions from model's LinguisticMetadata
            _current_instr_full = ""
            _current_instr_display = "(none)"
            try:
                with connect_semantic_model(DATASET, readonly=True, workspace=WORKSPACE) as _tom_cur:
                    for _c in _tom_cur.model.Cultures:
                        if str(_c.Name) == "en-US" and _c.LinguisticMetadata and _c.LinguisticMetadata.Content:
                            _lm_cur = json.loads(str(_c.LinguisticMetadata.Content))
                            _ci_cur = _lm_cur.get("CustomInstructions", "")
                            if _ci_cur and str(_ci_cur).strip():
                                _current_instr_full = str(_ci_cur)
                                _current_instr_display = _current_instr_full[:120] + ("..." if len(_current_instr_full) > 120 else "")
                            break
            except Exception:
                pass

            # Also generate via AI if available — with RICH CONTEXT
            if USE_AI_DESCRIPTIONS:
                try:
                    # Build rich model brief for LLM
                    _tdf_full = meta.get("tables", pd.DataFrame())
                    _cdf_full = meta.get("columns", pd.DataFrame())
                    _rels_full = meta.get("relationships", pd.DataFrame())

                    # Tables with descriptions (if any)
                    _table_ctx_lines = []
                    if len(_tdf_full) > 0 and "Table Name" in _tdf_full.columns:
                        for _, _tr in _tdf_full.head(12).iterrows():
                            _tn = str(_tr["Table Name"])
                            _td = str(_tr.get("Description","") or "").strip()
                            if _td:
                                _table_ctx_lines.append(f"  - {_tn}: {_td[:80]}")
                            else:
                                # Show column count instead
                                _n_cols = len(_cdf_full[_cdf_full.get("Table Name","") == _tn]) if "Table Name" in _cdf_full.columns else 0
                                _table_ctx_lines.append(f"  - {_tn} ({_n_cols} columns)")

                    # Measures with expressions
                    _msr_ctx_lines = []
                    if "Measure Name" in mdf.columns:
                        for _, _mr in mdf.head(12).iterrows():
                            _mn = str(_mr["Measure Name"])
                            _me = str(_mr.get("Measure Expression","") or "").strip()[:100]
                            if _me:
                                _msr_ctx_lines.append(f"  - {_mn}: {_me}")
                            else:
                                _msr_ctx_lines.append(f"  - {_mn}")

                    # Relationships
                    _rels_ctx_lines = []
                    if len(_rels_full) > 0:
                        for _, _rr in _rels_full.head(10).iterrows():
                            _ft = _rr.get("From Table","")
                            _fc = _rr.get("From Column","")
                            _tt = _rr.get("To Table","")
                            _tc = _rr.get("To Column","")
                            if _ft and _tt:
                                _rels_ctx_lines.append(f"  - {_ft}[{_fc}] -> {_tt}[{_tc}]")

                    # User context (from the AI_USER_CONTEXT textarea)
                    _user_ctx_block = ""
                    try:
                        _uc = globals().get("AI_USER_CONTEXT", "") or ""
                        if _uc.strip():
                            _user_ctx_block = f"\n\nUser-provided business context:\n{_uc.strip()[:400]}"
                    except Exception:
                        pass

                    # Optionally include current instructions for improvement-style prompting
                    _cur_ctx = ""
                    if _current_instr_full:
                        _cur_ctx = f"\n\nCurrent instructions (improve/replace these):\n{_current_instr_full[:300]}"

                    _ai_prompt = (
                        "Write 3-4 concise sentences of Power BI Copilot instructions for THIS specific semantic model. "
                        "Be specific — mention actual table and measure names.\n\n"
                        f"Model structure:\n"
                        f"- Fact tables: {', '.join(_fact_tables[:5]) or 'none detected'}\n"
                        f"- Dimension tables: {', '.join(_dim_tables[:10]) or 'none detected'}\n\n"
                        f"Tables:\n" + "\n".join(_table_ctx_lines[:8]) + "\n\n"
                        f"Key measures:\n" + "\n".join(_msr_ctx_lines[:10]) + "\n\n"
                        f"Relationships:\n" + ("\n".join(_rels_ctx_lines[:10]) if _rels_ctx_lines else "  (none)")
                        + _user_ctx_block + _cur_ctx + "\n\n"
                        "Write instructions covering: (1) business domain of this model, "
                        "(2) which measures to prefer for Q&A, (3) any important rules or terminology. "
                        "Be specific — no generic boilerplate. Return only the instruction text."
                    )
                    _ai_result = _ai_generate([_ai_prompt], "AI instructions")
                    if _ai_result and _ai_result[0]:
                        _ai_instr_text = str(_ai_result[0])[:500]
                        _instr_source = "AI"
                    else:
                        _instr_source = "Template"
                except Exception:
                    _instr_source = "Template"
            else:
                _instr_source = "Template"

            fixes.append({
                "category": "AI Prep", "item": "Model Description (Copilot context)",
                "current": _current_instr_full if _current_instr_full else "(none)",
                "proposed": _ai_instr_text,
                "source": _instr_source, "type": "model_description",
                "value": _ai_instr_text, "risk": "Low"
            })

            # ── Role descriptions (auto-generate for roles without one) ──
            try:
                with connect_semantic_model(DATASET, readonly=True, workspace=WORKSPACE) as _tom_roles:
                    for _role in _tom_roles.model.Roles:
                        _rn = str(_role.Name)
                        _rd = str(_role.Description) if _role.Description else ""
                        if _rd and _rd.strip():
                            continue  # already has description
                        # Collect context: RLS filters + permission
                        _perm = str(_role.ModelPermission) if hasattr(_role, "ModelPermission") else "Read"
                        _rls_exprs = []
                        try:
                            for _tp in _role.TablePermissions:
                                _tn = str(_tp.Table.Name)
                                _filt = str(_tp.FilterExpression).strip() if _tp.FilterExpression else ""
                                if _filt:
                                    _rls_exprs.append(f"{_tn}: {_filt[:80]}")
                        except Exception:
                            pass

                        # Template description
                        _role_desc = f"Security role '{_rn}' ({_perm} permission)"
                        if _rls_exprs:
                            _role_desc += f" — restricts access via RLS on {len(_rls_exprs)} table(s)"
                        else:
                            _role_desc += " — no row-level filters (full read access to all tables)"
                        _role_src = "Template"

                        # AI override
                        if USE_AI_DESCRIPTIONS and globals().get("_AI_AVAILABLE", False):
                            try:
                                _role_prompt = (
                                    f"Generate a 1-sentence business description for this Power BI security role.\n\n"
                                    f"Role name: {_rn}\n"
                                    f"Permission: {_perm}\n"
                                    f"RLS filters: {'; '.join(_rls_exprs) if _rls_exprs else '(none — full access)'}\n\n"
                                    f"Describe who this role is intended for and what data they can see. "
                                    f"Return only the description text (no quotes, no prefix)."
                                )
                                _role_ai = _ai_generate([_role_prompt], f"role description for {_rn}")
                                if _role_ai and _role_ai[0]:
                                    _role_desc = str(_role_ai[0]).strip()
                                    _role_src = "AI"
                            except Exception:
                                pass

                        fixes.append({
                            "category": "Role Description",
                            "item": f"Role: {_rn}",
                            "current": "(none)",
                            "proposed": _role_desc,
                            "source": _role_src,
                            "type": "role_description",
                            "name": _rn,
                            "value": _role_desc,
                            "risk": "Low"
                        })
            except Exception:
                pass
    except Exception:
        pass

    # ── Schema Reduction (isAvailableInMDX + LinguisticMetadata Entity Visibility) ──
    # Mark technical columns as excluded from Copilot — BOTH hidden and visible tech columns
    # need this treatment (Is Hidden alone doesn't hide from Copilot/Q&A)
    if "Column Name" in cdf.columns:
        _tech_patterns = ["key", "sk", "fk", "id", "lineage", "rownumber", "_id", "wwi"]
        for _, c in cdf.iterrows():
            cn = str(c.get("Column Name", ""))
            tn = str(c.get("Table Name", ""))
            if "RowNumber" in cn: continue
            # NOTE: Don't skip if Is Hidden=True — hidden columns still need
            # isAvailableInMDX=False + Entity Visibility=Hidden for full Copilot exclusion
            cl = cn.lower().replace("_", "").replace(" ", "")
            # Match pattern as suffix OR prefix OR contains
            _is_tech = False
            for _p in _tech_patterns:
                if cl.endswith(_p) or cl.startswith(_p):
                    _is_tech = True
                    break
            if _is_tech:
                # Check if already excluded from MDX
                _already_excluded = False
                try:
                    _is_mdx = c.get("Is Available In MDX", True)
                    if not _is_mdx:
                        _already_excluded = True
                except Exception:
                    pass
                if not _already_excluded:
                    _is_hidden = bool(c.get("Is Hidden", False))
                    _current_state = "Hidden in model, visible to Copilot" if _is_hidden else "In Copilot scope"
                    fixes.append({
                        "category": "AI Prep", "item": f"Schema: {tn}.{cn}",
                        "current": _current_state, "proposed": "Excluded from Copilot (isAvailableInMDX + Entity Visibility)",
                        "source": "Rule", "type": "schema_reduction",
                        "table": tn, "name": cn, "risk": "Low"
                    })

    return fixes


# ── Refresh Helpers ──
def _refresh_data(ds, ws=None):
    """Full data refresh — reloads all data from sources."""
    print(f"  Refreshing data for '{ds}'...")
    try:
        fabric.refresh_dataset(ds, workspace=ws)
        print("  Data refresh completed successfully.")
        return True
    except Exception as e:
        print(f"  Data refresh failed: {str(e)[:200]}")
        return False

def _refresh_schema(ds, ws=None):
    """Schema-only refresh — syncs table/column schema with the underlying lakehouse.
    Only applicable for DirectLake models. For Import/DQ models, falls back to Calculate refresh.
    """
    print(f"  Refreshing schema for '{ds}'...")
    try:
        # Try DirectLake schema sync first (sempy_labs)
        try:
            from sempy_labs import update_direct_lake_model_lakehouse_schema
            update_direct_lake_model_lakehouse_schema(dataset=ds, workspace=ws)
            print("  DirectLake schema refresh completed — lakehouse schema synced.")
            return True
        except ImportError:
            pass
        except Exception as dl_err:
            if "direct lake" in str(dl_err).lower() or "not supported" in str(dl_err).lower():
                pass  # Not a DirectLake model, try Calculate
            else:
                print(f"  DirectLake schema sync warning: {str(dl_err)[:120]}")
        # Fallback: Calculate refresh (recalcs metadata without reloading source data)
        try:
            with connect_semantic_model(dataset=ds, workspace=ws, readonly=False) as tom:
                import Microsoft.AnalysisServices.Tabular as TOM_NS
                tom.model.RequestRefresh(TOM_NS.RefreshType.Calculate)
                tom.model.SaveChanges()
            print("  Schema refresh completed (Calculate mode).")
            return True
        except Exception as calc_err:
            print(f"  Calculate refresh failed: {str(calc_err)[:120]}")
            # Last fallback: standard refresh
            fabric.refresh_dataset(ds, workspace=ws)
            print("  Fell back to standard refresh.")
            return True
    except Exception as e:
        print(f"  Schema refresh failed: {str(e)[:200]}")
        return False

# ═══════════════════════════════════════════════════════
#  RUN EVERYTHING
# ═══════════════════════════════════════════════════════

def run_scan():
    global results, meta, fix_plan
    import time as _time
    _t0 = _time.time()
    print("Model Health & Security Suite")
    print("=" * 55)
    print(f"Model: {DATASET} | Workspace: {WORKSPACE or '(current)'}\n")

    meta = preflight(DATASET, WORKSPACE)
    if meta is None: return
    n_tables = len(meta['tables'])
    n_cols = len(meta['columns'])
    n_measures = len(meta['measures'])
    print(f"Connected: {n_tables} tables, {n_cols} columns, {n_measures} measures")
    if n_measures > 100:
        print(f"  Note: {n_measures} measures to test — this may take a few minutes.")
    print()

    results = {}
    tool_names = [
        # ── Unique to this suite ──
        "Copilot Readiness","Lineage","Test Framework","Model Diff",
        "Report Visuals","Security Audit","Security X-Ray","Data Quality","DAX Dependencies",
        # ── Powered by Semantic Link Labs ──
        "Best Practices","Vertipaq Performance","Direct Lake","Capacity"]
    tool_fns = [
        # ── Unique to this suite ──
        ("copilot_readiness", lambda: tool_copilot(DATASET, meta, WORKSPACE)),
        ("lineage", lambda: tool_lineage(DATASET, meta, WORKSPACE)),
        ("test_framework", lambda: tool_test_framework(DATASET, meta, WORKSPACE)),
        ("model_diff", lambda: tool_diff(DATASET, COMPARE_DATASET, meta, WORKSPACE, globals().get("COMPARE_WORKSPACE", WORKSPACE))),
        ("report_visuals", lambda: tool_report_visuals(DATASET, meta, WORKSPACE)),
        ("security_audit", lambda: tool_security_audit(DATASET, meta, WORKSPACE)),
        ("security_xray", lambda: tool_security_xray(DATASET, meta, WORKSPACE)),
        ("data_quality", lambda: tool_data_quality(DATASET, meta, WORKSPACE)),
        ("dax_dependencies", lambda: tool_dax_deps(DATASET, meta, WORKSPACE)),
        # ── Powered by Semantic Link Labs ──
        ("bpa", lambda: tool_bpa(DATASET, WORKSPACE)),
        ("vertipaq", lambda: tool_vertipaq(DATASET, WORKSPACE)),
        ("direct_lake", lambda: tool_direct_lake(DATASET, meta, WORKSPACE)),
        ("capacity", lambda: tool_capacity(DATASET, WORKSPACE)),
    ]
    results = {}
    _t0 = _time.time()
    _enabled = sum(1 for k,_ in tool_fns if TOOLS.get(k,True))
    _run_idx = 0
    for i, (key, fn) in enumerate(tool_fns):
        if not TOOLS.get(key, True):
            results[key] = make_result(tool_names[i], 0, [], ["Skipped (disabled in config)"], extra={"skipped":True})
            continue
        _run_idx += 1
        print(f"  [{_run_idx}/{_enabled}] Running {tool_names[i]}...", end=" ", flush=True)
        _ts = _time.time()
        try:
            results[key] = fn()
        except Exception as e:
            results[key] = make_result(key, 0, [], [f"Error: {str(e)[:150]}"])
        r = results[key]
        elapsed = _time.time() - _ts
        print(f"{r['score']:5.1f}% ({r['grade']}) [{elapsed:.1f}s]")

    # Overall
    ws2,wt=0,0
    for k,w in WEIGHTS.items():
        if k in results and not results[k].get("extra",{}).get("skipped",False):
            ws2+=results[k]["score"]*w; wt+=w
    overall=round(ws2/wt,1) if wt>0 else 0
    results["_overall"]={"score":overall,"grade":score_to_grade(overall),"model":DATASET}
    total_time = _time.time() - _t0

    def _gc(grade):
        if grade.startswith("A"): return "#107C10"
        if grade.startswith("B"): return "#0078D4"
        if grade.startswith("C"): return "#FFB900"
        if grade.startswith("D"): return "#E87400"
        return "#D13438"
    def _bar(sc):
        c=_gc(score_to_grade(sc)); w=max(1,sc*1.2)
        return '<div style="background:#EDEBE9;border-radius:4px;height:12px;width:120px;display:inline-block;vertical-align:middle"><div style="background:'+c+';border-radius:4px;height:12px;width:'+str(int(w))+'px"></div></div>'
    def _bdg(gr):
        return '<span style="background:'+_gc(gr)+';color:#fff;padding:2px 10px;border-radius:12px;font-weight:700;font-size:13px">'+gr+'</span>'

    oc=_gc(score_to_grade(overall))
    h='<div style="font-family:Segoe UI,sans-serif;max-width:960px">'
    # Header card
    h+='<div style="background:linear-gradient(135deg,#1B3A4B,#117865);border-radius:10px;padding:16px 20px;margin:12px 0;display:flex;align-items:center;gap:16px">'
    h+='<div style="font-size:36px;font-weight:800;color:#fff">'+str(overall)+'%</div>'
    h+='<div>'+_bdg(score_to_grade(overall))+'</div>'
    h+='<div style="flex:1;text-align:right"><div style="color:#fff;font-weight:600;font-size:14px">'+DATASET+'</div><div style="color:#7DBDAB;font-size:12px">'+str(int(total_time))+'s scan</div></div></div>'

    # Interpretation guide (collapsible)
    h+='<details style="background:#F0F6F5;border:1px solid #B3D9D2;border-radius:8px;margin:16px 0;font-size:13px;line-height:1.6">'
    h+='<summary style="padding:12px 16px;font-weight:700;font-size:14px;color:#005A9E;cursor:pointer">How to Read This Report</summary><div style="padding:0 16px 16px 16px">'
    h+='<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'
    h+='<div><b>Score Cards</b> &mdash; Each tool scores 0-100%. '
    h+='<span style="color:#107C10">A/A+</span> = excellent, '
    h+='<span style="color:#0078D4">B</span> = good, '
    h+='<span style="color:#FFB900">C</span> = needs attention, '
    h+='<span style="color:#D13438">D/F</span> = action required.</div>'
    h+='<div><b>Detailed Findings</b> &mdash; Expands each tool\'s checks. '
    h+='<span style="color:#D13438">&#9679; FAIL</span> = must fix, '
    h+='<span style="color:#FFB900">&#9679; WARN</span> = review recommended, '
    h+='<span style="color:#107C10">&#9679; PASS</span> = no action needed.</div>'
    h+='<div><b>Fix Plan</b> &mdash; Auto-generated fixes (descriptions, format strings, visibility). '
    h+='All are LOW risk (metadata only). Edit <code>SELECTED_FIXES</code> to choose which to apply.</div>'
    h+='<div><b>Model Diff</b> &mdash; Compares current model against <code>'+str(COMPARE_DATASET or "none")+'</code>. '
    h+='"New" = only in current model. "Only in compare" = missing from current. "Modified" = changed fields.</div>'
    h+='<div><b>Lineage</b> &mdash; Traces each measure back to its source tables and columns. '
    h+='Shows dependency count, blast radius (how many measures break if a column changes), and orphan measures.</div>'
    h+='<div><b>Report Visuals</b> &mdash; Scans connected Power BI reports for which measures/columns are used in visuals. '
    h+='Identifies broken references and unused measures not in any report.</div>'
    h+='</div></div></details>'


    # Score grid
    _sempy_tools={"Best Practices","Vertipaq Performance","Direct Lake","Capacity"}
    _tool_desc={"Test Framework":"DAX measure eval + relationship checks","Copilot Readiness":"Descriptions, synonyms, naming, visibility","Model Diff":"Differences vs compare model","Best Practices":"sempy_labs BPA rules","Vertipaq Performance":"sempy_labs memory analyzer","DAX Dependencies":"Measure complexity + blast radius","Data Quality":"Null rates, referential integrity","Direct Lake":"sempy_labs DL checks","Capacity":"sempy_labs workspace scan","Lineage":"Measure-to-source tracing + blast radius","Report Visuals":"Visual objects, broken refs, unused measures","Security Audit":"RLS/OLS, role members, permission matrix","Security X-Ray":"Effective access map \u2014 RLS+OLS+AAD groups+workspace roles \u2192 real users"}
    _tool_notes={
        "Copilot Readiness": "Measures how well your model supports Copilot, Q&A, and Power BI self-service. Focuses on descriptions, synonyms, hidden surrogate keys, format strings, and AI Prep (CustomInstructions + schema simplification).",
        "Lineage": "Traces where each measure gets its data from — sources, related tables, dependencies between measures. Identifies blast radius (columns/measures used by many others = fragile) and dead code (columns no measure references).",
        "Test Framework": "Validates that every DAX measure actually evaluates without error, and that relationships are configured correctly (active, right cardinality, right cross-filter direction). A healthy model has all measures passing and no broken relationships.",
        "Model Diff": "Compares this model against another model (same or different workspace). Catches drift between dev/prod environments — added/removed/changed tables, columns, measures, relationships.",
        "Report Visuals": "Scans reports connected to this semantic model. Finds broken references (visuals pointing at measures/columns that no longer exist) and unused measures (measures defined in the model but not used in any report).",
        "Security Audit": "Top-level compliance check: are roles defined, do they have descriptions, are members assigned, are RLS filters in place? This is the summary complement to Security X-Ray.",
        "Security X-Ray": "Deep security analysis. Combines RLS filters + OLS restrictions + role members + AAD group expansion + workspace roles into an effective access map. Flags users who bypass RLS via workspace Admin/Member/Contributor privileges — a critical blind spot in standard Power BI UI.",
        "Data Quality": "Referential integrity checks and null rate analysis. Catches orphan rows in fact tables, high null rates in key columns, and cardinality issues that affect report accuracy.",
        "DAX Dependencies": "Analyzes how measures depend on each other. High fan-out measures (used by many others) are fragile — one change cascades. High fan-in measures (calling many others) are complex. Essential for understanding change impact before modifying measures.",
        "Best Practices": "Runs the sempy_labs Best Practice Analyzer (BPA) — catches anti-patterns like FILTER+ALL, missing sort columns, COUNTROWS wrapped in CALCULATE, etc.",
        "Vertipaq Performance": "Memory analysis per table/column. Finds oversized dictionaries, high-cardinality columns, and unused columns that bloat the model without adding value.",
        "Direct Lake": "Checks Direct Lake compatibility. Calculated columns and calculated tables force fallback to import mode (slower). This tool identifies them.",
        "Capacity": "Lists all semantic models in the workspace with sizes and refresh status — useful for capacity planning.",
    }
    h+='<div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;margin:16px 0">'
    for _k,_r in results.items():
        if _k.startswith("_"): continue
        _skipped = _r.get("extra",{}).get("skipped",False)
        if _skipped: continue
        _sc=_r["score"]; _gr=_r["grade"]; _nm=_r["name"]; _gc2=_gc(_gr)
        if _nm in _sempy_tools: continue  # Show in separate section below
        _fails=len([d for d in _r.get("details",[]) if d.get("status") in ("FAIL","WARN")])
        if _fails:
            _itxt = '<span style="color:#D13438;font-size:12px">' + str(_fails) + ' issues</span>'
        else:
            _itxt = '<span style="color:#107C10;font-size:12px">No issues</span>'
        h+='<div style="border:1px solid #EDEBE9;border-radius:8px;padding:10px 12px;background:#fff;border-left:4px solid '+_gc2+'">'
        h+='<div style="font-weight:600;font-size:14px;margin-bottom:2px">'+_nm+'</div>'
        _is_sempy = _nm in _sempy_tools
        _desc_line = _tool_desc.get(_nm,"")
        if _is_sempy:
            h+='<div style="font-size:11px;color:#605E5C;margin-bottom:4px">'+_desc_line+' <span style="background:#D2ECE8;color:#117865;font-size:9px;padding:1px 6px;border-radius:8px;font-weight:600">sempy_labs</span></div>'
        else:
            h+='<div style="font-size:11px;color:#605E5C;margin-bottom:4px">'+_desc_line+'</div>'
        h+='<div style="font-size:22px;font-weight:700;color:'+_gc2+'">'+str(int(_sc))+'%</div>'
        h+='<div>'+_bar(_sc)+' '+_bdg(_gr)+'</div>'
        h+='<div style="margin-top:6px">'+_itxt+'</div></div>'
    h+='</div>'

    # ── Powered by Semantic Link Labs section ──
    h+='<div style="margin:16px 0 8px 0;font-size:11px;color:#605E5C;text-transform:uppercase;letter-spacing:1px;border-top:1px solid #EDEBE9;padding-top:12px">Powered by Semantic Link Labs</div>'
    h+='<div style="display:grid;grid-template-columns:1fr 1fr 1fr 1fr;gap:10px;margin:0 0 16px 0">'
    for _k2,_r2 in results.items():
        if _k2.startswith("_"): continue
        if _r2.get("extra",{}).get("skipped",False): continue
        _nm2 = _r2["name"]
        if _nm2 not in _sempy_tools: continue
        _sc2=_r2["score"]; _gr2=_r2["grade"]; _gc3=_gc(_gr2)
        h+='<div style="border:1px solid #D2ECE8;border-radius:6px;padding:10px;background:#F0F6F5;text-align:center">'
        h+='<div style="font-size:12px;font-weight:600;color:#117865">'+_nm2+'</div>'
        h+='<div style="font-size:22px;font-weight:700;color:'+_gc3+'">'+str(int(_sc2))+'% <span style="font-size:12px">'+_gr2+'</span></div>'
        h+='</div>'
    h+='</div>'

    _all_detail_dfs = []
        # Detailed findings — uses div sections (not <details> — unsupported in Fabric)
    h+='<details style="margin-top:24px"><summary style="font-size:18px;font-weight:700;border-bottom:2px solid #EDEBE9;padding-bottom:8px;cursor:pointer">Detailed Findings</summary><div style="margin-top:8px">'
    for _k,_r in results.items():
        if _k.startswith("_"): continue
        if _r.get("extra",{}).get("skipped",False): continue
        _dets=_r.get("details",[]); _recs=_r.get("recommendations",[])
        _fails=[d for d in _dets if d.get("status") in ("FAIL","WARN")]
        _passes=[d for d in _dets if d.get("status")=="PASS"]
        _infos=[d for d in _dets if d.get("status")=="INFO"]
        if not _dets and not _recs: continue
        _gc2=_gc(_r["grade"])
        # Section header (collapsible)
        _has_issues = len(_fails) > 0
        _open_attr = ''  # All tool sections collapsed by default
        h+='<details'+_open_attr+' style="margin:8px 0;border:1px solid #EDEBE9;border-radius:8px;overflow:hidden">'
        h+='<summary style="padding:10px 14px;background:#FAF9F8;font-weight:600;display:flex;align-items:center;gap:8px;cursor:pointer;font-size:13px">'
        h+='<span style="width:10px;height:10px;border-radius:50%;background:'+_gc2+';display:inline-block"></span>'
        h+=_r["name"]+' &mdash; '+str(int(_r["score"]))+'% ('+_r["grade"]+')'
        h+='<span style="margin-left:auto;font-weight:400;font-size:12px;color:#605E5C">'
        if _fails: h+='<span style="color:#D13438">'+str(len(_fails))+' fail</span> &middot; '
        if _infos: h+=str(len(_infos))+' info &middot; '
        h+=str(len(_passes))+' pass</span></summary>'
        h+='<div style="padding:10px 14px">'
        _tnote = _tool_notes.get(_r["name"], "")
        if _tnote:
            h+='<div style="font-size:12px;color:#605E5C;background:#FAF9F8;padding:8px 10px;border-radius:6px;margin-bottom:8px;border-left:3px solid #0078D4;line-height:1.5">'+_tnote+'</div>'

        # Stats pill row: quick overview of fail/warn/info/pass counts
        _warns_count = len([d for d in _dets if d.get("status") == "WARN"])
        _stats_parts = []
        if _fails:
            _stats_parts.append('<span style="background:#FDE7E8;color:#D13438;padding:4px 12px;border-radius:14px;font-weight:600;font-size:12px">&#9679; '+str(len(_fails))+' Fail</span>')
        if _warns_count:
            _stats_parts.append('<span style="background:#FFF8E0;color:#8A6D00;padding:4px 12px;border-radius:14px;font-weight:600;font-size:12px">&#9679; '+str(_warns_count)+' Warn</span>')
        if _infos:
            _stats_parts.append('<span style="background:#F0F6FF;color:#0078D4;padding:4px 12px;border-radius:14px;font-weight:600;font-size:12px">&#9679; '+str(len(_infos))+' Info</span>')
        if _passes:
            _stats_parts.append('<span style="background:#E6F2E6;color:#107C10;padding:4px 12px;border-radius:14px;font-weight:600;font-size:12px">&#10003; '+str(len(_passes))+' Pass</span>')
        if _stats_parts:
            h+='<div style="margin:8px 0 14px 0;display:flex;gap:6px;flex-wrap:wrap">'+''.join(_stats_parts)+'</div>'

        # ── Custom renderer: Security Audit pivot table ──
        if _r["name"] == "Security Audit":
            _roles_pivot = _r.get("extra", {}).get("roles_pivot", [])
            if _roles_pivot:
                h+='<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px;overflow:hidden">'
                h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F0F6F5;color:#117865;list-style:none;user-select:none">'
                h+='<span style="color:#117865;font-weight:700;margin-right:8px">&#9656;</span>'
                h+='Roles Pivot &mdash; one row per role <span style="opacity:0.7;font-weight:400">('+str(len(_roles_pivot))+' roles)</span>'
                h+='</summary><div style="padding:12px 14px;background:#FFF">'
                h+='<table style="width:100%;border-collapse:collapse;font-size:12px"><thead>'
                h+='<tr style="background:#FAF9F8"><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Role</th>'
                h+='<th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Permission</th>'
                h+='<th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Description</th>'
                h+='<th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Members</th>'
                h+='<th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">RLS</th>'
                h+='<th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">OLS</th></tr></thead><tbody>'
                for _rpi, _rpr in enumerate(_roles_pivot):
                    _bg = "#FFFFFF" if _rpi%2==0 else "#FAF9F8"
                    h+='<tr style="background:'+_bg+'">'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-weight:600">'+str(_rpr.get("Role",""))+'</td>'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9">'+str(_rpr.get("Permission",""))+'</td>'
                    _desc = str(_rpr.get("Description",""))
                    _desc_style = 'color:#8A6D00;font-style:italic' if _desc == '(none)' else 'color:#323130'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;'+_desc_style+'">'+_desc+'</td>'
                    # Mask members list
                    _members_str = str(_rpr.get("Members",""))
                    if MASK_PII and _members_str and _members_str != "(none)":
                        _mem_parts = [m.strip() for m in _members_str.split(",")]
                        _masked_parts = []
                        for _mp in _mem_parts:
                            if "@" in _mp:
                                _masked_parts.append(_mask_pii(_mp, "email"))
                            elif _mp.startswith("obj:") or _mp.count("-") >= 3:
                                _masked_parts.append(_mask_pii(_mp, "group"))
                            else:
                                _masked_parts.append(_mask_pii(_mp, "name"))
                        _members_str = ", ".join(_masked_parts)
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-size:11px">'+_members_str+'</td>'
                    _rls = str(_rpr.get("RLS",""))
                    _rls_style = 'color:#A19F9D' if _rls == '(none)' else 'color:#0078D4;font-family:Consolas,monospace;font-size:11px'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;'+_rls_style+';max-width:300px">'+_rls+'</td>'
                    _ols = str(_rpr.get("OLS",""))
                    _ols_style = 'color:#A19F9D' if _ols == '(none)' else 'color:#323130;font-size:11px'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;'+_ols_style+'">'+_ols+'</td>'
                    h+='</tr>'
                h+='</tbody></table></div></details>'

        # ── Custom renderer: Security X-Ray Effective Access Map tabs ──
        if _r["name"] == "Security X-Ray":
            _xray_data = _r.get("extra", {}).get("xray", {})
            _effective = _xray_data.get("effective", [])
            _rls_rows = _xray_data.get("rls", []) or []
            _ols_rows = _xray_data.get("ols", []) or []
            _wsa_rows = _xray_data.get("workspace_access", []) or []
            _mem_rows = _xray_data.get("members", []) or []
            _dyn_roles = _xray_data.get("dynamic_roles", []) or []
            _graph_ok = _xray_data.get("graph_available", False)
            _rest_ok = _xray_data.get("fabric_rest_available", False)
            _n_bypass = sum(1 for _er in _effective if "bypass" in str(_er.get("Effective_View","")).lower())
            _n_users = len({_er.get("User","") for _er in _effective})

            # 6 stat cards + Dynamic card
            _crd = lambda l,v,c="#1B3A4B": '<div style="flex:1 1 0;min-width:90px;text-align:center;padding:8px 6px;background:#fff;border:1px solid #EDEBE9;border-radius:6px;margin:3px"><div style="font-size:9px;color:#605E5C;text-transform:uppercase;letter-spacing:0.5px">'+l+'</div><div style="font-size:18px;font-weight:700;color:'+c+'">'+str(v)+'</div></div>'
            h+='<div style="display:flex;flex-wrap:wrap;margin:8px 0 10px 0">'
            h+=_crd("Users", _n_users)
            h+=_crd("RLS Filters", len(_rls_rows))
            h+=_crd("OLS", len(_ols_rows))
            h+=_crd("WS Roles", len(_wsa_rows))
            h+=_crd("Members", len(_mem_rows))
            h+=_crd("Bypass", _n_bypass, "#D13438" if _n_bypass>0 else "#107C10")
            h+=_crd("Dynamic", len(_dyn_roles), "#FFB900" if _dyn_roles else "#605E5C")
            h+='</div>'

            # Availability bar
            h+='<div style="font-size:11px;color:#605E5C;padding:6px 10px;background:#FAF9F8;border-radius:6px;margin-bottom:12px;border-left:3px solid #0078D4">'
            h+='Graph: <span style="color:'+("#107C10" if _graph_ok else "#FFB900")+'">'+("available" if _graph_ok else "unavailable")+'</span>'
            h+=' &middot; Fabric REST: <span style="color:'+("#107C10" if _rest_ok else "#FFB900")+'">'+("available" if _rest_ok else "unavailable")+'</span>'
            if _dyn_roles: h+=' &middot; Dynamic roles: '+", ".join(_dyn_roles)
            h+='</div>'

            if _effective:
                # Apply prettification to effective rows too (in case cache has names)
                try:
                    import re as _re_xr
                    _op = _re_xr.compile(r'obj:([0-9a-f-]{30,})@[0-9a-f-]+')
                    for _er in _effective:
                        for _f in ("User","Source"):
                            _v = str(_er.get(_f,""))
                            if "obj:" in _v:
                                _er[_f] = _op.sub(lambda m: globals().get("_group_name_cache",{}).get(m.group(1)) or m.group(0), _v)
                except Exception:
                    pass

                # Bucket by severity
                _bypass_rows = [e for e in _effective if "bypass" in str(e.get("Effective_View","")).lower()]
                _filtered_rows = [e for e in _effective if "filtered" in str(e.get("Effective_View","")).lower() or "hidden" in str(e.get("Effective_View","")).lower() or "dynamic" in str(e.get("Effective_View","")).lower()]
                _norestrict_rows = [e for e in _effective if e not in _bypass_rows and e not in _filtered_rows]

                def _render_access_table(_rows):
                    if not _rows:
                        return '<div style="padding:16px;color:#605E5C;text-align:center;font-style:italic">No users in this category</div>'
                    _h = '<table style="width:100%;border-collapse:collapse;font-size:12px">'
                    _h += '<tr style="background:#FAF9F8"><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">User</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Source</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Workspace Role</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Model Role</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">RLS Filter</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Effective View</th></tr>'
                    for _ri, _row in enumerate(_rows):
                        _bg = "#FFFFFF" if _ri%2==0 else "#FAF9F8"
                        _ev = str(_row.get("Effective_View",""))
                        _ev_color = "#D13438" if "bypass" in _ev.lower() else ("#107C10" if "filtered" in _ev.lower() or "hidden" in _ev.lower() else "#605E5C")
                        _h += '<tr style="background:'+_bg+'">'
                        _u_val = str(_row.get("User",""))
                        _u_kind = "email" if "@" in _u_val else "name"
                        _user_masked = _mask_pii(_u_val, _u_kind)
                        _src_val = str(_row.get("Source",""))
                        _src_masked = _mask_pii(_src_val)
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-weight:500">'+_user_masked+'</td>'
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-size:11px">'+_src_masked+'</td>'
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9">'+str(_row.get("Workspace_Role",""))+'</td>'
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9">'+str(_row.get("Model_Role",""))+'</td>'
                        _rls_filter_val = str(_row.get("RLS_Filter","") or "")
                        if _rls_filter_val.strip() in ("", "\u2014", "-", "None"):
                            _rls_filter_val = '<span style="color:#A19F9D;font-style:italic">No filter condition applied</span>'
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-family:Consolas,monospace;font-size:10px;max-width:200px">'+_rls_filter_val+'</td>'
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;color:'+_ev_color+';font-weight:600">'+_ev+'</td>'
                        _h += '</tr>'
                    _h += '</table>'
                    return _h

                h+='<details style="margin:8px 0;border:1px solid #D13438;border-radius:8px;overflow:hidden">'
                h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#FDE7E8;color:#D13438;list-style:none">'
                h+='<span style="color:#D13438;font-weight:700;margin-right:8px">&#9656;</span>'
                h+='Effective Access Map &mdash; Bypass users <span style="opacity:0.7;font-weight:400">('+str(len(_bypass_rows))+' users)</span>'
                if _bypass_rows: h+=' <span style="margin-left:auto;color:#D13438;font-weight:700">&#9679; Critical</span>'
                h+='</summary><div style="padding:12px 14px;background:#FFF">'+_render_access_table(_bypass_rows)+'</div></details>'

                h+='<details style="margin:8px 0;border:1px solid #D2ECE8;border-radius:8px;overflow:hidden">'
                h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F0F6F5;color:#107C10;list-style:none">'
                h+='<span style="color:#107C10;font-weight:700;margin-right:8px">&#9656;</span>'
                h+='Effective Access Map &mdash; Filtered users <span style="opacity:0.7;font-weight:400">('+str(len(_filtered_rows))+' users)</span>'
                h+='</summary><div style="padding:12px 14px;background:#FFF">'+_render_access_table(_filtered_rows)+'</div></details>'

                h+='<details style="margin:8px 0;border:1px solid #EDEBE9;border-radius:8px;overflow:hidden">'
                h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F3F2F1;color:#605E5C;list-style:none">'
                h+='<span style="color:#605E5C;font-weight:700;margin-right:8px">&#9656;</span>'
                h+='Effective Access Map &mdash; No Restrictions <span style="opacity:0.7;font-weight:400">('+str(len(_norestrict_rows))+' users)</span>'
                h+='</summary><div style="padding:12px 14px;background:#FFF">'+_render_access_table(_norestrict_rows)+'</div></details>'

            # Workspace Role Assignments by role type
            _wsa = _xray_data.get("workspace_access", [])
            if _wsa:
                try:
                    _op = _re_xr.compile(r'obj:([0-9a-f-]{30,})@[0-9a-f-]+')
                    for _wr in _wsa:
                        for _f in ("Principal_Name","Email"):
                            _v = str(_wr.get(_f,""))
                            if "obj:" in _v:
                                _wr[_f] = _op.sub(lambda m: globals().get("_group_name_cache",{}).get(m.group(1)) or m.group(0), _v)
                except Exception:
                    pass
                _by_role = {}
                for _wr in _wsa:
                    _role_type = str(_wr.get("Workspace_Role","Unknown"))
                    _by_role.setdefault(_role_type, []).append(_wr)

                _role_order = ["Admin", "Member", "Contributor", "Viewer"]
                _ordered_keys = [k for k in _role_order if k in _by_role] + [k for k in _by_role if k not in _role_order]

                h+='<details style="margin:12px 0 8px 0;border:1px solid #EDEBE9;border-radius:8px;overflow:hidden">'
                h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F3F2F1;color:#323130;list-style:none">'
                h+='<span style="color:#117865;font-weight:700;margin-right:8px">&#9656;</span>'
                h+='Workspace Role Assignments <span style="opacity:0.7;font-weight:400">('+str(len(_wsa))+' total)</span>'
                h+='</summary><div style="padding:12px 14px;background:#FFF">'
                for _rk in _ordered_keys:
                    _rows = _by_role[_rk]
                    _rk_color = "#D13438" if _rk=="Admin" else ("#FFB900" if _rk in ("Member","Contributor") else "#117865")
                    h+='<details style="margin:6px 0;border:1px solid #EDEBE9;border-radius:6px">'
                    h+='<summary style="padding:8px 12px;cursor:pointer;font-size:12px;font-weight:600;background:#FAF9F8;color:'+_rk_color+';list-style:none">'
                    h+='<span style="margin-right:8px">&#9656;</span>'
                    h+=_rk+' <span style="opacity:0.7;font-weight:400">('+str(len(_rows))+')</span>'
                    h+='</summary><div style="padding:8px 12px">'
                    h+='<table style="width:100%;border-collapse:collapse;font-size:12px">'
                    h+='<tr style="background:#FAF9F8"><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Principal Name</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Type</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Email</th></tr>'
                    # Resolve expanded_groups once per renderer (for nested user display)
                    _exp_groups = _xray_data.get("expanded_groups", {}) or {}
                    for _wri, _wr in enumerate(_rows):
                        _bg = "#FFFFFF" if _wri%2==0 else "#FAF9F8"
                        _ptype = str(_wr.get("Principal_Type",""))
                        _pid = str(_wr.get("Principal_ID",""))
                        _pname = str(_wr.get("Principal_Name",""))
                        _pemail = str(_wr.get("Email",""))
                        # Apply PII masking
                        _pname_masked = _mask_pii(_pname, "group" if "group" in _ptype.lower() else "name")
                        _pemail_masked = _mask_pii(_pemail, "email")
                        h+='<tr style="background:'+_bg+'">'
                        h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-weight:500">'+_pname_masked+'</td>'
                        h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9">'+_ptype+'</td>'
                        h+='<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-size:11px">'+_pemail_masked+'</td>'
                        h+='</tr>'
                        # If this is a Group, nest expanded users as sub-rows
                        if "group" in _ptype.lower() and _pid in _exp_groups and _exp_groups[_pid]:
                            _members = _exp_groups[_pid]
                            for _mem in _members[:20]:
                                _upn = _mem.get("upn") or _mem.get("displayName","")
                                _dn = _mem.get("displayName","") or _upn
                                _dn_masked = _mask_pii(_dn, "name")
                                _upn_masked = _mask_pii(_upn, "email")
                                h+='<tr style="background:'+_bg+';font-size:11px">'
                                h+='<td style="padding:4px 10px 4px 28px;border-bottom:1px solid #EDEBE9;color:#605E5C">&#8627; '+_dn_masked+'</td>'
                                h+='<td style="padding:4px 10px;border-bottom:1px solid #EDEBE9;color:#605E5C;font-style:italic">Member</td>'
                                h+='<td style="padding:4px 10px;border-bottom:1px solid #EDEBE9;color:#605E5C;font-size:10px">'+_upn_masked+'</td>'
                                h+='</tr>'
                            if len(_members) > 20:
                                h+='<tr style="background:'+_bg+'"><td colspan="3" style="padding:4px 10px 4px 28px;color:#605E5C;font-size:10px;font-style:italic">... and '+str(len(_members)-20)+' more member(s)</td></tr>'
                    h+='</table></div></details>'
                h+='</div></details>'

            # ── Additional flat tables: RLS Filters, OLS Restrictions, Raw Role Members ──
            def _flat_tbl(_rows, _title, _hide=None, _empty_msg="No data"):
                if not _rows:
                    return ''
                _hide = _hide or set()
                _cols = [k for k in _rows[0].keys() if k not in _hide]
                _h = '<details style="margin:8px 0;border:1px solid #EDEBE9;border-radius:8px">'
                _h += '<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F3F2F1;color:#323130;list-style:none">'
                _h += '<span style="color:#117865;font-weight:700;margin-right:8px">&#9656;</span>'
                _h += _title+' <span style="opacity:0.7;font-weight:400">('+str(len(_rows))+')</span>'
                _h += '</summary><div style="padding:12px 14px;background:#FFF">'
                _h += '<div style="overflow-x:auto"><table style="width:100%;border-collapse:collapse;font-size:12px">'
                _h += '<tr style="background:#FAF9F8">'
                for _k in _cols:
                    _h += '<th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">'+str(_k).replace("_"," ")+'</th>'
                _h += '</tr>'
                for _ri, _row in enumerate(_rows[:50]):
                    _bg = "#FFFFFF" if _ri%2==0 else "#FAF9F8"
                    _h += '<tr style="background:'+_bg+'">'
                    for _k in _cols:
                        _v = str(_row.get(_k, "") or "")
                        if _v.lower() == "none" or _v == "nan": _v = "\u2014"
                        _h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-size:11px">'+_v[:200]+'</td>'
                    _h += '</tr>'
                if len(_rows) > 50:
                    _h += '<tr><td colspan="'+str(len(_cols))+'" style="padding:6px 10px;color:#605E5C;font-style:italic">... and '+str(len(_rows)-50)+' more rows</td></tr>'
                _h += '</table></div></div></details>'
                return _h

            h += _flat_tbl(_rls_rows, "RLS Filters")
            h += _flat_tbl(_ols_rows, "OLS Restrictions")
            # Raw Role Members — custom renderer with nested users for groups
            if _mem_rows:
                _exp_groups_mem = _xray_data.get("expanded_groups", {}) or {}
                h += '<details style="margin:8px 0;border:1px solid #EDEBE9;border-radius:8px">'
                h += '<summary style="padding:10px 14px;cursor:pointer;font-size:13px;font-weight:600;background:#F3F2F1;color:#323130;list-style:none">'
                h += '<span style="color:#117865;font-weight:700;margin-right:8px">&#9656;</span>'
                h += 'Raw Role Members <span style="opacity:0.7;font-weight:400">('+str(len(_mem_rows))+')</span>'
                h += '</summary><div style="padding:12px 14px;background:#FFF">'
                h += '<div style="overflow-x:auto"><table style="width:100%;border-collapse:collapse;font-size:12px">'
                h += '<tr style="background:#FAF9F8"><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Role</th>'
                h += '<th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Member Name</th>'
                h += '<th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Member Type</th></tr>'
                for _mri, _mrow in enumerate(_mem_rows):
                    _bg = "#FFFFFF" if _mri%2==0 else "#FAF9F8"
                    _m_role = str(_mrow.get("Role",""))
                    _m_name = str(_mrow.get("Member_Name",""))
                    _m_type = str(_mrow.get("Member_Type",""))
                    _m_id = str(_mrow.get("Member_ID",""))
                    # Type display: 1=User, 2=Group, 3=External
                    _m_type_display = _m_type
                    _is_group_type_for_mask = _m_type in ("2", "3") or "group" in _m_type.lower()
                    if _m_type == "1": _m_type_display = "User"
                    elif _m_type == "2": _m_type_display = "Group"
                    elif _m_type == "3": _m_type_display = "External/SPN"
                    # Mask member name appropriately
                    _m_name_masked = _mask_pii(_m_name, "group" if _is_group_type_for_mask else ("email" if "@" in _m_name else "name"))
                    h += '<tr style="background:'+_bg+'">'
                    h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-weight:500">'+_m_role+'</td>'
                    h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9;font-size:11px">'+_m_name_masked+'</td>'
                    h += '<td style="padding:6px 10px;border-bottom:1px solid #EDEBE9">'+_m_type_display+'</td>'
                    h += '</tr>'
                    # If this is a group and we have expanded users, show them nested
                    if _is_group_type_for_mask and _m_id and _m_id in _exp_groups_mem and _exp_groups_mem[_m_id]:
                        _members = _exp_groups_mem[_m_id]
                        for _user in _members[:20]:
                            _upn = _user.get("upn") or _user.get("displayName","")
                            _dn = _user.get("displayName","") or _upn
                            _dn_masked = _mask_pii(_dn, "name")
                            _upn_masked = _mask_pii(_upn, "email")
                            h += '<tr style="background:'+_bg+';font-size:11px">'
                            h += '<td style="padding:4px 10px;border-bottom:1px solid #EDEBE9;color:#A19F9D">&nbsp;</td>'
                            h += '<td style="padding:4px 10px 4px 28px;border-bottom:1px solid #EDEBE9;color:#605E5C">&#8627; '+_dn_masked+' <span style="color:#A19F9D;font-size:10px">('+_upn_masked+')</span></td>'
                            h += '<td style="padding:4px 10px;border-bottom:1px solid #EDEBE9;color:#605E5C;font-style:italic">Member</td>'
                            h += '</tr>'
                        if len(_members) > 20:
                            h += '<tr style="background:'+_bg+'"><td></td><td colspan="2" style="padding:4px 10px 4px 28px;color:#605E5C;font-size:10px;font-style:italic">... and '+str(len(_members)-20)+' more member(s)</td></tr>'
                h += '</table></div></div></details>'

        # ALL items, sorted, then grouped by category for tools with many distinct categories.
        _all_sorted = sorted(_dets, key=lambda d: {"FAIL":0,"WARN":1,"INFO":2,"PASS":3}.get(d.get("status","PASS"),4))
        # Skip the generic item listing for security tools (custom renderer already covers everything)
        if _r["name"] in ("Security Audit", "Security X-Ray"):
            _all_sorted = []  # forces the generic accordion to be skipped entirely
        if _all_sorted:
            # Count distinct non-empty categories
            _cats = {}
            for _d in _all_sorted:
                _c = str(_d.get("category","") or "Other").strip()
                if not _c:
                    _c = "Other"
                _cats.setdefault(_c, []).append(_d)
            # Tools that benefit from category grouping (many distinct categories)
            # Security Audit and Security X-Ray have custom pivot/tab renderers above — skip generic accordion
            _skip_generic = _r["name"] in ("Security Audit", "Security X-Ray")
            _group_by_cat = (not _skip_generic) and len(_cats) > 1 and len(_all_sorted) > 3

            def _render_row(_d, _di):
                _st = _d.get("status","PASS")
                if _st=="FAIL": _ic="<span style=\"color:#D13438\">&#9679;</span>"; _rbg="#FDE7E8"
                elif _st=="WARN": _ic="<span style=\"color:#FFB900\">&#9679;</span>"; _rbg="#FFF8E0"
                elif _st=="INFO": _ic="<span style=\"color:#0078D4\">&#9679;</span>"; _rbg="#F0F6F5"
                else: _ic="<span style=\"color:#107C10;font-size:16px;font-weight:700\">&#10003;</span>"; _rbg="#E6F2E6" if _di%2==0 else "#FAF9F8"
                _row = '<tr style="background:'+_rbg+'"><td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;white-space:nowrap">'+_ic+' '+_st+'</td>'
                _row += '<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;font-weight:500">'+str(_d["item"])+'</td>'
                _row += '<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;color:#605E5C;max-width:400px">'+str(_d.get("info",""))+'</td></tr>'
                return _row

            if _group_by_cat:
                # Render each category as its own collapsible sub-section
                # Category order: by severity (categories with FAILs first, then WARNs, then INFOs)
                def _cat_severity(items):
                    _s = {d.get("status","PASS") for d in items}
                    if "FAIL" in _s: return 0
                    if "WARN" in _s: return 1
                    if "INFO" in _s: return 2
                    return 3
                _cat_order = sorted(_cats.items(), key=lambda kv: (_cat_severity(kv[1]), -len(kv[1])))

                # Vertical accordion: each category is a full-width expandable row with chevron
                h+='<div style="margin:8px 0;display:block">'
                for _cat_name, _cat_items in _cat_order:
                    _cat_fails = sum(1 for d in _cat_items if d.get("status") == "FAIL")
                    _cat_warns = sum(1 for d in _cat_items if d.get("status") == "WARN")
                    _cat_passes = sum(1 for d in _cat_items if d.get("status") == "PASS")
                    # Severity color
                    if _cat_fails > 0:
                        _hdr_bg = "#FDE7E8"; _hdr_border = "#D13438"; _hdr_color = "#D13438"; _chevron_color = "#D13438"
                    elif _cat_warns > 0:
                        _hdr_bg = "#FFF8E0"; _hdr_border = "#FFB900"; _hdr_color = "#8A6D00"; _chevron_color = "#8A6D00"
                    else:
                        _hdr_bg = "#F3F2F1"; _hdr_border = "#EDEBE9"; _hdr_color = "#323130"; _chevron_color = "#117865"
                    h+='<details style="display:block;margin:6px 0;border:1px solid '+_hdr_border+';border-radius:8px;overflow:hidden">'
                    h+='<summary style="padding:10px 14px;cursor:pointer;font-size:13px;color:'+_hdr_color+';font-weight:600;list-style:none;user-select:none;background:'+_hdr_bg+';display:flex;align-items:center;gap:10px">'
                    h+='<span style="color:'+_chevron_color+';font-size:12px;font-weight:700;min-width:14px;display:inline-block" class="chev">&#9656;</span>'
                    h+='<span>'+_cat_name+'</span>'
                    h+='<span style="opacity:0.7;font-weight:400;font-size:12px">('+str(len(_cat_items))+')</span>'
                    if _cat_fails > 0: h+=' <span style="color:#D13438;font-weight:700;margin-left:auto">&#9679; '+str(_cat_fails)+' fail</span>'
                    elif _cat_warns > 0: h+=' <span style="color:#FFB900;font-weight:700;margin-left:auto">&#9679; '+str(_cat_warns)+' warn</span>'
                    else: h+=' <span style="color:#107C10;font-weight:700;margin-left:auto;font-size:14px">&#10003; '+str(_cat_passes)+' pass</span>'
                    h+='</summary>'
                    h+='<div style="padding:10px 14px;background:#FFFFFF">'
                    h+='<table style="width:100%;border-collapse:collapse;font-size:13px">'
                    h+='<tr style="background:#FAF9F8"><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Status</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">Item</th><th style="text-align:left;padding:6px 10px;border-bottom:1px solid #C8C6C4">' + ("Sample Output" if _r["name"] == "Test Framework" else "Info") + '</th></tr>'
                    for _di, _d in enumerate(_cat_items[:30]):
                        h+=_render_row(_d, _di)
                    if len(_cat_items) > 30:
                        h+='<tr><td colspan="3" style="padding:6px 10px;color:#605E5C;font-style:italic">... and '+str(len(_cat_items)-30)+' more</td></tr>'
                    h+='</table></div></details>'
                h+='</div>'
            else:
                # Original single-table rendering
                h+='<table style="width:100%;border-collapse:collapse;font-size:13px;margin-bottom:8px">'
                h+='<tr style="background:#F3F2F1"><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Status</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Category</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Item</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">' + ("Sample Output" if _r["name"] == "Test Framework" else "Info") + '</th></tr>'
                for _di,_d in enumerate(_all_sorted[:40]):
                    _st = _d.get("status","PASS")
                    if _st=="FAIL": _ic="<span style=\"color:#D13438\">&#9679;</span>"; _rbg="#FDE7E8"
                    elif _st=="WARN": _ic="<span style=\"color:#FFB900\">&#9679;</span>"; _rbg="#FFF8E0"
                    elif _st=="INFO": _ic="<span style=\"color:#0078D4\">&#9679;</span>"; _rbg="#F0F6F5" if _di%2==0 else "#F0F6F5"
                    else: _ic="<span style=\"color:#107C10;font-size:16px;font-weight:700\">&#10003;</span>"; _rbg="#E6F2E6" if _di%2==0 else "#FAF9F8"
                    h+='<tr style="background:'+_rbg+'"><td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;white-space:nowrap">'+_ic+' '+_st+'</td>'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8">'+str(_d.get("category",""))+'</td>'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;font-weight:500">'+str(_d["item"])+'</td>'
                    h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;color:#605E5C;max-width:300px;overflow:hidden;text-overflow:ellipsis">'+str(_d.get("info",""))+'</td></tr>'
                if len(_all_sorted)>40:
                    h+='<tr><td colspan="4" style="padding:6px 10px;color:#605E5C;font-style:italic">... and '+str(len(_all_sorted)-40)+' more (see DataFrame below)</td></tr>'
                h+='</table>'
        # Recommendations
        if _recs:
            h+='<div style="background:#F0F6F5;border-radius:6px;padding:10px 14px;margin-top:8px">'
            for _rec in _recs:
                h+='<div style="font-size:13px;color:#0078D4;padding:2px 0">&#8594; '+str(_rec)+'</div>'
            h+='</div>'
        if not _all_sorted and not _recs:
            h+='<div style="color:#605E5C;font-size:13px">No checks available</div>'
        h+='</div></details>'
        if len(_dets) > 8:
            _all_detail_dfs.append((_r["name"], pd.DataFrame([{"Status":d["status"],"Category":d.get("category",""),"Item":d["item"],"Info":d.get("info","")} for d in _dets])))

    h+='</div></details>'

    # ── Post-process: resolve obj:GUID@tenant to AD group names across ALL tools ──
    # _group_name_cache was populated by Security X-Ray via Graph API
    try:
        import re as _re_pretty
        _OBJ_PATTERN = _re_pretty.compile(r'obj:([0-9a-f-]{30,})@[0-9a-f-]+')
        _cache_avail = '_group_name_cache' in globals() and len(globals().get('_group_name_cache', {})) > 0

        def _prettify_objects(_text):
            """Replace obj:GUID@tenant with cached friendly name when available."""
            if not _text or 'obj:' not in str(_text):
                return _text
            def _sub(_m):
                _gid = _m.group(1)
                _name = globals().get('_group_name_cache', {}).get(_gid)
                return _name if _name else _m.group(0)
            return _OBJ_PATTERN.sub(_sub, str(_text))

        # Count ALL obj:GUID occurrences (even if cache not available) for debugging
        _obj_total_count = 0
        for _tk, _tr in results.items():
            if _tk.startswith('_'):
                continue
            for _det in _tr.get('details', []):
                for _field in ('item', 'info'):
                    if 'obj:' in str(_det.get(_field, '')):
                        _obj_total_count += 1
        _cache_size = len(globals().get('_group_name_cache', {}))
        if _obj_total_count > 0:
            print(f"  Found {_obj_total_count} obj:GUID reference(s); cache has {_cache_size} resolved name(s)")

        if _cache_avail:
            _prettify_count = 0
            for _tk, _tr in results.items():
                if _tk.startswith('_'):
                    continue
                for _det in _tr.get('details', []):
                    for _field in ('item', 'info'):
                        _orig = _det.get(_field, '')
                        _new = _prettify_objects(_orig)
                        if _new != _orig:
                            _det[_field] = _new
                            _prettify_count += 1
            if _prettify_count > 0:
                print(f"  Resolved {_prettify_count} obj:GUID reference(s) to AD group names")
    except Exception as _pe:
        pass  # best-effort prettification, don't break scan

    # Fix plan
    fix_plan=generate_fix_plan(results, meta)
    results["_fixes"]=fix_plan
    _cc={"Description":"#0078D4","Format":"#2D9B83","Visibility":"#FFB900","Summarize By":"#117865"}
    if fix_plan:
        cats=Counter(f["category"] for f in fix_plan)
        h+='<details style="margin-top:16px"><summary style="font-size:18px;font-weight:700;border-bottom:2px solid #EDEBE9;padding-bottom:8px;cursor:pointer">Fix Plan &mdash; '+str(len(fix_plan))+' fixes</summary><div style="margin-top:8px">'
        h+='<div style="display:flex;gap:8px;margin:8px 0;flex-wrap:wrap">'
        for cat,cnt in cats.most_common():
            c2=_cc.get(cat,"#605E5C")
            h+='<span style="background:'+c2+'18;color:'+c2+';border:1px solid '+c2+'40;padding:4px 12px;border-radius:16px;font-size:13px;font-weight:600">'+cat+' ('+str(cnt)+')</span>'
        h+='</div>'
        from itertools import groupby as _gb
        # Group by category and sort by size (largest first) so all categories are visible at top
        _cat_groups = {}
        for _f in fix_plan:
            _cat_groups.setdefault(_f["category"], []).append(_f)
        # Sort categories by item count (descending) so Synonym/biggest appear first
        for cat, cl in sorted(_cat_groups.items(), key=lambda kv: -len(kv[1])):
            c2=_cc.get(cat,"#605E5C")
            h+='<details style="margin:8px 0;border:1px solid '+c2+'44;border-radius:8px;overflow:hidden">'
            h+='<summary style="padding:10px 14px;background:'+c2+'12;font-weight:600;font-size:13px;color:'+c2+';cursor:pointer;list-style:none;user-select:none">'
            h+='<span style="color:'+c2+';font-weight:700;margin-right:8px">&#9656;</span>'
            h+=cat+' <span style="opacity:0.7;font-weight:400">('+str(len(cl))+' fixes)</span>'
            h+='</summary><div style="padding:0">'
            h+='<table style="width:100%;border-collapse:collapse;font-size:13px">'
            h+='<tr style="background:#FAF9F8"><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">#</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Item</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Current</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Proposed</th><th style="text-align:left;padding:8px 10px;border-bottom:2px solid #C8C6C4">Source</th></tr>'
            for _i,_f in enumerate(cl):
                _bg="#fff" if _i%2==0 else "#FAF9F8"
                h+='<tr style="background:'+_bg+'"><td style="padding:6px 10px;border-bottom:1px solid #FAF9F8">'+str(_i+1)+'</td>'
                h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;font-weight:500">'+str(_f["item"])+'</td>'
                h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;color:#D13438"><em>'+str(_f.get("current",""))+'</em></td>'
                h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;color:#107C10;font-weight:500">'+str(_f.get("proposed",""))+'</td>'
                h+='<td style="padding:6px 10px;border-bottom:1px solid #FAF9F8;font-size:11px;color:#605E5C">'+str(_f.get("source",""))+'</td></tr>'
            h+='</table></div></details>'
        h+='<div style="background:#FFF8E0;border-radius:6px;padding:10px 14px;margin-top:8px;font-size:13px;color:#7A6400">All fixes are LOW risk (metadata only). Edit <code>SELECTED_FIXES</code> in the next cell to choose which to apply.</div>'
        h+='</div></details>'
    else:
        h+='<div style="background:#E6F2E6;border-radius:8px;padding:16px;text-align:center;color:#107C10;font-weight:600;margin:16px 0">&check; No fixable issues. Model looks great!</div>'
    h+='</div>'
    # ── Export HTML + PDF ──
    import datetime as _dt
    _ts = _dt.datetime.now().strftime("%Y%m%d_%H%M")
    _css = """body{font-family:'Segoe UI',sans-serif;background:#FAF9F8;margin:0;padding:20px}
.container{max-width:960px;margin:0 auto;background:#fff;border-radius:12px;padding:24px;box-shadow:0 1px 3px rgba(0,0,0,.1)}
details{margin:10px 0;border:1px solid #EDEBE9;border-radius:8px;overflow:hidden}
summary{padding:12px 16px;background:#FAF9F8;cursor:pointer;font-weight:600}
summary:hover{background:#F3F2F1}
table{width:100%;border-collapse:collapse;font-size:13px}
th{text-align:left;padding:8px 10px;background:#F3F2F1;border-bottom:2px solid #C8C6C4}
td{padding:6px 10px;border-bottom:1px solid #FAF9F8}
.ts{text-align:center;color:#A19F9D;font-size:12px;margin-top:16px}
@media print{details{break-inside:avoid}details[open] summary~*{display:block!important}}
@page{size:A4 landscape;margin:15mm}"""
    _stamp = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
    _full_html = "<!DOCTYPE html><html><head><meta charset=\"utf-8\"><title>Model Health & Security Report - "+DATASET+"</title><style>"+_css+"</style></head><body><div class=\"container\">"+h+"<div class=\"ts\">Generated: "+_stamp+" | Model Health & Security Suite</div></div></body></html>"

    # Save HTML
    _html_path = None
    for _try_path in ["/lakehouse/default/Files/health_report_"+DATASET+"_"+_ts+".html",
                      "health_report_"+DATASET+"_"+_ts+".html"]:
        try:
            with open(_try_path, "w", encoding="utf-8") as _f:
                _f.write(_full_html)
            _html_path = _try_path
            print("\n  HTML report saved: " + _try_path)
            break
        except Exception:
            continue
    if not _html_path:
        print("\n  Could not save HTML report")

    # PDF export
    _pdf_path = (_html_path or "report").replace(".html", ".pdf")
    _pdf_ok = False
    try:
        from weasyprint import HTML as _WH
        _WH(string=_full_html).write_pdf(_pdf_path)
        _pdf_ok = True
        print("  PDF report saved: " + _pdf_path)
    except ImportError:
        pass
    except Exception as _e:
        print("  PDF failed: " + str(_e)[:100])
    if not _pdf_ok:
        print("  PDF tip: Open HTML in browser, Ctrl+P to save as PDF")
        print("  Or: pip install weasyprint (requires system deps)")

    # Route output based on mode
    # Route output based on mode — delegated to _render_widgets so Cell 3
    # can re-render the same tabs after a browser refresh without re-scanning.
    _render_widgets(results, h, _all_detail_dfs, _html_path)
    global _LAST_SCAN
    _LAST_SCAN = {"results": results, "h": h, "detail_dfs": _all_detail_dfs, "html_path": _html_path}

    # Show full sortable DataFrames for tools with many items — only in non-widget mode.
    # In widget mode, these are already routed to the Detailed Findings tab via the HTML report
    # and to the Security tab via the Security X-Ray renderer.
    if not _WIDGET_MODE:
        for _df_name, _df_data in _all_detail_dfs:
            _sep = "\u2500" * 50
            print("\n" + _sep)
            print(f"  {_df_name} \u2014 Full Details ({len(_df_data)} items)")
            print(_sep)
            display(_df_data)

# run_scan() is called from Cell 3 (widget console) via the Run Scan button.
print("Engine loaded: 13 tools + Security X-Ray ready. Run next cell for interactive console.")

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 10, Finished, Available, Finished, False)

  AI: Azure OpenAI (Fabric Copilot) ready
Engine loaded: 13 tools + Security X-Ray ready. Run next cell for interactive console.


In [7]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import sempy.fabric as fabric
import io, contextlib  # for capturing scan log to a separate cell

# ── State ──
_state = {"workspace": None, "dataset": None, "compare": None}

# ── Load workspaces ──
def _load_workspaces():
    try:
        ws = fabric.list_workspaces()
        col = next((c for c in ["Name","name","Workspace Name"] if c in ws.columns), ws.columns[0])
        return sorted(ws[col].tolist())
    except:
        return ["(run in Fabric)"]

def _load_models(ws_name):
    try:
        ds = fabric.list_datasets(workspace=ws_name)
        col = next((c for c in ["Dataset Name","Name","name"] if c in ds.columns), ds.columns[0])
        return sorted(ds[col].tolist())
    except:
        return []


# ── Widget lock: disable interactive widgets during callbacks to prevent
# Fabric auto-save conflicts when widget state changes mid-save. ──
_ALL_INTERACTIVE = []  # populated after widget creation

def _lock_widgets():
    """Disable all interactive widgets to prevent Fabric auto-save conflicts."""
    for w in _ALL_INTERACTIVE:
        try:
            w.disabled = True
        except Exception:
            pass

def _unlock_widgets():
    """Re-enable all interactive widgets."""
    for w in _ALL_INTERACTIVE:
        try:
            w.disabled = False
        except Exception:
            pass

# ── Header ──
_header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#1B3A4B,#117865);padding:20px 24px;border-radius:12px 12px 0 0;margin-bottom:0">
  <div style="font-size:22px;font-weight:700;color:#fff;letter-spacing:0.5px">Model Health & Security Suite</div>
  <div style="font-size:13px;color:#7DBDAB;margin-top:4px">13-tool diagnostic &amp; remediation toolkit powered by Semantic Link</div>
</div>
""")

# Refresh hint — explains how to get the console back after a browser refresh
_refresh_hint = widgets.HTML(
    "<div style='background:#F0F6F5;border:1px solid #B3D9D2;border-radius:6px;"
    "padding:8px 12px;font-size:11px;color:#005A9E;margin:8px 2px 0 2px'>"
    "<b>Tip:</b> If the console disappears after a browser refresh, just re-run this cell. "
    "As long as the kernel is alive, your last scan results will auto-restore into the tabs."
    "</div>"
)

# ── Workspace / Model selectors ──
_ws_list = _load_workspaces() # ['Sample']# 
# Dropdowns — explicit min_width prevents column collapse when options change
ws_dd = widgets.Dropdown(
    options=_ws_list, description="",
    layout=widgets.Layout(width="100%", min_width="200px")
)
model_dd = widgets.Dropdown(
    options=[], description="",
    layout=widgets.Layout(width="100%", min_width="200px")
)
compare_dd = widgets.Dropdown(
    options=["(none)"], description="",
    layout=widgets.Layout(width="100%", min_width="200px")
)
compare_ws_dd = widgets.Dropdown(
    options=["(same workspace)"], description="",
    layout=widgets.Layout(width="100%", min_width="200px")
)

def _on_ws_change(change):
    _lock_widgets()
    try:
        models = _load_models(change["new"])
        model_dd.options = models
        compare_dd.options = ["(none)"] + models
        compare_ws_dd.options = ["(same workspace)"] + _ws_list
    finally:
        _unlock_widgets()

def _on_compare_ws_change(change):
    _lock_widgets()
    try:
        cws = change["new"]
        if cws == "(same workspace)":
            compare_dd.options = ["(none)"] + list(model_dd.options)
        else:
            compare_dd.options = ["(none)"] + _load_models(cws)
    finally:
        _unlock_widgets()

compare_ws_dd.observe(_on_compare_ws_change, names="value")

ws_dd.observe(_on_ws_change, names="value")
if _ws_list:
    _on_ws_change({"new": _ws_list[0]})

def _make_field(label, widget, hint=""):
    # Pad every field with a hint slot so all three columns have identical height,
    # preventing flex row jitter when the user opens a dropdown.
    items = [widgets.HTML(f"<div style='font-size:12px;font-weight:600;color:#323130;margin-bottom:4px'>{label}</div>")]
    items.append(widget)
    _hint_html = f"<div style='font-size:11px;color:#A19F9D;margin-top:2px;min-height:14px'>{hint or '&nbsp;'}</div>"
    items.append(widgets.HTML(_hint_html))
    return widgets.VBox(items, layout=widgets.Layout(
        flex="1 1 33%", margin="0 6px", min_height="72px",
        align_items="stretch"
    ))

_conn_row = widgets.HBox([
    _make_field("Workspace", ws_dd),
    _make_field("Semantic Model", model_dd),
], layout=widgets.Layout(margin="0 0 6px 0", align_items="flex-start"))

_compare_row = widgets.HBox([
    _make_field("Compare Workspace", compare_ws_dd, "For Model Diff"),
    _make_field("Compare Model", compare_dd, "Select model to diff against"),
], layout=widgets.Layout(margin="0 0 12px 0", align_items="flex-start"))

# ── Tool checkboxes ──
_unique_tools = [
    ("copilot_readiness", "Copilot Readiness", True),
    ("lineage", "Lineage", True),
    ("test_framework", "Test Framework", True),
    ("model_diff", "Model Diff", True),
    ("report_visuals", "Report Visuals", True),
    ("security_audit", "Security Audit", True),
    ("security_xray", "Security X-Ray", True),
    ("data_quality", "Data Quality", True),
    ("dax_dependencies", "DAX Dependencies", True),
]
_sempy_tools_list = [
    ("bpa", "Best Practices", False),
    ("vertipaq", "Vertipaq Performance", False),
    ("direct_lake", "Direct Lake", False),
    ("capacity", "Capacity", False),
]

_tool_cbs = {}
for key, label, default in _unique_tools + _sempy_tools_list:
    _tool_cbs[key] = widgets.Checkbox(value=default, description=label, indent=False,
                                       layout=widgets.Layout(width="auto"))

_unique_box = widgets.VBox(
    [widgets.HTML("<div style='font-size:12px;font-weight:700;color:#1B3A4B;margin-bottom:4px'>Unique Tools</div>")] +
    [_tool_cbs[k] for k, _, _ in _unique_tools],
    layout=widgets.Layout(flex="1 1 50%", padding="8px", border="1px solid #EDEBE9", border_radius="8px", margin="0 4px")
)
_sempy_box = widgets.VBox(
    [widgets.HTML("<div style='font-size:12px;font-weight:700;color:#117865;margin-bottom:4px'>Powered by sempy_labs</div>")] +
    [_tool_cbs[k] for k, _, _ in _sempy_tools_list],
    layout=widgets.Layout(flex="1 1 50%", padding="8px", border="1px solid #D2ECE8", border_radius="8px",
                          background="#F0F6F5", margin="0 4px")
)
_tools_row = widgets.HBox([_unique_box, _sempy_box], layout=widgets.Layout(margin="0 0 4px 0"))

# ── Options ──
_ai_desc_cb = widgets.Checkbox(value=True, description="Use AI Descriptions (if available)", indent=False)
_ai_context_ta = widgets.Textarea(
    value="",
    placeholder="Example: This is a sales analytics model for a healthcare distribution company. "
                "Key metrics are revenue, profit margin, and order volume. "
                "Dimension tables include customers, cities, employees, and stock items. "
                "Use business-friendly language, avoid technical jargon.",
    layout=widgets.Layout(width="100%", height="80px"),
)
_ai_context_label = widgets.HTML(
    '<div style="font-size:11px;color:#605E5C;margin:4px 0 2px 0">'
    '<b>AI Context</b> — describe your business domain so AI generates relevant descriptions & synonyms</div>')
_ai_context_box = widgets.VBox([_ai_context_label, _ai_context_ta],
    layout=widgets.Layout(display="flex", margin="4px 0 0 0"))

def _on_ai_toggle(change):
    _ai_context_box.layout.display = "flex" if change["new"] else "none"
_ai_desc_cb.observe(_on_ai_toggle, names="value")
_extract_members_cb = widgets.Checkbox(value=False, description="Extract role members (deep scan)", indent=False)
_spn_cb = widgets.Checkbox(value=False, description="Use Service Principal", indent=False)
_verbose_cb = widgets.Checkbox(value=False, description="Verbose Logs", indent=False,
    layout=widgets.Layout(width="140px"),
    style={"description_width": "initial"})
_mask_pii_cb = widgets.Checkbox(value=False, description="Mask PII (Demo Mode)", indent=False,
    layout=widgets.Layout(width="180px"),
    style={"description_width": "initial"})
_spn_tenant = widgets.Text(placeholder="Tenant ID", layout=widgets.Layout(width="200px"))
_spn_client = widgets.Text(placeholder="Client ID", layout=widgets.Layout(width="200px"))
_spn_secret = widgets.Password(placeholder="Client Secret", layout=widgets.Layout(width="200px"))
_spn_fields = widgets.HBox([_spn_tenant, _spn_client, _spn_secret],
                            layout=widgets.Layout(display="none", margin="4px 0"))

def _on_spn_toggle(change):
    _spn_fields.layout.display = "flex" if change["new"] else "none"
_spn_cb.observe(_on_spn_toggle, names="value")

_options_box = widgets.VBox([
    widgets.HTML("<div style='font-size:12px;font-weight:700;color:#323130;margin-bottom:4px'>Options</div>"),
    widgets.HBox([_ai_desc_cb, _extract_members_cb, _verbose_cb, _mask_pii_cb]),
    widgets.HBox([_spn_cb, _spn_fields]),
    _ai_context_box,
], layout=widgets.Layout(padding="8px", border="1px solid #EDEBE9", border_radius="8px", margin="0 0 12px 0"))

# ── Action buttons ──
_select_all_btn = widgets.Button(description="Select All", button_style="",
                                  layout=widgets.Layout(width="90px", height="28px"),
                                  style={"font_size": "11px"})
_deselect_all_btn = widgets.Button(description="Deselect All", button_style="",
                                    layout=widgets.Layout(width="90px", height="28px"),
                                    style={"font_size": "11px"})
_restore_btn = widgets.Button(description="Restore Last", button_style="info", icon="history",
                               layout=widgets.Layout(width="140px", height="38px"))
_refresh_schema_btn = widgets.Button(description="Schema Refresh", button_style="info",
                                      icon="refresh", layout=widgets.Layout(width="150px", height="38px"))
_refresh_data_btn = widgets.Button(description="Data Refresh", button_style="info",
                                    icon="database", layout=widgets.Layout(width="140px", height="38px"))
_run_btn = widgets.Button(description="  Run Scan", button_style="success", icon="play",
                           layout=widgets.Layout(width="160px", height="38px"),
                           style={"font_weight": "bold"})

def _on_select_all(_):
    for cb in _tool_cbs.values(): cb.value = True
def _on_deselect_all(_):
    for cb in _tool_cbs.values(): cb.value = False
_select_all_btn.on_click(_on_select_all)
_deselect_all_btn.on_click(_on_deselect_all)

_sel_row = widgets.HBox([
    _select_all_btn, _deselect_all_btn
], layout=widgets.Layout(margin="0 0 4px 0"))

_action_bar = widgets.HBox([
                             _restore_btn, _refresh_schema_btn, _refresh_data_btn,
                             widgets.HTML("<div style='flex:1'></div>"),
                             _run_btn],
                            layout=widgets.Layout(margin="0 0 12px 0"))

# ── Output areas ──
# Progress area stays small — just tool-by-tool progress during the scan.
_progress_out = widgets.Output(layout=widgets.Layout(height="180px", max_height="180px", overflow_y="auto",
                                                       border="1px solid #EDEBE9", border_radius="8px",
                                                       padding="8px", margin="0 0 8px 0",
                                                       background="#FAF9F8"))
_report_out = widgets.Output(layout=widgets.Layout(margin="8px 0 0 0"))
_report_html = widgets.HTML(
    value='<div style="padding:20px;color:#605E5C;font-family:Segoe UI,sans-serif;text-align:center;border:1px dashed #EDEBE9;border-radius:8px;margin:8px 0;background:#FAF9F8"><span style="font-size:24px">📊</span><br/><span style="font-size:14px;font-weight:600;color:#323130">Click Run Scan, then run the next cell (View Report).</span><br/><span style="font-size:12px">Results will appear in an interactive report below.</span></div>',
    layout=widgets.Layout(margin="8px 0 0 0", width="100%", min_height="120px"),
)

# Each tab is a bounded scrollable region so it never pushes the console around.
_tab_layout = widgets.Layout(min_height="420px", max_height="720px", overflow_y="auto",
                              padding="12px", border="1px solid #EDEBE9", border_radius="0 0 8px 8px")
_tab_scores = widgets.VBox(layout=_tab_layout)
_tab_findings = widgets.VBox(layout=_tab_layout)
_tab_fixplan = widgets.VBox(layout=_tab_layout)
_tab_security_audit = widgets.VBox(layout=_tab_layout)
_tab_security_xray = widgets.VBox(layout=_tab_layout)
_tab_export = widgets.VBox(layout=_tab_layout)

# Alias for backwards compat with any existing code paths that still reference _tab_security
_tab_security = _tab_security_audit

_results_tabs = widgets.Tab(
    children=[_tab_scores, _tab_findings, _tab_fixplan,
              _tab_security_audit, _tab_security_xray, _tab_export],
    layout=widgets.Layout(margin="8px 0"))
_results_tabs.set_title(0, "Score Cards")
_results_tabs.set_title(1, "Detailed Findings")
_results_tabs.set_title(2, "Fix Plan")
_results_tabs.set_title(3, "Security Audit")
_results_tabs.set_title(4, "Security X-Ray")
_results_tabs.set_title(5, "Export")

# Tiny caption under the tab bar — tells users how to navigate
_tabs_caption = widgets.HTML(
    "<div style='font-size:11px;color:#605E5C;margin:4px 2px 10px 2px'>"
    "Click a tab above to switch sections. Each tab scrolls independently, so the console stays in place."
    "</div>"
)

# Wrap the tab panel + caption in a container VBox so we can reliably hide it
# as a single unit. Hiding the Tab widget directly doesn't work in Fabric —
# the tab bar still renders. Hiding the outer VBox collapses everything.
_results_container = widgets.VBox(
    [_results_tabs, _tabs_caption],
    layout=widgets.Layout(display="none", margin="0", padding="0")
)

def _show_results_panel():
    # Explicit "flex" is more reliable than "" for re-rendering a hidden VBox
    _results_container.layout.display = "flex"

def _hide_results_panel():
    _results_container.layout.display = "none"

# ── Run handler ──

def _render_dashboard_to(out_widget):
    """Render the cached scan HTML into an Output widget as a VBox of chunks.
    Matches last year's winner pattern — works from button callbacks."""
    _ls = globals().get('_LAST_SCAN', None)
    if not _ls or not _ls.get('h'):
        return False
    _h = _ls['h']
    _CHUNK_LIMIT = 50000
    _marker = '</details>'
    _parts = _h.split(_marker)
    _sections = []
    for _i, _p in enumerate(_parts):
        if _i < len(_parts) - 1:
            _sections.append(_p + _marker)
        else:
            _sections.append(_p)
    _sections = [_s for _s in _sections if _s.strip()]

    _chunks, _current = [], ""
    for _s in _sections:
        if len(_current) + len(_s) < _CHUNK_LIMIT:
            _current += _s
        else:
            if _current:
                _chunks.append(_current)
            _current = _s
    if _current:
        _chunks.append(_current)

    _html_widgets = [widgets.HTML(value=_c) for _c in _chunks]
    _dashboard = widgets.VBox(
        _html_widgets,
        layout=widgets.Layout(width="100%", padding="8px")
    )
    out_widget.clear_output()
    with out_widget:
        display(_dashboard)
    return True

def _on_run(_):
    global DATASET, WORKSPACE, COMPARE_DATASET, COMPARE_WORKSPACE, USE_AI_DESCRIPTIONS, TOOLS
    global SPN_TENANT_ID, SPN_CLIENT_ID, SPN_CLIENT_SECRET, USE_SPN, VERBOSE_LOG, MASK_PII, _WIDGET_MODE, _EXTRACT_MEMBERS

    WORKSPACE = ws_dd.value
    DATASET = model_dd.value
    COMPARE_DATASET = compare_dd.value if compare_dd.value != "(none)" else None
    _cws = compare_ws_dd.value
    COMPARE_WORKSPACE = None if _cws == "(same workspace)" else _cws
    USE_AI_DESCRIPTIONS = _ai_desc_cb.value
    global AI_USER_CONTEXT
    AI_USER_CONTEXT = _ai_context_ta.value.strip() if _ai_desc_cb.value else ""
    _EXTRACT_MEMBERS = _extract_members_cb.value
    VERBOSE_LOG = _verbose_cb.value
    MASK_PII = _mask_pii_cb.value
    try:
        _reset_pii_maps()
    except Exception:
        pass
    TOOLS = {k: cb.value for k, cb in _tool_cbs.items()}
    _WIDGET_MODE = True

    if _spn_cb.value:
        SPN_TENANT_ID = _spn_tenant.value
        SPN_CLIENT_ID = _spn_client.value
        SPN_CLIENT_SECRET = _spn_secret.value
        USE_SPN = True
    else:
        SPN_TENANT_ID = SPN_CLIENT_ID = SPN_CLIENT_SECRET = None
        USE_SPN = False

    if not DATASET:
        with _progress_out:
            print("Please select a semantic model.")
        return

    _run_btn.disabled = True
    _run_btn.description = "  Scanning..."
    _run_btn.icon = "spinner"

    # Clear outputs
    _progress_out.clear_output()
    _report_out.clear_output()
    _progress_out.layout.height = "220px"
    _progress_out.layout.max_height = "220px"
    for tab in [_tab_scores, _tab_findings, _tab_fixplan,
                _tab_security_audit, _tab_security_xray, _tab_export]:
        tab.children = []

    # Run scan with stdout captured into _LAST_SCAN_LOG (so form stays visible)
    # The full scan log is viewable via the next cell ("Scan Log")
    global _SKIP_WIDGET_RENDER, _LAST_SCAN_LOG
    _SKIP_WIDGET_RENDER = True
    _scan_ok = False
    _log_buf = io.StringIO()

    # Collapse progress output area — only minimal status shown
    _progress_out.layout.height = "auto"
    _progress_out.layout.max_height = "90px"
    _progress_out.layout.overflow_y = "auto"

    with _progress_out:
        print("  Scanning... (log is captured — run the next cell to view details)")

    try:
        with contextlib.redirect_stdout(_log_buf):
            run_scan()
        _scan_ok = True
    except Exception as e:
        import traceback as _tb
        _log_buf.write(f"\n  ERROR: {e}\n")
        _log_buf.write(_tb.format_exc()[:2000])
        with _progress_out:
            print(f"  ERROR: {e}")
    finally:
        _SKIP_WIDGET_RENDER = False
        _run_btn.disabled = False
        _run_btn.description = "  Run Scan"
        _run_btn.icon = "play"
        _LAST_SCAN_LOG = _log_buf.getvalue()

    # Show minimal completion status — form stays visible above
    if _scan_ok:
        _ls = globals().get('_LAST_SCAN', None)
        if _ls and _ls.get('h'):
            _progress_out.clear_output()
            with _progress_out:
                _n = len(_ls.get('results', {})) if _ls else 0
                _loglines = _LAST_SCAN_LOG.count("\n")
                print(f"  ✓ Scan complete: {_n} tool(s) ran ({_loglines} log lines).")
                print("  → Run next cell (Scan Log) to see full scan output")
                print("  → Run cell after that (View Report) for the HTML dashboard")
                _hp = _ls.get('html_path')
                if _hp:
                    print(f"  HTML saved: {_hp}")


_run_btn.on_click(_on_run)

def _on_restore(_):
    """Re-render from the last cached scan state."""
    _progress_out.clear_output()
    _ls = globals().get('_LAST_SCAN', None)
    if not _ls or not _ls.get('h'):
        with _progress_out:
            print("No cached scan found. Click Run Scan first.")
        return
    _progress_out.layout.height = "auto"
    _progress_out.layout.max_height = "60px"
    with _progress_out:
        _model = _ls.get('results', {}).get('_overall', {}).get('model', '?')
        print(f"Restored: {_model}. Run Cell 5 to view the dashboard.")

_restore_btn.on_click(_on_restore)

def _on_refresh_schema(_):
    """Schema-only refresh — syncs metadata from lakehouse without reloading data."""
    global DATASET, WORKSPACE
    WORKSPACE = ws_dd.value
    DATASET = model_dd.value
    if not DATASET:
        with _progress_out:
            print("Please select a semantic model first.")
        return
    _refresh_schema_btn.disabled = True
    _refresh_schema_btn.description = "Refreshing..."
    _progress_out.clear_output()
    try:
        with _progress_out:
            _silencer = globals().get('_silence_noisy_warnings', None)
            if USE_SPN and SPN_TENANT_ID and SPN_CLIENT_ID and SPN_CLIENT_SECRET:
                from sempy.fabric import set_service_principal
                with set_service_principal(tenant_id=SPN_TENANT_ID, client_id=SPN_CLIENT_ID, client_secret=SPN_CLIENT_SECRET):
                    if _silencer:
                        with _silencer(): _refresh_schema(DATASET, WORKSPACE)
                    else:
                        _refresh_schema(DATASET, WORKSPACE)
            else:
                if _silencer:
                    with _silencer(): _refresh_schema(DATASET, WORKSPACE)
                else:
                    _refresh_schema(DATASET, WORKSPACE)
            print("\nSchema refresh done. Click Run Scan to see updated results.")
    except Exception as e:
        with _progress_out:
            print(f"Error: {e}")
    finally:
        _refresh_schema_btn.disabled = False
        _refresh_schema_btn.description = "Refresh Schema"

def _on_refresh_data(_):
    """Full data refresh — reloads all data from sources."""
    global DATASET, WORKSPACE
    WORKSPACE = ws_dd.value
    DATASET = model_dd.value
    if not DATASET:
        with _progress_out:
            print("Please select a semantic model first.")
        return
    _refresh_data_btn.disabled = True
    _refresh_data_btn.description = "Refreshing..."
    _progress_out.clear_output()
    try:
        with _progress_out:
            _silencer = globals().get('_silence_noisy_warnings', None)
            if USE_SPN and SPN_TENANT_ID and SPN_CLIENT_ID and SPN_CLIENT_SECRET:
                from sempy.fabric import set_service_principal
                with set_service_principal(tenant_id=SPN_TENANT_ID, client_id=SPN_CLIENT_ID, client_secret=SPN_CLIENT_SECRET):
                    if _silencer:
                        with _silencer(): _refresh_data(DATASET, WORKSPACE)
                    else:
                        _refresh_data(DATASET, WORKSPACE)
            else:
                if _silencer:
                    with _silencer(): _refresh_data(DATASET, WORKSPACE)
                else:
                    _refresh_data(DATASET, WORKSPACE)
            print("\nData refresh done. Click Run Scan to see updated results.")
    except Exception as e:
        with _progress_out:
            print(f"Error: {e}")
    finally:
        _refresh_data_btn.disabled = False
        _refresh_data_btn.description = "Refresh Data"

_refresh_schema_btn.on_click(_on_refresh_schema)
_refresh_data_btn.on_click(_on_refresh_data)

# ── Assemble UI ──
_app = widgets.VBox([
    _header,
    _refresh_hint,
    widgets.VBox([
        _conn_row,
        _compare_row,
        widgets.HTML("<hr style='border:none;border-top:1px solid #EDEBE9;margin:4px 0'/>"),
        _tools_row,
        _sel_row,
        _options_box,
        _action_bar,
        _progress_out,
        _report_html,
        _results_container,
        _report_out,
    ], layout=widgets.Layout(border="1px solid #EDEBE9", border_radius="0 0 12px 12px",
                              padding="16px", background="#FFFFFF")),
])

# Register interactive widgets for lock/unlock during callbacks
_ALL_INTERACTIVE.extend([ws_dd, model_dd, compare_ws_dd, compare_dd,
    _run_btn, _restore_btn, _refresh_schema_btn, _refresh_data_btn,
    _select_all_btn, _deselect_all_btn, _ai_desc_cb, _ai_context_ta, _extract_members_cb, _mask_pii_cb,
    _verbose_cb, _spn_cb])
_ALL_INTERACTIVE.extend(list(_tool_cbs.values()))

display(_app)

# ── Cache-hint: if a previous scan exists, tell the user to click
# "Restore Last Scan" rather than auto-restoring. Automatic restore during
# Cell 3 execution causes display() calls inside `with _tab_X:` blocks to
# leak out of the Output widgets before the widget DOM is attached — so we
# gate it behind an explicit button click, which fires after the widget is
# fully rendered and attached.
try:
    _ls = globals().get('_LAST_SCAN', None)
    if _ls and _ls.get('results'):
        with _progress_out:
            _model = _ls['results'].get('_overall', {}).get('model', '?')
            print(f"Previous scan available for '{_model}'.")
            print("Click 'Restore Last Scan' to reload the tabs from cache, or 'Run Scan' for fresh data.")
except Exception:
    pass

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 29, Finished, Available, Finished, False)

Previous scan available for 'msft_sample_semanticmodel'.
Click 'Restore Last Scan' to reload the tabs from cache, or 'Run Scan' for fresh data.


In [4]:
# ══════════════════════════════════════════════════════════════
#  SCAN LOG — full verbose output from the most recent Run Scan
#  (captured during Cell 3 so the config form stays visible)
# ══════════════════════════════════════════════════════════════
try:
    if _LAST_SCAN_LOG and _LAST_SCAN_LOG.strip():
        print(_LAST_SCAN_LOG)
    else:
        print("No scan log yet. Click Run Scan in Cell 3 first.")
except NameError:
    print("No scan log yet. Run Cell 2, then Cell 3, click Run Scan.")


StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 25, Finished, Available, Finished, False)

Model Health & Security Suite
Model: msft_sample_semanticmodel | Workspace: demo-workspace-A

Connected: 6 tables, 93 columns, 16 measures

  [1/9] Running Copilot Readiness...  43.9% (F) [4.6s]
  [2/9] Running Lineage... 100.0% (A+) [0.0s]
  [3/9] Running Test Framework... 100.0% (A+) [2.4s]
  [4/9] Running Model Diff... 100.0% (A+) [0.0s]
  [5/9] Running Report Visuals... 100.0% (A+) [0.2s]
  [6/9] Running Security Audit...  67.0% (C+) [1.3s]
  [7/9] Running Security X-Ray...         [X-Ray] Step 1/6: Reading RLS + OLS via TOM...
        [X-Ray] Step 1/6: Done — 3 RLS, 2 OLS [1.1s]
        [X-Ray] Step 2/6: Querying role members via DAX INFO...
        [X-Ray] Step 2/6: Done — 4 member(s) [0.2s]
        [X-Ray] Step 3/6: Acquiring Graph token + expanding AAD groups...
        [X-Ray] Graph token acquired via SPN
        [X-Ray] Step 3/6: Expanding 4 group(s)...
        [X-Ray] Step 3/6: Done — 4 group(s) expanded [1.8s]
        [X-Ray] Step 4/6: Fetching workspace role assignments...

In [5]:
# ══════════════════════════════════════════════════════════════
#  VIEW REPORT (optional) — The report renders automatically in the
#  console (Cell 3) after Run Scan. Use this cell to re-render
#  from cache without running the scan again.
# ══════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display

try:
    _ls = _LAST_SCAN
except NameError:
    _ls = None

if _ls is None or not _ls.get("h"):
    display(widgets.HTML(
        '<div style="padding:20px;color:#605E5C;font-family:Segoe UI,sans-serif;'
        'text-align:center;border:1px dashed #EDEBE9;border-radius:8px;background:#FAF9F8">'
        '<span style="font-size:24px">📊</span><br/>'
        '<span style="font-size:14px;font-weight:600">No scan results found.</span><br/>'
        '<span style="font-size:12px">Click Run Scan in the console above first, then re-run this cell.</span>'
        '</div>'
    ))
else:
    _h = _ls["h"]
    _model = _ls.get("results", {}).get("_overall", {}).get("model", "?")
    _n_tools = len([k for k in _ls.get("results", {}).keys() if not k.startswith("_")])
    _hp = _ls.get("html_path")
    print(f"Rendering report for: {_model}")
    print(f"Tools: {_n_tools} | HTML size: {len(_h)//1024} KB")
    if _hp:
        print(f"Saved: {_hp}")
    print()

    # Split HTML at </details> boundaries into chunks ≤ 50KB each.
    # Each chunk becomes its own widgets.HTML. Fabric's widget renderer
    # handles individual widgets fine; the VBox container stacks them.
    _CHUNK_LIMIT = 50000
    _marker = "</details>"
    _parts = _h.split(_marker)
    _sections = []
    for _i, _p in enumerate(_parts):
        if _i < len(_parts) - 1:
            _sections.append(_p + _marker)
        else:
            _sections.append(_p)
    _sections = [_s for _s in _sections if _s.strip()]

    _chunks = []
    _current = ""
    for _s in _sections:
        if len(_current) + len(_s) < _CHUNK_LIMIT:
            _current += _s
        else:
            if _current:
                _chunks.append(_current)
            _current = _s
    if _current:
        _chunks.append(_current)

    print(f"Rendering {len(_chunks)} HTML widget(s) in VBox:")
    for _i, _c in enumerate(_chunks):
        print(f"  Chunk {_i+1}: {len(_c)//1024}KB")

    # Wrap each chunk in widgets.HTML, then display them in a VBox.
    # This is the pattern from last year's Fabric Gallery winner.
    _html_widgets = [widgets.HTML(value=_c) for _c in _chunks]
    _dashboard = widgets.VBox(
        _html_widgets,
        layout=widgets.Layout(width="100%", padding="8px")
    )
    display(_dashboard)


StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 26, Finished, Available, Finished, False)

Rendering report for: msft_sample_semanticmodel
Tools: 13 | HTML size: 196 KB
Saved: health_report_msft_sample_semanticmodel_20260419_0454.html

Rendering 5 HTML widget(s) in VBox:
  Chunk 1: 44KB
  Chunk 2: 32KB
  Chunk 3: 47KB
  Chunk 4: 38KB
  Chunk 5: 33KB


StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 27, Finished, Available, Finished, False)

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 30, Finished, Available, Finished, False)

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 31, Finished, Available, Finished, False)

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 32, Finished, Available, Finished, False)

In [6]:
# ══════════════════════════════════════════════════════════════
#  APPLY & VERIFY — Review fixes below, edit selection, then click Run
# ══════════════════════════════════════════════════════════════

# ── Preview available fixes ──
# Run this cell first with SELECTED_FIXES = "none" to see the list,
# then change to "all" or pick specific numbers and re-run.
try:
    if fix_plan:
        # Show schema context summary
        _ai_count = sum(1 for f in fix_plan if f.get("source") == "AI")
        _tmpl_count = sum(1 for f in fix_plan if f.get("source") == "Template")
        _rule_count = sum(1 for f in fix_plan if f.get("source") == "Rule")
        print(f"Available fixes ({len(fix_plan)} total):")
        print(f"  Generation method: AI={_ai_count}  Template={_tmpl_count}  Rule={_rule_count}")
        if _ai_count > 0:
            print(f"  AI prompts include: table columns, relationships, data types, DAX expressions")
        print()
        print(f"{'#':>3}  {'Source':<9} {'Category':<16} {'Item':<35} {'Current':<12} {'Proposed'}")
        print("-" * 110)
        for _i, _f in enumerate(fix_plan):
            _src = _f.get("source", "Rule")[:8]
            print(f"{_i:>3}  {_src:<9} {_f.get('category',''):<16} {_f['item'][:33]:<35} {str(_f.get('current',''))[:10]:<12} {str(_f.get('proposed',''))[:40]}")
        print("-" * 110)
        print(f"Edit SELECTED_FIXES below, then re-run this cell.")
    else:
        print("No fixes available. Run Scan first (Cell 3).")
except NameError:
    print("Run Scan first (Cell 3) to generate the fix plan.")
print()

# Which fixes to apply?
SELECTED_FIXES = "none"        # "none" (default, shows preview) | "all" | [0, 2, 5] (list of fix numbers)
EXCLUDED_FIXES = []            # [3, 7] = skip these numbers (used with "all")

# ── Apply ──
def get_approved(plan, selected, excluded):
    if selected == "none": return []
    if selected == "all": return [f for i,f in enumerate(plan) if i not in excluded]
    if isinstance(selected, list): return [plan[i] for i in selected if i<len(plan)]
    return plan

def apply_fixes(fixes, ds, ws=None):
    applied, errors = [], []
    with connect_semantic_model(ds, readonly=False, workspace=ws) as tom:
        for fix in fixes:
            # Skip system objects
            if "RowNumber-" in fix.get("name","") or "RowNumber-" in fix.get("item",""):
                continue
            try:
                # ── Role description ──
                if fix["type"] == "role_description":
                    for role in tom.model.Roles:
                        if str(role.Name) == fix.get("name", ""):
                            role.Description = fix.get("proposed", fix.get("value", ""))
                            applied.append(fix)
                            break
                    continue
                if fix["type"]=="description" and fix["obj_type"]=="table":
                    for table in tom.model.Tables:
                        if table.Name==fix["name"]: table.Description=fix.get("proposed",""); applied.append(fix); break
                elif fix["type"]=="description" and fix["obj_type"]=="column":
                    for table in tom.model.Tables:
                        if table.Name==fix.get("table"):
                            for col in table.Columns:
                                if "RowNumber-" in col.Name: continue
                                if col.Name==fix["name"]: col.Description=fix.get("proposed",""); applied.append(fix); break
                            break
                elif fix["type"]=="measure_description":
                    for table in tom.model.Tables:
                        for measure in table.Measures:
                            if measure.Name==fix["name"]: measure.Description=fix.get("proposed",""); applied.append(fix); break
                elif fix["type"]=="hide_column":
                    for table in tom.model.Tables:
                        if table.Name==fix.get("table"):
                            for col in table.Columns:
                                if "RowNumber-" in col.Name: continue
                                if col.Name==fix["name"]: col.IsHidden=True; applied.append(fix); break
                            break
                elif fix["type"]=="format_string":
                    for table in tom.model.Tables:
                        for measure in table.Measures:
                            if measure.Name==fix["name"]: measure.FormatString=fix["value"]; applied.append(fix); break
                elif fix["type"]=="summarize_by":
                    for table in tom.model.Tables:
                        if table.Name==fix.get("table"):
                            for col in table.Columns:
                                if "RowNumber-" in col.Name: continue
                                if col.Name==fix["name"]: col.SummarizeBy=0; applied.append(fix); break
                            break
                elif fix["type"]=="model_description":
                    # Write AI instructions into linguistic metadata (CustomInstructions)
                    # This is the same field the Prep for AI UI reads/writes.
                    _instr_text = str(fix.get("value", ""))
                    # Also set model description as a visible summary
                    tom.model.Description = _instr_text[:500]
                    # Write to linguistic metadata
                    try:
                        _culture_lm = None
                        for _c in tom.model.Cultures:
                            if str(_c.Name) == "en-US":
                                _culture_lm = _c
                                break
                        if _culture_lm is None:
                            try:
                                tom.add_culture("en-US")
                            except Exception:
                                pass
                            for _cc in tom.model.Cultures:
                                if str(_cc.Name) == "en-US":
                                    _culture_lm = _cc
                                    break
                        # Read existing linguistic metadata or create new
                        _lm_meta = {}
                        if _culture_lm.LinguisticMetadata and _culture_lm.LinguisticMetadata.Content:
                            try:
                                _lm_meta = json.loads(_culture_lm.LinguisticMetadata.Content)
                            except Exception:
                                pass
                        if not _lm_meta:
                            _lm_meta = {"Version": "4.1.0", "Language": "en-US", "Entities": {}, "Relationships": {}}
                        # Set CustomInstructions (this is what Prep for AI reads)
                        _lm_meta["CustomInstructions"] = _instr_text
                        _lm_json = json.dumps(_lm_meta)
                        if _culture_lm.LinguisticMetadata is None:
                            _asm = type(tom.model).Assembly
                            _lm_cls = _asm.GetType("Microsoft.AnalysisServices.Tabular.LinguisticMetadata")
                            _ct_cls = _asm.GetType("Microsoft.AnalysisServices.Tabular.ContentType")
                            _new_lm = _lm_cls()
                            _new_lm.ContentType = _ct_cls.Json
                            _new_lm.Content = _lm_json
                            _culture_lm.LinguisticMetadata = _new_lm
                        else:
                            _culture_lm.LinguisticMetadata.Content = _lm_json
                        applied.append(fix)
                    except Exception as _lme:
                        errors.append({"fix": fix, "error": f"LinguisticMetadata write failed: {str(_lme)[:100]}"})
                elif fix["type"]=="schema_reduction":
                    # Dual-path hide: isAvailableInMDX (MDX optimization) + LinguisticMetadata Entity Visibility (Copilot/Q&A visibility)
                    _applied_this = False
                    for table in tom.model.Tables:
                        if table.Name == fix.get("table"):
                            for col in table.Columns:
                                if "RowNumber-" in col.Name: continue
                                if col.Name == fix["name"]:
                                    col.IsAvailableInMDX = False
                                    _applied_this = True
                                    break
                            break
                    # Also set LinguisticMetadata Entity Visibility = Hidden for Copilot-aware schema simplification
                    if _applied_this:
                        try:
                            import json as _json_lm
                            import re as _re_sn
                            def _to_snake(_name):
                                _s = _re_sn.sub(r"(?<=[a-z0-9])([A-Z])", r"_", str(_name))
                                _s = _re_sn.sub(r"(?<=[A-Z])([A-Z][a-z])", r"_", _s)
                                return _s.lower()
                            for _cul in tom.model.Cultures:
                                if str(_cul.Name) == "en-US":
                                    _lm_meta = {}
                                    if _cul.LinguisticMetadata and _cul.LinguisticMetadata.Content:
                                        try:
                                            _lm_meta = _json_lm.loads(str(_cul.LinguisticMetadata.Content))
                                        except Exception:
                                            _lm_meta = {}
                                    if not _lm_meta:
                                        _lm_meta = {"Version":"4.1.0","Language":"en-US","Entities":{},"Relationships":{}}
                                    _ents = _lm_meta.setdefault("Entities", {})
                                    _ent_key = f"{_to_snake(fix.get('table',''))}.{_to_snake(fix['name'])}"
                                    _ents[_ent_key] = {
                                        "Definition":{"Binding":{"ConceptualEntity":fix.get("table",""),"ConceptualProperty":fix["name"]}},
                                        "State":"Generated","Terms":[],
                                        "Visibility":{"Value":"Hidden","State":"Authored"}
                                    }
                                    _cul.LinguisticMetadata.Content = _json_lm.dumps(_lm_meta)
                                    break
                        except Exception as _lv_err:
                            print(f"  Note: Entity Visibility write skipped for {fix.get('table','')}.{fix['name']}: {str(_lv_err)[:80]}")
                        applied.append(fix)
                elif fix["type"]=="synonym":
                    # Create en-US Culture if not exists, then add ObjectTranslation Caption
                    import Microsoft.AnalysisServices.Tabular as TOM_NS
                    _culture = None
                    for _c in tom.model.Cultures:
                        if str(_c.Name) == "en-US":
                            _culture = _c
                            break
                    if _culture is None:
                        _culture = TOM_NS.Culture()
                        _culture.Name = "en-US"
                        tom.model.Cultures.Add(_culture)
                    # Find the target object
                    _obj = None
                    _ot = fix.get("obj_type", "")
                    if _ot == "table":
                        for _t in tom.model.Tables:
                            if _t.Name == fix["name"]: _obj = _t; break
                    elif _ot == "column":
                        for _t in tom.model.Tables:
                            if _t.Name == fix.get("table"):
                                for _col in _t.Columns:
                                    if _col.Name == fix["name"]: _obj = _col; break
                                break
                    elif _ot == "measure":
                        for _t in tom.model.Tables:
                            for _m in _t.Measures:
                                if _m.Name == fix["name"]: _obj = _m; break
                            if _obj: break
                    if _obj is not None:
                        _trans = TOM_NS.ObjectTranslation()
                        _trans.Object = _obj
                        _trans.Property = TOM_NS.TranslatedProperty.Caption
                        _trans.Value = str(fix.get("value", fix.get("proposed", "")))
                        try:
                            _culture.ObjectTranslations.Add(_trans)
                            applied.append(fix)
                        except Exception as _te:
                            # May already exist — try update instead
                            try:
                                for _existing in _culture.ObjectTranslations:
                                    if _existing.Object == _obj and str(_existing.Property) == "Caption":
                                        _existing.Value = str(fix.get("value", fix.get("proposed", "")))
                                        applied.append(fix)
                                        break
                            except Exception:
                                errors.append({"fix": fix, "error": str(_te)[:100]})
            except Exception as e:
                errors.append({"fix":fix,"error":str(e)[:150]})
    return applied, errors

approved = get_approved(fix_plan, SELECTED_FIXES, EXCLUDED_FIXES)

if len(approved)==0:
    print("No fixes selected. Edit SELECTED_FIXES above.")
else:
    old_overall = results.get("_overall",{}).get("score",0)

    print(f"Applying {len(approved)} fix(es)...")
    if USE_SPN:
        with set_service_principal(tenant_id=SPN_TENANT_ID,client_id=SPN_CLIENT_ID,client_secret=SPN_CLIENT_SECRET):
            applied, errors = apply_fixes(approved, DATASET, WORKSPACE)
    else:
        applied, errors = apply_fixes(approved, DATASET, WORKSPACE)

    print(f"Applied: {len(applied)}/{len(approved)}")
    if errors:
        print(f"Errors: {len(errors)}")
        for e in errors[:3]: print(f"  {e['fix']['item']}: {e['error']}")

    # Show what was applied
    applied_df = pd.DataFrame([{"Category":f["category"],"Item":f["item"],
                                "Change":f"{f.get('current','')} -> {f.get('proposed','')}"} for f in applied])
    if len(applied_df)>0: display(applied_df)

    # ── Schema Refresh (sync metadata before re-scan) ──
    print(f"\nRefreshing schema to sync applied changes...")
    try:
        _refresh_schema(DATASET, WORKSPACE)
    except Exception as _re:
        print(f"  Schema refresh skipped: {str(_re)[:100]}")

    # ── Verify ──
    print(f"\nRe-scanning to verify...")
    def _verify():
        global results
        meta2 = preflight(DATASET, WORKSPACE)
        if meta2 is None: return
        r2={}
        for key, fn in [("test_framework",lambda:tool_test_framework(DATASET,meta2,WORKSPACE)),
                        ("copilot_readiness",lambda:tool_copilot(DATASET,meta2,WORKSPACE)),
                        ("bpa",lambda:tool_bpa(DATASET,WORKSPACE)),
                        ("dax_dependencies",lambda:tool_dax_deps(DATASET,meta2,WORKSPACE)),
                        ("data_quality",lambda:tool_data_quality(DATASET,meta2,WORKSPACE)),
                        ("direct_lake",lambda:tool_direct_lake(DATASET,meta2,WORKSPACE)),
                        ("capacity",lambda:tool_capacity(DATASET,WORKSPACE))]:
            try: r2[key]=fn()
            except: pass

        ws2,wt=0,0
        for k,w in WEIGHTS.items():
            if k in r2 and not r2[k].get("extra",{}).get("skipped",False):
                ws2+=r2[k]["score"]*w; wt+=w
        new_overall=round(ws2/wt,1) if wt>0 else 0

        # Before/After
        print(f"\n{'='*55}")
        print(f"  BEFORE vs AFTER")
        print(f"{'='*55}")
        delta_rows = []
        for k in WEIGHTS:
            if k in results and k in r2 and not results[k].get("extra",{}).get("skipped",False):
                b,a=results[k]["score"],r2[k]["score"]
                d=a-b
                delta_rows.append({"Tool":results[k]["name"],"Before":f"{b:.1f}%","After":f"{a:.1f}%",
                                  "Delta":f"{'+' if d>0 else ''}{d:.1f}%"})
        delta_rows.append({"Tool":"OVERALL","Before":f"{old_overall:.1f}%","After":f"{new_overall:.1f}%",
                          "Delta":f"{'+' if new_overall>old_overall else ''}{new_overall-old_overall:.1f}%"})
        # Render as colored HTML table (proper visual diff)
        _ba_html = '<table style="width:100%;border-collapse:collapse;font-size:13px;margin:12px 0;font-family:Segoe UI,sans-serif">'
        _ba_html += '<thead><tr style="background:#1B3A4B;color:#fff">'
        _ba_html += '<th style="padding:10px 14px;text-align:left">Tool</th>'
        _ba_html += '<th style="padding:10px 14px;text-align:center">Before</th>'
        _ba_html += '<th style="padding:10px 14px;text-align:center">After</th>'
        _ba_html += '<th style="padding:10px 14px;text-align:center">Change</th>'
        _ba_html += '</tr></thead><tbody>'
        for _ri, _row in enumerate(delta_rows):
            _d_str = _row["Delta"]
            _d_num = 0.0
            try: _d_num = float(_d_str.replace("+","").replace("%",""))
            except: pass
            _d_color = "#107C10" if _d_num > 0 else ("#D13438" if _d_num < 0 else "#605E5C")
            _is_overall = _row["Tool"] == "OVERALL"
            _bg = "#F0F6F5" if _is_overall else ("#fff" if _ri%2==0 else "#FAF9F8")
            _weight = "700" if _is_overall else "500"
            _border = "border-top:2px solid #1B3A4B;" if _is_overall else ""
            _ba_html += f'<tr style="background:{_bg};{_border}">'
            _ba_html += f'<td style="padding:8px 14px;border-bottom:1px solid #EDEBE9;font-weight:{_weight}">{_row["Tool"]}</td>'
            _ba_html += f'<td style="padding:8px 14px;text-align:center;border-bottom:1px solid #EDEBE9;color:#605E5C">{_row["Before"]}</td>'
            _ba_html += f'<td style="padding:8px 14px;text-align:center;border-bottom:1px solid #EDEBE9;font-weight:600">{_row["After"]}</td>'
            _ba_html += f'<td style="padding:8px 14px;text-align:center;border-bottom:1px solid #EDEBE9;color:{_d_color};font-weight:700">{_d_str}</td>'
            _ba_html += '</tr>'
        _ba_html += '</tbody></table>'
        try:
            displayHTML(_ba_html)
        except Exception:
            try:
                from IPython.display import display as _dsp, HTML as _HT
                _dsp(_HT(_ba_html))
            except Exception:
                display(pd.DataFrame(delta_rows))

        if new_overall>old_overall:
            print(f"\n  Improved: {score_to_grade(old_overall)} -> {score_to_grade(new_overall)} (+{new_overall-old_overall:.1f}%)")
        results.update(r2)

    if USE_SPN:
        with set_service_principal(tenant_id=SPN_TENANT_ID,client_id=SPN_CLIENT_ID,client_secret=SPN_CLIENT_SECRET):
            _verify()
    else:
        _verify()

StatementMeta(, a122e937-a10c-d30c-a7db-316995a292ed, 28, Finished, Available, Finished, False)

Available fixes (107 total):
  Generation method: AI=77  Template=0  Rule=30
  AI prompts include: table columns, relationships, data types, DAX expressions

  #  Source    Category         Item                                Current      Proposed
--------------------------------------------------------------------------------------------------------------
  0  AI        Description      Column: dimension_city.ValidTo      (empty)      City validity end
  1  AI        Description      Column: dimension_city.LineageKey   (empty)      City lineage key
  2  AI        Description      Column: dimension_customer.Custom   (empty)      Customer identifier
  3  AI        Description      Column: dimension_customer.WWICus   (empty)      Customer identifier
  4  AI        Description      Column: dimension_customer.Custom   (empty)      Customer name
  5  AI        Description      Column: dimension_customer.BillTo   (empty)      Billing customer name
  6  AI        Description      Column: dime